# Optical source selection

This diagnostic notebook compares four optical configurations for the
ET downscaling workflow:

1. Sentinel-2 at 20 m
2. HLS-S30 at 30 m
3. HLS-L30 at 30 m
4. HLS-S30 + HLS-L30 at 30 m

The comparison is performed before selecting the final optical source
or prediction-grid resolution.

The evaluation is organized in three stages:

1. Product availability and temporal complementarity.
2. Station × MODIS-period spatial coverage and common optical predictors.
3. ET-model performance under identical validation folds.

No optical source is selected a priori.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import ee
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "Repository root not found. Expected pyproject.toml."
    )


REPO_ROOT = find_repo_root()
SRC_PATH = REPO_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("Repository:", REPO_ROOT)
print("Source path:", SRC_PATH)

Repository: C:\Users\User\Desktop\cristian\GEE\26-et-downscaling-fundacion
Source path: C:\Users\User\Desktop\cristian\GEE\26-et-downscaling-fundacion\src


In [2]:
EE_PROJECT = "ee-change"

ee.Initialize(project=EE_PROJECT)
ee.Number(1).getInfo()

print("Earth Engine initialized with project:", EE_PROJECT)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Earth Engine initialized with project: ee-change


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


In [3]:
from et_downscaling.hls import (
    build_hls_medoid,
    get_hls_collection,
)

from et_downscaling.modis import (
    build_modis_inputs,
    get_modis_period_end,
)

from et_downscaling.sentinel2 import (
    build_s2_medoid,
    get_sentinel2_collection,
)

In [4]:
modis_inputs = build_modis_inputs()

station_footprints = modis_inputs[
    "station_footprints"
]

analysis_geometry = (
    station_footprints
    .geometry()
)


s2_collection = (
    get_sentinel2_collection(
        station_footprints
    )
)


hls_collection = (
    get_hls_collection(
        station_footprints
    )
)


hls_s30_collection = (
    hls_collection
    .filter(
        ee.Filter.eq(
            "sensor",
            "S30",
        )
    )
)


hls_l30_collection = (
    hls_collection
    .filter(
        ee.Filter.eq(
            "sensor",
            "L30",
        )
    )
)


OPTICAL_CANDIDATES = {
    "S2": {
        "collection": s2_collection,
        "scale_m": 20,
        "medoid_builder": build_s2_medoid,
    },
    "HLS_S30": {
        "collection": hls_s30_collection,
        "scale_m": 30,
        "medoid_builder": build_hls_medoid,
    },
    "HLS_L30": {
        "collection": hls_l30_collection,
        "scale_m": 30,
        "medoid_builder": build_hls_medoid,
    },
    "HLS_COMBINED": {
        "collection": hls_collection,
        "scale_m": 30,
        "medoid_builder": build_hls_medoid,
    },
}

print("Optical candidates:")
for name, config in OPTICAL_CANDIDATES.items():
    print(
        name,
        "->",
        config["scale_m"],
        "m",
    )

Optical candidates:
S2 -> 20 m
HLS_S30 -> 30 m
HLS_L30 -> 30 m
HLS_COMBINED -> 30 m


In [5]:
availability_rows = []

for name, config in OPTICAL_CANDIDATES.items():
    collection = config["collection"]

    product_count = (
        collection
        .size()
        .getInfo()
    )

    distinct_dates = (
        collection
        .aggregate_array(
            "date_key"
        )
        .distinct()
        .size()
        .getInfo()
    )

    availability_rows.append(
        {
            "source": name,
            "scale_m": config["scale_m"],
            "products": product_count,
            "distinct_dates": distinct_dates,
        }
    )


availability_summary = pd.DataFrame(
    availability_rows
)

display(
    availability_summary
)

,source,scale_m,products,distinct_dates
0,S2,20,439,217
1,HLS_S30,30,3161,638
2,HLS_L30,30,216,108
3,HLS_COMBINED,30,3377,681


In [6]:
s30_dates = set(
    hls_s30_collection
    .aggregate_array("date_key")
    .distinct()
    .getInfo()
)

l30_dates = set(
    hls_l30_collection
    .aggregate_array("date_key")
    .distinct()
    .getInfo()
)


hls_date_summary = pd.DataFrame(
    [
        {
            "metric": "S30 dates",
            "count": len(s30_dates),
        },
        {
            "metric": "L30 dates",
            "count": len(l30_dates),
        },
        {
            "metric": "Shared S30-L30 dates",
            "count": len(
                s30_dates & l30_dates
            ),
        },
        {
            "metric": "S30-only dates",
            "count": len(
                s30_dates - l30_dates
            ),
        },
        {
            "metric": "L30-only dates",
            "count": len(
                l30_dates - s30_dates
            ),
        },
        {
            "metric": "Combined unique dates",
            "count": len(
                s30_dates | l30_dates
            ),
        },
    ]
)

display(
    hls_date_summary
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,metric,count
0,S30 dates,638
1,L30 dates,108
2,Shared S30-L30 dates,65
3,S30-only dates,573
4,L30-only dates,43
5,Combined unique dates,681


In [7]:
def summarize_station_availability(
    source_name: str,
    collection: ee.ImageCollection,
    scale_m: int,
) -> list[dict]:
    def summarize_feature(feature):
        feature = ee.Feature(feature)

        station_collection = (
            ee.ImageCollection(collection)
            .filterBounds(
                feature.geometry()
            )
        )

        return ee.Feature(
            None,
            {
                "source": source_name,
                "station": feature.get("station"),
                "station_id": feature.get("station_id"),
                "scale_m": scale_m,
                "products": station_collection.size(),
                "distinct_dates": (
                    ee.List(
                        station_collection.aggregate_array(
                            "date_key"
                        )
                    )
                    .distinct()
                    .size()
                ),
            },
        )

    result = ee.FeatureCollection(
        station_footprints.map(
            summarize_feature
        )
    )

    info = result.getInfo()

    return [
        feature["properties"]
        for feature in info["features"]
    ]


station_availability_rows = []

for source_name, config in OPTICAL_CANDIDATES.items():
    rows = summarize_station_availability(
        source_name=source_name,
        collection=config["collection"],
        scale_m=config["scale_m"],
    )

    station_availability_rows.extend(rows)


station_availability = pd.DataFrame(
    station_availability_rows
)

display(
    station_availability.sort_values(
        [
            "station",
            "source",
        ]
    )
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,distinct_dates,products,scale_m,source,station,station_id
17,669,2985,30,HLS_COMBINED,Bananera,00000000000000000002
12,107,107,30,HLS_L30,Bananera,00000000000000000002
7,627,2878,30,HLS_S30,Bananera,00000000000000000002
2,217,218,20,S2,Bananera,00000000000000000002
19,670,2986,30,HLS_COMBINED,Bosque seco,00000000000000000004
14,108,108,30,HLS_L30,Bosque seco,00000000000000000004
9,627,2878,30,HLS_S30,Bosque seco,00000000000000000004
4,217,218,20,S2,Bosque seco,00000000000000000004
18,680,3376,30,HLS_COMBINED,Manglar,00000000000000000003
13,107,215,30,HLS_L30,Manglar,00000000000000000003


In [8]:
station_date_pivot = (
    station_availability
    .pivot(
        index="station",
        columns="source",
        values="distinct_dates",
    )
)

display(
    station_date_pivot
)

source,HLS_COMBINED,HLS_L30,HLS_S30,S2
station,,,,
Bananera,669,107,627,217
Bosque seco,670,108,627,217
Manglar,680,107,638,217
Palma,669,107,627,217
Pastos limpios,669,107,627,217


In [9]:
test_station = (
    station_footprints
    .filter(
        ee.Filter.eq(
            "station",
            "Bananera",
        )
    )
    .first()
)

test_geometry = (
    ee.Feature(
        test_station
    )
    .geometry()
)


hls_s30_test = (
    hls_s30_collection
    .filterBounds(
        test_geometry
    )
)


sample_hls_metadata = (
    hls_s30_test
    .limit(20)
    .map(
        lambda image: ee.Feature(
            None,
            {
                "system_index":
                    image.get(
                        "system:index"
                    ),

                "system_date":
                    image.date().format(
                        "yyyy-MM-dd HH:mm:ss"
                    ),

                "date_key":
                    image.get(
                        "date_key"
                    ),

                "sensing_time":
                    image.get(
                        "SENSING_TIME"
                    ),

                "mgrs_tile":
                    image.get(
                        "MGRS_TILE_ID"
                    ),

                "product_uri":
                    image.get(
                        "PRODUCT_URI"
                    ),
            },
        )
    )
)

sample_info = (
    sample_hls_metadata
    .getInfo()
)

sample_rows = [
    feature["properties"]
    for feature in sample_info["features"]
]

sample_metadata_df = (
    pd.DataFrame(
        sample_rows
    )
)

display(
    sample_metadata_df
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,date_key,mgrs_tile,product_uri,sensing_time,system_date,system_index
0,2021-01-03,18PWS,S2A_MSIL1C_20210103T152641_N0209_R025_T18PWS_2...,None,2021-01-03 15:26:41,1_T18PWS_20210103T152641
1,2021-01-06,18PWS,S2A_MSIL1C_20210106T153621_N0209_R068_T18PWS_2...,None,2021-01-06 15:36:21,1_T18PWS_20210106T153621
2,2021-01-08,18PWS,S2B_MSIL1C_20210108T152639_N0209_R025_T18PWS_2...,None,2021-01-08 15:26:39,1_T18PWS_20210108T152639
3,2021-01-11,18PWS,S2B_MSIL1C_20210111T153619_N0209_R068_T18PWS_2...,None,2021-01-11 15:36:19,1_T18PWS_20210111T153619
4,2021-01-13,18PWS,S2A_MSIL1C_20210113T152641_N0209_R025_T18PWS_2...,None,2021-01-13 15:26:41,1_T18PWS_20210113T152641
5,2021-01-16,18PWS,S2A_MSIL1C_20210116T153621_N0209_R068_T18PWS_2...,None,2021-01-16 15:36:21,1_T18PWS_20210116T153621
6,2021-01-18,18PWS,S2B_MSIL1C_20210118T152639_N0209_R025_T18PWS_2...,None,2021-01-18 15:26:39,1_T18PWS_20210118T152639
7,2021-01-21,18PWS,S2B_MSIL1C_20210121T153619_N0209_R068_T18PWS_2...,None,2021-01-21 15:36:19,1_T18PWS_20210121T153619
8,2021-01-23,18PWS,S2A_MSIL1C_20210123T152641_N0209_R025_T18PWS_2...,None,2021-01-23 15:26:41,1_T18PWS_20210123T152641
9,2021-01-26,18PWS,S2A_MSIL1C_20210126T153621_N0209_R068_T18PWS_2...,None,2021-01-26 15:36:21,1_T18PWS_20210126T153621


In [11]:
first_hls_image = ee.Image(
    hls_s30_test.first()
)

property_names = (
    first_hls_image
    .propertyNames()
    .getInfo()
)

date_property_check = pd.DataFrame(
    [
        {
            "property": "system:time_start",
            "available": (
                "system:time_start"
                in property_names
            ),
        },
        {
            "property": "SENSING_TIME",
            "available": (
                "SENSING_TIME"
                in property_names
            ),
        },
        {
            "property": "PRODUCT_URI",
            "available": (
                "PRODUCT_URI"
                in property_names
            ),
        },
        {
            "property": "MGRS_TILE_ID",
            "available": (
                "MGRS_TILE_ID"
                in property_names
            ),
        },
    ]
)

display(date_property_check)

,property,available
0,system:time_start,True
1,SENSING_TIME,False
2,PRODUCT_URI,True
3,MGRS_TILE_ID,True


In [12]:
tile_histogram = (
    hls_s30_test
    .aggregate_histogram(
        "MGRS_TILE_ID"
    )
    .getInfo()
)

tile_table = (
    pd.DataFrame(
        [
            {
                "mgrs_tile": tile,
                "products": count,
            }
            for tile, count
            in tile_histogram.items()
        ]
    )
    .sort_values(
        "products",
        ascending=False,
    )
)

display(
    tile_table
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,mgrs_tile,products
21,18PWS,275
16,01WCR,159
19,01WCU,154
18,01WCT,141
14,01WCP,138
9,01UBT,128
23,60UXC,124
20,01WCV,117
11,01VCL,115
26,60WWE,114


In [13]:
from et_downscaling.config import (
    START_DATE,
    END_DATE,
)


s2_raw_test = (
    ee.ImageCollection(
        "COPERNICUS/S2_SR_HARMONIZED"
    )
    .filterBounds(
        test_geometry
    )
    .filterDate(
        START_DATE,
        END_DATE,
    )
)


s2_linked_test = (
    s2_collection
    .filterBounds(
        test_geometry
    )
)


s2_collection_check = pd.DataFrame(
    [
        {
            "collection": "S2 raw",
            "products": (
                s2_raw_test
                .size()
                .getInfo()
            ),
            "distinct_dates": (
                s2_raw_test
                .aggregate_array(
                    "system:time_start"
                )
                .distinct()
                .size()
                .getInfo()
            ),
        },
        {
            "collection": "S2 linked",
            "products": (
                s2_linked_test
                .size()
                .getInfo()
            ),
            "distinct_dates": (
                s2_linked_test
                .aggregate_array(
                    "date_key"
                )
                .distinct()
                .size()
                .getInfo()
            ),
        },
    ]
)

display(
    s2_collection_check
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,collection,products,distinct_dates
0,S2 raw,218,218
1,S2 linked,218,217


In [14]:
s2_tile_histogram = (
    s2_raw_test
    .aggregate_histogram(
        "MGRS_TILE"
    )
    .getInfo()
)

s2_tile_table = (
    pd.DataFrame(
        [
            {
                "mgrs_tile": tile,
                "products": count,
            }
            for tile, count
            in s2_tile_histogram.items()
        ]
    )
    .sort_values(
        "products",
        ascending=False,
    )
)

display(s2_tile_table)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,mgrs_tile,products
0,18PWS,218


In [15]:
local_mgrs_tiles = list(
    s2_tile_histogram.keys()
)

hls_s30_local_tiles = (
    hls_s30_collection
    .filter(
        ee.Filter.inList(
            "MGRS_TILE_ID",
            local_mgrs_tiles,
        )
    )
)

local_hls_summary = pd.DataFrame(
    [
        {
            "source": "HLS_S30_all_filterBounds",
            "products": (
                hls_s30_test
                .size()
                .getInfo()
            ),
            "distinct_dates": (
                hls_s30_test
                .aggregate_array("date_key")
                .distinct()
                .size()
                .getInfo()
            ),
        },
        {
            "source": "HLS_S30_local_MGRS",
            "products": (
                hls_s30_local_tiles
                .size()
                .getInfo()
            ),
            "distinct_dates": (
                hls_s30_local_tiles
                .aggregate_array("date_key")
                .distinct()
                .size()
                .getInfo()
            ),
        },
    ]
)

display(local_hls_summary)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,source,products,distinct_dates
0,HLS_S30_all_filterBounds,2878,627
1,HLS_S30_local_MGRS,276,276


In [16]:
from et_downscaling.hls import (
    prepare_hls_s30,
)


def add_valid_pixel_count(image):
    image = ee.Image(image)

    prepared = prepare_hls_s30(
        image
    )

    valid_mask = (
        prepared
        .select("Red")
        .mask()
        .rename("valid")
    )

    valid_count = (
        valid_mask
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=test_geometry,
            scale=30,
            maxPixels=1e6,
        )
        .get("valid")
    )

    return (
        image
        .set(
            "valid_pixels_at_station",
            valid_count,
        )
    )


hls_s30_local_checked = (
    hls_s30_local_tiles
    .map(
        add_valid_pixel_count
    )
)


hls_local_valid_summary = pd.DataFrame(
    [
        {
            "metric": "local tile products",
            "count": (
                hls_s30_local_checked
                .size()
                .getInfo()
            ),
        },
        {
            "metric": "products with valid pixels",
            "count": (
                hls_s30_local_checked
                .filter(
                    ee.Filter.gt(
                        "valid_pixels_at_station",
                        0,
                    )
                )
                .size()
                .getInfo()
            ),
        },
        {
            "metric": "dates with valid pixels",
            "count": (
                hls_s30_local_checked
                .filter(
                    ee.Filter.gt(
                        "valid_pixels_at_station",
                        0,
                    )
                )
                .aggregate_array(
                    "date_key"
                )
                .distinct()
                .size()
                .getInfo()
            ),
        },
    ]
)

display(
    hls_local_valid_summary
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,metric,count
0,local tile products,276
1,products with valid pixels,113
2,dates with valid pixels,113


In [17]:
def add_hls_mask_diagnostics(image):
    image = ee.Image(image)

    # --------------------------------------------------------
    # 1. Raw spectral data availability
    # --------------------------------------------------------

    raw_spectral = (
        image
        .select(
            [
                "B2",
                "B3",
                "B4",
                "B8A",
                "B11",
                "B12",
            ]
        )
    )

    raw_valid = (
        raw_spectral
        .mask()
        .reduce(
            ee.Reducer.min()
        )
        .rename(
            "raw_valid"
        )
    )

    # --------------------------------------------------------
    # 2. HLS Fmask clear condition
    # --------------------------------------------------------

    fmask = image.select(
        "Fmask"
    )

    no_cloud = (
        fmask
        .bitwiseAnd(1 << 1)
        .eq(0)
    )

    no_adjacent = (
        fmask
        .bitwiseAnd(1 << 2)
        .eq(0)
    )

    no_shadow = (
        fmask
        .bitwiseAnd(1 << 3)
        .eq(0)
    )

    no_snow = (
        fmask
        .bitwiseAnd(1 << 4)
        .eq(0)
    )

    clear = (
        no_cloud
        .And(no_adjacent)
        .And(no_shadow)
        .And(no_snow)
        .rename("clear")
    )

    raw_clear = (
        raw_valid
        .And(clear)
        .rename(
            "raw_clear"
        )
    )

    # --------------------------------------------------------
    # 3. Positive-reflectance condition used by current code
    # --------------------------------------------------------

    positive = (
        raw_spectral
        .gt(0)
        .reduce(
            ee.Reducer.min()
        )
        .rename(
            "positive"
        )
    )

    full_current_mask = (
        raw_clear
        .And(positive)
        .rename(
            "full_current"
        )
    )

    diagnostic_stack = (
        raw_valid
        .addBands(raw_clear)
        .addBands(full_current_mask)
    )

    counts = (
        diagnostic_stack
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=test_geometry,
            scale=30,
            maxPixels=1e6,
        )
    )

    return (
        image
        .set(
            {
                "raw_valid_pixels":
                    counts.get(
                        "raw_valid"
                    ),

                "raw_clear_pixels":
                    counts.get(
                        "raw_clear"
                    ),

                "full_current_pixels":
                    counts.get(
                        "full_current"
                    ),
            }
        )
    )


hls_s30_mask_diagnostics = (
    hls_s30_local_tiles
    .map(
        add_hls_mask_diagnostics
    )
)


mask_diagnostic_summary = pd.DataFrame(
    [
        {
            "stage": "Local MGRS products",
            "products": (
                hls_s30_mask_diagnostics
                .size()
                .getInfo()
            ),
        },
        {
            "stage": "Raw data over footprint",
            "products": (
                hls_s30_mask_diagnostics
                .filter(
                    ee.Filter.gt(
                        "raw_valid_pixels",
                        0,
                    )
                )
                .size()
                .getInfo()
            ),
        },
        {
            "stage": "Raw data + Fmask clear",
            "products": (
                hls_s30_mask_diagnostics
                .filter(
                    ee.Filter.gt(
                        "raw_clear_pixels",
                        0,
                    )
                )
                .size()
                .getInfo()
            ),
        },
        {
            "stage": "Current full HLS mask",
            "products": (
                hls_s30_mask_diagnostics
                .filter(
                    ee.Filter.gt(
                        "full_current_pixels",
                        0,
                    )
                )
                .size()
                .getInfo()
            ),
        },
    ]
)

display(
    mask_diagnostic_summary
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,stage,products
0,Local MGRS products,276
1,Raw data over footprint,190
2,Raw data + Fmask clear,113
3,Current full HLS mask,113


In [18]:
# ============================================================
# Sentinel-2 usable observations over the test footprint
# ============================================================

from et_downscaling.sentinel2 import (
    prepare_sentinel2,
)


def add_s2_valid_pixel_count(image):
    image = ee.Image(image)

    prepared = prepare_sentinel2(
        image
    )

    valid_mask = (
        prepared
        .select("Red")
        .mask()
        .rename("valid")
    )

    valid_count = (
        valid_mask
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=test_geometry,
            scale=20,
            maxPixels=1e6,
        )
        .get("valid")
    )

    return image.set(
        "valid_pixels_at_station",
        valid_count,
    )


s2_checked = (
    s2_linked_test
    .map(
        add_s2_valid_pixel_count
    )
)


s2_valid_products = (
    s2_checked
    .filter(
        ee.Filter.gt(
            "valid_pixels_at_station",
            0,
        )
    )
)


# ============================================================
# HLS-S30 high-aerosol diagnostic
# ============================================================

def add_hls_high_aerosol_diagnostic(image):
    image = ee.Image(image)

    raw_spectral = (
        image
        .select(
            [
                "B2",
                "B3",
                "B4",
                "B8A",
                "B11",
                "B12",
            ]
        )
    )

    spectral_valid = (
        raw_spectral
        .mask()
        .reduce(
            ee.Reducer.min()
        )
    )

    fmask = image.select(
        "Fmask"
    )

    no_cloud = (
        fmask
        .bitwiseAnd(1 << 1)
        .eq(0)
    )

    no_adjacent = (
        fmask
        .bitwiseAnd(1 << 2)
        .eq(0)
    )

    no_shadow = (
        fmask
        .bitwiseAnd(1 << 3)
        .eq(0)
    )

    no_snow = (
        fmask
        .bitwiseAnd(1 << 4)
        .eq(0)
    )

    aerosol_level = (
        fmask
        .rightShift(6)
        .bitwiseAnd(3)
    )

    not_high_aerosol = (
        aerosol_level
        .neq(3)
    )

    clear_mask = (
        spectral_valid
        .And(no_cloud)
        .And(no_adjacent)
        .And(no_shadow)
        .And(no_snow)
    )

    clear_no_high_aerosol = (
        clear_mask
        .And(
            not_high_aerosol
        )
        .rename(
            "valid"
        )
    )

    valid_count = (
        clear_no_high_aerosol
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=test_geometry,
            scale=30,
            maxPixels=1e6,
        )
        .get("valid")
    )

    return image.set(
        "valid_pixels_no_high_aerosol",
        valid_count,
    )


hls_s30_aerosol_checked = (
    hls_s30_local_tiles
    .map(
        add_hls_high_aerosol_diagnostic
    )
)


hls_s30_aerosol_valid = (
    hls_s30_aerosol_checked
    .filter(
        ee.Filter.gt(
            "valid_pixels_no_high_aerosol",
            0,
        )
    )
)


# ============================================================
# Comparable summary
# ============================================================

qa_comparison = pd.DataFrame(
    [
        {
            "source": "S2",
            "stage": "Raw products",
            "products": (
                s2_linked_test
                .size()
                .getInfo()
            ),
            "dates": (
                s2_linked_test
                .aggregate_array("date_key")
                .distinct()
                .size()
                .getInfo()
            ),
        },
        {
            "source": "S2",
            "stage": "Current QA",
            "products": (
                s2_valid_products
                .size()
                .getInfo()
            ),
            "dates": (
                s2_valid_products
                .aggregate_array("date_key")
                .distinct()
                .size()
                .getInfo()
            ),
        },
        {
            "source": "HLS_S30",
            "stage": "Raw local tile",
            "products": 276,
            "dates": 276,
        },
        {
            "source": "HLS_S30",
            "stage": "Raw data over footprint",
            "products": 190,
            "dates": None,
        },
        {
            "source": "HLS_S30",
            "stage": "Current Fmask clear",
            "products": 113,
            "dates": 113,
        },
        {
            "source": "HLS_S30",
            "stage": "Fmask + no high aerosol",
            "products": (
                hls_s30_aerosol_valid
                .size()
                .getInfo()
            ),
            "dates": (
                hls_s30_aerosol_valid
                .aggregate_array("date_key")
                .distinct()
                .size()
                .getInfo()
            ),
        },
    ]
)

display(
    qa_comparison
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,source,stage,products,dates
0,S2,Raw products,218,217.0
1,S2,Current QA,162,161.0
2,HLS_S30,Raw local tile,276,276.0
3,HLS_S30,Raw data over footprint,190,NaN
4,HLS_S30,Current Fmask clear,113,113.0
5,HLS_S30,Fmask + no high aerosol,81,81.0


In [19]:
def get_s2_valid_dates(
    footprint,
):
    footprint = ee.Feature(
        footprint
    )

    geometry = footprint.geometry()

    collection = (
        s2_collection
        .filterBounds(
            geometry
        )
    )

    def add_valid_pixels(image):
        image = ee.Image(image)

        prepared = prepare_sentinel2(
            image
        )

        valid_pixels = (
            prepared
            .select("Red")
            .mask()
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=geometry,
                scale=20,
                maxPixels=1e6,
            )
            .get("Red")
        )

        return image.set(
            "valid_pixels",
            valid_pixels,
        )

    valid_collection = (
        collection
        .map(
            add_valid_pixels
        )
        .filter(
            ee.Filter.gt(
                "valid_pixels",
                0,
            )
        )
    )

    return (
        valid_collection
        .aggregate_array(
            "date_key"
        )
        .distinct()
        .getInfo()
    )


def get_local_mgrs_tiles(
    footprint,
):
    footprint = ee.Feature(
        footprint
    )

    geometry = footprint.geometry()

    raw_s2 = (
        ee.ImageCollection(
            "COPERNICUS/S2_SR_HARMONIZED"
        )
        .filterBounds(
            geometry
        )
    )

    return (
        raw_s2
        .aggregate_array(
            "MGRS_TILE"
        )
        .distinct()
        .getInfo()
    )


def get_hls_valid_dates(
    footprint,
    collection,
    local_mgrs_tiles,
):
    footprint = ee.Feature(
        footprint
    )

    geometry = footprint.geometry()

    local_collection = (
        ee.ImageCollection(
            collection
        )
        .filter(
            ee.Filter.inList(
                "MGRS_TILE_ID",
                local_mgrs_tiles,
            )
        )
    )

    def add_valid_pixels(image):
        image = ee.Image(image)

        fmask = (
            image.select(
                "Fmask"
            )
        )

        no_cloud = (
            fmask
            .bitwiseAnd(
                1 << 1
            )
            .eq(0)
        )

        no_adjacent = (
            fmask
            .bitwiseAnd(
                1 << 2
            )
            .eq(0)
        )

        no_shadow = (
            fmask
            .bitwiseAnd(
                1 << 3
            )
            .eq(0)
        )

        no_snow = (
            fmask
            .bitwiseAnd(
                1 << 4
            )
            .eq(0)
        )

        aerosol_level = (
            fmask
            .rightShift(6)
            .bitwiseAnd(3)
        )

        no_high_aerosol = (
            aerosol_level
            .neq(3)
        )

        valid_mask = (
            image
            .select("B4")
            .mask()
            .And(
                no_cloud
            )
            .And(
                no_adjacent
            )
            .And(
                no_shadow
            )
            .And(
                no_snow
            )
            .And(
                no_high_aerosol
            )
            .rename(
                "valid"
            )
        )

        valid_pixels = (
            valid_mask
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=geometry,
                scale=30,
                maxPixels=1e6,
            )
            .get("valid")
        )

        return image.set(
            "valid_pixels",
            valid_pixels,
        )

    valid_collection = (
        local_collection
        .map(
            add_valid_pixels
        )
        .filter(
            ee.Filter.gt(
                "valid_pixels",
                0,
            )
        )
    )

    return (
        valid_collection
        .aggregate_array(
            "date_key"
        )
        .distinct()
        .getInfo()
    )

In [20]:
station_features = (
    station_footprints
    .toList(
        station_footprints.size()
    )
)

station_count = (
    station_footprints
    .size()
    .getInfo()
)

optical_availability_rows = []


for index in range(station_count):

    footprint = ee.Feature(
        station_features.get(
            index
        )
    )

    station_name = (
        footprint
        .get("station")
        .getInfo()
    )

    print(
        "Processing:",
        station_name,
    )

    local_mgrs_tiles = (
        get_local_mgrs_tiles(
            footprint
        )
    )

    s2_dates = set(
        get_s2_valid_dates(
            footprint
        )
    )

    s30_dates = set(
        get_hls_valid_dates(
            footprint,
            hls_s30_collection,
            local_mgrs_tiles,
        )
    )

    l30_dates = set(
        get_hls_valid_dates(
            footprint,
            hls_l30_collection,
            local_mgrs_tiles,
        )
    )

    combined_dates = (
        s30_dates
        | l30_dates
    )

    optical_availability_rows.extend(
        [
            {
                "station": station_name,
                "source": "S2",
                "valid_dates": len(
                    s2_dates
                ),
            },
            {
                "station": station_name,
                "source": "HLS_S30",
                "valid_dates": len(
                    s30_dates
                ),
            },
            {
                "station": station_name,
                "source": "HLS_L30",
                "valid_dates": len(
                    l30_dates
                ),
            },
            {
                "station": station_name,
                "source": "HLS_COMBINED",
                "valid_dates": len(
                    combined_dates
                ),
            },
        ]
    )


optical_valid_dates = pd.DataFrame(
    optical_availability_rows
)

display(
    optical_valid_dates
)


optical_valid_dates_pivot = (
    optical_valid_dates
    .pivot(
        index="station",
        columns="source",
        values="valid_dates",
    )
)

display(
    optical_valid_dates_pivot
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Processing: Pastos limpios
Processing: Palma
Processing: Bananera
Processing: Manglar
Processing: Bosque seco


,station,source,valid_dates
0,Pastos limpios,S2,154
1,Pastos limpios,HLS_S30,90
2,Pastos limpios,HLS_L30,0
3,Pastos limpios,HLS_COMBINED,90
4,Palma,S2,158
5,Palma,HLS_S30,101
6,Palma,HLS_L30,0
7,Palma,HLS_COMBINED,101
8,Bananera,S2,161
9,Bananera,HLS_S30,81


source,HLS_COMBINED,HLS_L30,HLS_S30,S2
station,,,,
Bananera,81,0,81,161
Bosque seco,112,0,112,165
Manglar,125,0,125,158
Palma,101,0,101,158
Pastos limpios,90,0,90,154


In [21]:
# ============================================================
# Inspect HLS-L30 metadata and identifiers
# ============================================================

first_l30 = ee.Image(
    hls_l30_collection.first()
)

l30_property_names = (
    first_l30
    .propertyNames()
    .getInfo()
)

print("MGRS_TILE_ID available:", "MGRS_TILE_ID" in l30_property_names)
print("LANDSAT_PRODUCT_ID available:", "LANDSAT_PRODUCT_ID" in l30_property_names)
print()
print("Available properties:")
print(l30_property_names)


l30_sample = (
    hls_l30_collection
    .limit(20)
    .map(
        lambda image: ee.Feature(
            None,
            {
                "system_index":
                    image.get(
                        "system:index"
                    ),

                "system_date":
                    image.date().format(
                        "yyyy-MM-dd HH:mm:ss"
                    ),

                "mgrs_tile_id":
                    image.get(
                        "MGRS_TILE_ID"
                    ),

                "landsat_product_id":
                    image.get(
                        "LANDSAT_PRODUCT_ID"
                    ),

                "spatial_coverage":
                    image.get(
                        "SPATIAL_COVERAGE"
                    ),

                "cloud_coverage":
                    image.get(
                        "CLOUD_COVERAGE"
                    ),
            },
        )
    )
)


l30_sample_info = (
    l30_sample
    .getInfo()
)

l30_sample_df = pd.DataFrame(
    [
        feature["properties"]
        for feature
        in l30_sample_info["features"]
    ]
)

display(
    l30_sample_df
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


MGRS_TILE_ID available: False
LANDSAT_PRODUCT_ID available: True

Available properties:
['hls_sensor', 'system:version', 'system:id', 'sensor', 'date_key', 'system:index', 'B6_scale', 'B11_scale', 'B1_scale', 'HLS_PROCESSING_TIME', 'MEAN_SUN_AZIMUTH_ANGLE', 'B4_scale', 'SAA_scale', 'TIRS_SSM_MODEL', 'MEAN_VIEW_AZIMUTH_ANGLE', 'system:footprint', 'MEAN_SUN_ZENITH_ANGLE', 'CLOUD_COVERAGE', 'B2_scale', 'B7_scale', 'B10_scale', 'SPATIAL_COVERAGE', 'system:time_end', 'LANDSAT_PRODUCT_ID', 'B5_scale', 'SZA_scale', 'system:time_start', 'B9_scale', 'VZA_scale', 'ACCODE', 'B3_scale', 'MEAN_VIEW_ZENITH_ANGLE', 'NBAR_SOLAR_ZENITH', 'USGS_SOFTWARE', 'system:asset_size', 'TIRS_SSM_POSITION_STATUS', 'VAA_scale', 'system:bands', 'system:band_names']


,cloud_coverage,landsat_product_id,mgrs_tile_id,spatial_coverage,system_date,system_index
0,38,LC08_L1TP_009052_20210111_20210308_02_T1,None,100,2021-01-11 15:17:08,2_T18PWS_20210111T151708
1,24,LC08_L1TP_009052_20210111_20210308_02_T1,None,100,2021-01-11 15:17:08,2_T18PWT_20210111T151708
2,50,LC08_L1TP_009052_20210127_20210305_02_T1,None,100,2021-01-27 15:17:04,2_T18PWS_20210127T151704
3,99,LC08_L1TP_009052_20210127_20210305_02_T1,None,99,2021-01-27 15:17:04,2_T18PWT_20210127T151704
4,0,LC08_L1TP_009052_20210228_20210311_02_T1,None,100,2021-02-28 15:16:55,2_T18PWS_20210228T151655
5,1,LC08_L1TP_009052_20210228_20210311_02_T1,None,100,2021-02-28 15:16:55,2_T18PWT_20210228T151655
6,46,LC08_L1TP_009052_20210316_20210328_02_T1,None,100,2021-03-16 15:16:46,2_T18PWS_20210316T151646
7,63,LC08_L1TP_009052_20210316_20210328_02_T1,None,99,2021-03-16 15:16:46,2_T18PWT_20210316T151646
8,100,LC08_L1TP_009052_20210401_20210409_02_T1,None,100,2021-04-01 15:16:42,2_T18PWS_20210401T151642
9,87,LC08_L1TP_009052_20210401_20210409_02_T1,None,100,2021-04-01 15:16:42,2_T18PWT_20210401T151642


In [22]:
# ============================================================
# Derive HLS MGRS tile from system:index
# ============================================================

def add_hls_mgrs_tile_from_index(image):
    image = ee.Image(
        image
    )

    system_index = ee.String(
        image.get(
            "system:index"
        )
    )

    # Expected Earth Engine HLS index:
    #
    # 2_T18PWS_20210111T151708
    #
    # split("_")[1] -> T18PWS
    # slice(1)       -> 18PWS
    tile_token = ee.String(
        system_index
        .split("_")
        .get(1)
    )

    mgrs_tile = (
        tile_token
        .slice(1)
    )

    return (
        image
        .set(
            "derived_mgrs_tile",
            mgrs_tile,
        )
    )


hls_l30_with_tile = (
    hls_l30_collection
    .map(
        add_hls_mgrs_tile_from_index
    )
)


l30_local_tiles = (
    hls_l30_with_tile
    .filter(
        ee.Filter.inList(
            "derived_mgrs_tile",
            local_mgrs_tiles,
        )
    )
)


l30_local_summary = pd.DataFrame(
    [
        {
            "metric": "All L30 products",
            "count": (
                hls_l30_collection
                .size()
                .getInfo()
            ),
        },
        {
            "metric": "Local MGRS products",
            "count": (
                l30_local_tiles
                .size()
                .getInfo()
            ),
        },
        {
            "metric": "Local MGRS dates",
            "count": (
                l30_local_tiles
                .aggregate_array(
                    "date_key"
                )
                .distinct()
                .size()
                .getInfo()
            ),
        },
    ]
)

display(
    l30_local_summary
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,metric,count
0,All L30 products,216
1,Local MGRS products,108
2,Local MGRS dates,108


In [23]:
# ============================================================
# HLS-L30 QA diagnostics over the test footprint
# ============================================================

from et_downscaling.hls import (
    prepare_hls_l30,
)


def add_l30_mask_diagnostics(image):
    image = ee.Image(
        image
    )

    # --------------------------------------------------------
    # Common optical bands
    #
    # B2 = Blue
    # B3 = Green
    # B4 = Red
    # B5 = NIR
    # B6 = SWIR1
    # B7 = SWIR2
    # --------------------------------------------------------

    raw_spectral = (
        image
        .select(
            [
                "B2",
                "B3",
                "B4",
                "B5",
                "B6",
                "B7",
            ]
        )
    )

    raw_valid = (
        raw_spectral
        .mask()
        .reduce(
            ee.Reducer.min()
        )
        .rename(
            "raw_valid"
        )
    )

    # --------------------------------------------------------
    # Fmask
    # --------------------------------------------------------

    fmask = (
        image.select(
            "Fmask"
        )
    )

    no_cloud = (
        fmask
        .bitwiseAnd(
            1 << 1
        )
        .eq(0)
    )

    no_adjacent = (
        fmask
        .bitwiseAnd(
            1 << 2
        )
        .eq(0)
    )

    no_shadow = (
        fmask
        .bitwiseAnd(
            1 << 3
        )
        .eq(0)
    )

    no_snow = (
        fmask
        .bitwiseAnd(
            1 << 4
        )
        .eq(0)
    )

    clear_mask = (
        raw_valid
        .And(no_cloud)
        .And(no_adjacent)
        .And(no_shadow)
        .And(no_snow)
        .rename(
            "clear"
        )
    )

    # --------------------------------------------------------
    # Aerosol
    # --------------------------------------------------------

    aerosol_level = (
        fmask
        .rightShift(6)
        .bitwiseAnd(3)
    )

    no_high_aerosol = (
        aerosol_level
        .neq(3)
    )

    clear_no_high_aerosol = (
        clear_mask
        .And(
            no_high_aerosol
        )
        .rename(
            "clear_no_high_aerosol"
        )
    )

    # --------------------------------------------------------
    # Current production HLS mask
    # --------------------------------------------------------

    prepared_current = (
        prepare_hls_l30(
            image
        )
    )

    current_valid = (
        prepared_current
        .select("Red")
        .mask()
        .rename(
            "current_valid"
        )
    )

    # --------------------------------------------------------
    # Count pixels over the MODIS footprint
    # --------------------------------------------------------

    diagnostic_stack = (
        raw_valid
        .addBands(
            clear_mask
        )
        .addBands(
            clear_no_high_aerosol
        )
        .addBands(
            current_valid
        )
    )

    counts = (
        diagnostic_stack
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=test_geometry,
            scale=30,
            maxPixels=1e6,
        )
    )

    return (
        image
        .set(
            {
                "raw_valid_pixels":
                    counts.get(
                        "raw_valid"
                    ),

                "clear_pixels":
                    counts.get(
                        "clear"
                    ),

                "clear_no_high_aerosol_pixels":
                    counts.get(
                        "clear_no_high_aerosol"
                    ),

                "current_valid_pixels":
                    counts.get(
                        "current_valid"
                    ),
            }
        )
    )


l30_mask_diagnostics = (
    l30_local_tiles
    .map(
        add_l30_mask_diagnostics
    )
)

In [24]:
# ============================================================
# HLS-L30 QA stage summary
# ============================================================

l30_qa_summary = pd.DataFrame(
    [
        {
            "stage": "Local MGRS products",
            "products": (
                l30_mask_diagnostics
                .size()
                .getInfo()
            ),
        },
        {
            "stage": "Raw data over footprint",
            "products": (
                l30_mask_diagnostics
                .filter(
                    ee.Filter.gt(
                        "raw_valid_pixels",
                        0,
                    )
                )
                .size()
                .getInfo()
            ),
        },
        {
            "stage": "Fmask clear",
            "products": (
                l30_mask_diagnostics
                .filter(
                    ee.Filter.gt(
                        "clear_pixels",
                        0,
                    )
                )
                .size()
                .getInfo()
            ),
        },
        {
            "stage": "Fmask + no high aerosol",
            "products": (
                l30_mask_diagnostics
                .filter(
                    ee.Filter.gt(
                        "clear_no_high_aerosol_pixels",
                        0,
                    )
                )
                .size()
                .getInfo()
            ),
        },
        {
            "stage": "Current prepare_hls_l30",
            "products": (
                l30_mask_diagnostics
                .filter(
                    ee.Filter.gt(
                        "current_valid_pixels",
                        0,
                    )
                )
                .size()
                .getInfo()
            ),
        },
    ]
)

display(
    l30_qa_summary
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


,stage,products
0,Local MGRS products,108
1,Raw data over footprint,107
2,Fmask clear,52
3,Fmask + no high aerosol,44
4,Current prepare_hls_l30,52


In [25]:
# ============================================================
# Common optical availability
# ============================================================

def add_hls_mgrs_tile(image):
    """
    Derive the HLS MGRS tile consistently for both S30 and L30
    from the Earth Engine system:index.

    Examples
    --------
    1_T18PWS_20210103T152641 -> 18PWS
    2_T18PWS_20210111T151708 -> 18PWS
    """
    image = ee.Image(image)

    system_index = ee.String(
        image.get("system:index")
    )

    tile_token = ee.String(
        system_index
        .split("_")
        .get(1)
    )

    mgrs_tile = (
        tile_token
        .slice(1)
    )

    return image.set(
        "hls_mgrs_tile",
        mgrs_tile,
    )


hls_s30_tiled = (
    hls_s30_collection
    .map(add_hls_mgrs_tile)
)

hls_l30_tiled = (
    hls_l30_collection
    .map(add_hls_mgrs_tile)
)


def get_local_mgrs_tiles(
    footprint,
):
    """
    Identify local MGRS tiles independently from Sentinel-2.
    """
    footprint = ee.Feature(footprint)
    geometry = footprint.geometry()

    local_s2 = (
        ee.ImageCollection(
            "COPERNICUS/S2_SR_HARMONIZED"
        )
        .filterBounds(geometry)
    )

    return (
        local_s2
        .aggregate_array("MGRS_TILE")
        .distinct()
        .getInfo()
    )


def get_s2_common_valid_dates(
    footprint,
):
    """
    Return dates containing at least one valid pixel for the
    six-band common optical comparison.

    Common bands:
    Blue, Green, Red, NIR, SWIR1, SWIR2.

    Sentinel-2 QA uses the current Cloud Score+ threshold.
    """
    footprint = ee.Feature(footprint)
    geometry = footprint.geometry()

    collection = (
        s2_collection
        .filterBounds(geometry)
    )

    def add_valid_pixels(image):
        image = ee.Image(image)

        # 10 m common bands
        bands_10m = (
            image
            .select(
                [
                    "B2",
                    "B3",
                    "B4",
                ]
            )
        )

        # 20 m common bands
        bands_20m = (
            image
            .select(
                [
                    "B8A",
                    "B11",
                    "B12",
                ]
            )
        )

        valid_10m = (
            bands_10m
            .mask()
            .reduce(
                ee.Reducer.min()
            )
        )

        valid_20m = (
            bands_20m
            .mask()
            .reduce(
                ee.Reducer.min()
            )
        )

        cloud_score_valid = (
            image
            .select("cs_cdf")
            .mask()
        )

        clear = (
            image
            .select("cs_cdf")
            .gte(0.60)
        )

        valid_mask = (
            valid_10m
            .And(valid_20m)
            .And(cloud_score_valid)
            .And(clear)
            .rename("valid")
        )

        valid_pixels = (
            valid_mask
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=geometry,
                scale=20,
                maxPixels=1e6,
            )
            .get("valid")
        )

        return image.set(
            "common_valid_pixels",
            valid_pixels,
        )

    valid_collection = (
        collection
        .map(add_valid_pixels)
        .filter(
            ee.Filter.gt(
                "common_valid_pixels",
                0,
            )
        )
    )

    return set(
        valid_collection
        .aggregate_array("date_key")
        .distinct()
        .getInfo()
    )


def get_hls_common_valid_dates(
    footprint,
    collection,
    local_mgrs_tiles,
    sensor,
    exclude_high_aerosol=False,
):
    """
    Return HLS dates containing at least one valid common-band
    pixel over the footprint.
    """
    footprint = ee.Feature(footprint)
    geometry = footprint.geometry()

    local_collection = (
        ee.ImageCollection(collection)
        .filter(
            ee.Filter.inList(
                "hls_mgrs_tile",
                local_mgrs_tiles,
            )
        )
    )

    if sensor == "S30":
        common_source_bands = [
            "B2",
            "B3",
            "B4",
            "B8A",
            "B11",
            "B12",
        ]

    elif sensor == "L30":
        common_source_bands = [
            "B2",
            "B3",
            "B4",
            "B5",
            "B6",
            "B7",
        ]

    else:
        raise ValueError(
            f"Unsupported HLS sensor: {sensor}"
        )

    def add_valid_pixels(image):
        image = ee.Image(image)

        spectral_valid = (
            image
            .select(common_source_bands)
            .mask()
            .reduce(
                ee.Reducer.min()
            )
        )

        fmask = image.select("Fmask")

        no_cloud = (
            fmask
            .bitwiseAnd(1 << 1)
            .eq(0)
        )

        no_adjacent = (
            fmask
            .bitwiseAnd(1 << 2)
            .eq(0)
        )

        no_shadow = (
            fmask
            .bitwiseAnd(1 << 3)
            .eq(0)
        )

        no_snow = (
            fmask
            .bitwiseAnd(1 << 4)
            .eq(0)
        )

        valid_mask = (
            spectral_valid
            .And(no_cloud)
            .And(no_adjacent)
            .And(no_shadow)
            .And(no_snow)
        )

        if exclude_high_aerosol:
            aerosol_level = (
                fmask
                .rightShift(6)
                .bitwiseAnd(3)
            )

            valid_mask = (
                valid_mask
                .And(
                    aerosol_level.neq(3)
                )
            )

        valid_mask = (
            valid_mask
            .rename("valid")
        )

        valid_pixels = (
            valid_mask
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=geometry,
                scale=30,
                maxPixels=1e6,
            )
            .get("valid")
        )

        return image.set(
            "common_valid_pixels",
            valid_pixels,
        )

    valid_collection = (
        local_collection
        .map(add_valid_pixels)
        .filter(
            ee.Filter.gt(
                "common_valid_pixels",
                0,
            )
        )
    )

    return set(
        valid_collection
        .aggregate_array("date_key")
        .distinct()
        .getInfo()
    )

In [26]:
# ============================================================
# Common-band availability by station
# ============================================================

station_list = (
    station_footprints
    .toList(
        station_footprints.size()
    )
)

station_count = (
    station_footprints
    .size()
    .getInfo()
)

availability_rows = []


for index in range(station_count):

    footprint = ee.Feature(
        station_list.get(index)
    )

    station_name = (
        footprint
        .get("station")
        .getInfo()
    )

    print(
        "Processing:",
        station_name,
    )

    local_tiles = (
        get_local_mgrs_tiles(
            footprint
        )
    )

    s2_dates = (
        get_s2_common_valid_dates(
            footprint
        )
    )

    s30_dates = (
        get_hls_common_valid_dates(
            footprint=footprint,
            collection=hls_s30_tiled,
            local_mgrs_tiles=local_tiles,
            sensor="S30",
            exclude_high_aerosol=True,
        )
    )

    l30_dates = (
        get_hls_common_valid_dates(
            footprint=footprint,
            collection=hls_l30_tiled,
            local_mgrs_tiles=local_tiles,
            sensor="L30",
            exclude_high_aerosol=False,
        )
    )

    # Sensitivity analysis only.
    l30_no_high_aerosol_dates = (
        get_hls_common_valid_dates(
            footprint=footprint,
            collection=hls_l30_tiled,
            local_mgrs_tiles=local_tiles,
            sensor="L30",
            exclude_high_aerosol=True,
        )
    )

    combined_dates = (
        s30_dates
        | l30_dates
    )

    availability_rows.append(
        {
            "station": station_name,
            "S2": len(s2_dates),
            "HLS_S30": len(s30_dates),
            "HLS_L30": len(l30_dates),
            "HLS_L30_no_high_aerosol": len(
                l30_no_high_aerosol_dates
            ),
            "HLS_COMBINED": len(
                combined_dates
            ),
            "L30_added_to_S30": len(
                l30_dates - s30_dates
            ),
            "S30_L30_shared": len(
                s30_dates & l30_dates
            ),
        }
    )


common_availability = pd.DataFrame(
    availability_rows
)

display(common_availability)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Processing: Pastos limpios
Processing: Palma
Processing: Bananera
Processing: Manglar
Processing: Bosque seco


,station,S2,HLS_S30,HLS_L30,HLS_L30_no_high_aerosol,HLS_COMBINED,L30_added_to_S30,S30_L30_shared
0,Pastos limpios,155,90,53,49,136,46,7
1,Palma,158,101,51,46,145,44,7
2,Bananera,161,81,52,44,126,45,7
3,Manglar,158,125,64,59,182,57,7
4,Bosque seco,165,112,61,57,164,52,9


In [27]:
# ============================================================
# MODIS 8-day periods
# ============================================================

from datetime import date, timedelta


modis_collection = ee.ImageCollection(
    modis_inputs["collection"]
)

period_millis = (
    modis_collection
    .aggregate_array(
        "system:time_start"
    )
    .getInfo()
)

period_starts = sorted(
    {
        pd.to_datetime(
            value,
            unit="ms",
            utc=True,
        ).date()
        for value in period_millis
    }
)


def get_period_end(
    period_start: date,
) -> date:
    regular_end = (
        period_start
        + timedelta(days=8)
    )

    next_year_start = date(
        period_start.year + 1,
        1,
        1,
    )

    return min(
        regular_end,
        next_year_start,
    )


periods = [
    (
        period_start,
        get_period_end(
            period_start
        ),
    )
    for period_start in period_starts
]

print(
    "MODIS periods:",
    len(periods),
)

print(
    "First period:",
    periods[0],
)

print(
    "Last period:",
    periods[-1],
)

assert len(periods) == 138

MODIS periods: 138
First period: (datetime.date(2021, 1, 1), datetime.date(2021, 1, 9))
Last period: (datetime.date(2023, 12, 27), datetime.date(2024, 1, 1))


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


In [28]:
# ============================================================
# Convert valid dates to MODIS-period availability
# ============================================================

def parse_date_set(
    date_strings,
):
    return {
        pd.to_datetime(
            value
        ).date()
        for value in date_strings
    }


def get_period_date_counts(
    valid_dates,
):
    valid_dates = parse_date_set(
        valid_dates
    )

    counts = []

    for (
        period_start,
        period_end,
    ) in periods:

        count = sum(
            period_start
            <= observation_date
            < period_end
            for observation_date
            in valid_dates
        )

        counts.append(
            count
        )

    return counts


def get_available_periods(
    valid_dates,
):
    counts = get_period_date_counts(
        valid_dates
    )

    return {
        index
        for index, count
        in enumerate(counts)
        if count > 0
    }

In [29]:
# ============================================================
# Optical availability by MODIS period
# ============================================================

station_list = (
    station_footprints
    .toList(
        station_footprints.size()
    )
)

station_count = (
    station_footprints
    .size()
    .getInfo()
)

period_availability_rows = []


for index in range(
    station_count
):

    footprint = ee.Feature(
        station_list.get(
            index
        )
    )

    station_name = (
        footprint
        .get("station")
        .getInfo()
    )

    print(
        "Processing:",
        station_name,
    )

    local_tiles = (
        get_local_mgrs_tiles(
            footprint
        )
    )

    # --------------------------------------------------------
    # Sentinel-2
    # --------------------------------------------------------

    s2_dates = (
        get_s2_common_valid_dates(
            footprint
        )
    )

    # --------------------------------------------------------
    # HLS-S30
    # --------------------------------------------------------

    s30_dates = (
        get_hls_common_valid_dates(
            footprint=footprint,
            collection=hls_s30_tiled,
            local_mgrs_tiles=local_tiles,
            sensor="S30",
            exclude_high_aerosol=True,
        )
    )

    # --------------------------------------------------------
    # HLS-L30
    # --------------------------------------------------------

    l30_dates = (
        get_hls_common_valid_dates(
            footprint=footprint,
            collection=hls_l30_tiled,
            local_mgrs_tiles=local_tiles,
            sensor="L30",
            exclude_high_aerosol=False,
        )
    )

    combined_dates = (
        s30_dates
        | l30_dates
    )

    # --------------------------------------------------------
    # Convert dates to MODIS periods
    # --------------------------------------------------------

    s2_periods = (
        get_available_periods(
            s2_dates
        )
    )

    s30_periods = (
        get_available_periods(
            s30_dates
        )
    )

    l30_periods = (
        get_available_periods(
            l30_dates
        )
    )

    combined_periods = (
        get_available_periods(
            combined_dates
        )
    )

    # Periods that become available only because L30 exists.
    l30_rescued_periods = (
        combined_periods
        - s30_periods
    )

    # Shared availability between the two main alternatives.
    s2_hls_shared_periods = (
        s2_periods
        & combined_periods
    )

    period_availability_rows.append(
        {
            "station":
                station_name,

            "S2_periods":
                len(
                    s2_periods
                ),

            "HLS_S30_periods":
                len(
                    s30_periods
                ),

            "HLS_L30_periods":
                len(
                    l30_periods
                ),

            "HLS_COMBINED_periods":
                len(
                    combined_periods
                ),

            "L30_rescued_periods":
                len(
                    l30_rescued_periods
                ),

            "S2_HLS_shared_periods":
                len(
                    s2_hls_shared_periods
                ),

            "S2_missing_periods":
                len(periods)
                - len(s2_periods),

            "HLS_COMBINED_missing_periods":
                len(periods)
                - len(combined_periods),
        }
    )


period_availability = pd.DataFrame(
    period_availability_rows
)

display(
    period_availability
)

Processing: Pastos limpios
Processing: Palma
Processing: Bananera
Processing: Manglar
Processing: Bosque seco


,station,S2_periods,HLS_S30_periods,HLS_L30_periods,HLS_COMBINED_periods,L30_rescued_periods,S2_HLS_shared_periods,S2_missing_periods,HLS_COMBINED_missing_periods
0,Pastos limpios,114,75,53,87,12,84,24,51
1,Palma,119,86,51,96,10,95,19,42
2,Bananera,118,69,52,87,18,83,20,51
3,Manglar,116,98,64,114,16,106,22,24
4,Bosque seco,120,91,61,103,12,98,18,35


In [30]:
# ============================================================
# Common optical valid masks
# ============================================================

from et_downscaling.config import (
    S2_CLEAR_THRESHOLD,
    S2_QA_BAND,
)


def build_s2_common_valid_mask(image):
    """
    Build the common-band Sentinel-2 valid mask at 20 m.

    Common comparison bands:
    Blue, Green, Red, NIR, SWIR1, SWIR2.

    The 10 m validity and Cloud Score+ masks are aggregated to
    the native 20 m B8A grid using a strict all-subpixels rule.
    """
    image = ee.Image(image)

    reference_projection = (
        image
        .select("B8A")
        .projection()
    )

    valid_10m_spectral = (
        image
        .select(
            [
                "B2",
                "B3",
                "B4",
            ]
        )
        .mask()
        .reduce(
            ee.Reducer.min()
        )
    )

    valid_qa = (
        image
        .select(S2_QA_BAND)
        .mask()
    )

    clear_10m = (
        image
        .select(S2_QA_BAND)
        .gte(S2_CLEAR_THRESHOLD)
    )

    valid_clear_10m = (
        valid_10m_spectral
        .And(valid_qa)
        .And(clear_10m)
        .unmask(0)
    )

    # Require all four nested 10 m pixels to be valid and clear.
    valid_clear_20m = (
        valid_clear_10m
        .reduceResolution(
            reducer=ee.Reducer.min(),
            maxPixels=4,
        )
        .reproject(
            reference_projection
        )
        .eq(1)
    )

    valid_native_20m = (
        image
        .select(
            [
                "B8A",
                "B11",
                "B12",
            ]
        )
        .mask()
        .reduce(
            ee.Reducer.min()
        )
        .reproject(
            reference_projection
        )
    )

    return (
        valid_clear_20m
        .And(valid_native_20m)
        .rename("valid")
        .uint8()
    )


def build_hls_common_valid_mask(
    image,
    sensor,
    exclude_high_aerosol=False,
):
    """
    Build an HLS common-band valid mask at 30 m.
    """
    image = ee.Image(image)

    if sensor == "S30":
        source_bands = [
            "B2",
            "B3",
            "B4",
            "B8A",
            "B11",
            "B12",
        ]

    elif sensor == "L30":
        source_bands = [
            "B2",
            "B3",
            "B4",
            "B5",
            "B6",
            "B7",
        ]

    else:
        raise ValueError(
            f"Unsupported HLS sensor: {sensor}"
        )

    spectral_valid = (
        image
        .select(source_bands)
        .mask()
        .reduce(
            ee.Reducer.min()
        )
    )

    fmask = image.select("Fmask")

    no_cloud = (
        fmask
        .bitwiseAnd(1 << 1)
        .eq(0)
    )

    no_adjacent = (
        fmask
        .bitwiseAnd(1 << 2)
        .eq(0)
    )

    no_shadow = (
        fmask
        .bitwiseAnd(1 << 3)
        .eq(0)
    )

    no_snow = (
        fmask
        .bitwiseAnd(1 << 4)
        .eq(0)
    )

    valid = (
        spectral_valid
        .And(no_cloud)
        .And(no_adjacent)
        .And(no_shadow)
        .And(no_snow)
    )

    if exclude_high_aerosol:
        aerosol_level = (
            fmask
            .rightShift(6)
            .bitwiseAnd(3)
        )

        valid = (
            valid
            .And(
                aerosol_level.neq(3)
            )
        )

    return (
        valid
        .unmask(0)
        .rename("valid")
        .uint8()
    )

In [31]:
# ============================================================
# Union coverage inside a MODIS footprint
# ============================================================

def build_valid_union(
    collection,
    mask_function,
):
    """
    Return the spatial union of valid pixels from all images.

    A zero-valued fallback guarantees a valid output when the
    period contains no observations.
    """
    valid_collection = (
        ee.ImageCollection(collection)
        .map(mask_function)
    )

    fallback = (
        ee.Image.constant(0)
        .rename("valid")
        .uint8()
    )

    return (
        valid_collection
        .merge(
            ee.ImageCollection(
                [fallback]
            )
        )
        .max()
        .rename("valid")
        .uint8()
    )


def calculate_area_coverage_pct(
    valid_union,
    geometry,
    scale_m,
):
    """
    Calculate area-weighted valid coverage over the footprint.
    """
    geometry = ee.Geometry(geometry)

    valid_area = (
        ee.Image.pixelArea()
        .multiply(
            ee.Image(valid_union)
        )
        .rename("valid_area")
    )

    covered_area_m2 = ee.Number(
        valid_area
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=geometry,
            scale=scale_m,
            maxPixels=1e7,
        )
        .get("valid_area")
    )

    footprint_area_m2 = (
        geometry.area(1)
    )

    return (
        covered_area_m2
        .divide(
            footprint_area_m2
        )
        .multiply(100)
    )

In [32]:
# ============================================================
# Period-level common optical coverage
# ============================================================

coverage_rows = []

station_list = (
    station_footprints
    .toList(
        station_footprints.size()
    )
)

station_count = (
    station_footprints
    .size()
    .getInfo()
)


for station_index in range(
    station_count
):

    footprint = ee.Feature(
        station_list.get(
            station_index
        )
    )

    station_name = (
        footprint
        .get("station")
        .getInfo()
    )

    geometry = (
        footprint.geometry()
    )

    print(
        "Processing:",
        station_name,
    )

    local_tiles = (
        get_local_mgrs_tiles(
            footprint
        )
    )

    local_s30 = (
        hls_s30_tiled
        .filter(
            ee.Filter.inList(
                "hls_mgrs_tile",
                local_tiles,
            )
        )
    )

    local_l30 = (
        hls_l30_tiled
        .filter(
            ee.Filter.inList(
                "hls_mgrs_tile",
                local_tiles,
            )
        )
    )

    station_features = []


    for (
        period_start,
        period_end,
    ) in periods:

        start_string = (
            period_start.isoformat()
        )

        end_string = (
            period_end.isoformat()
        )

        # ----------------------------------------------------
        # Sentinel-2
        # ----------------------------------------------------

        s2_period = (
            s2_collection
            .filterBounds(geometry)
            .filterDate(
                start_string,
                end_string,
            )
        )

        s2_union = (
            build_valid_union(
                s2_period,
                build_s2_common_valid_mask,
            )
        )

        s2_coverage = (
            calculate_area_coverage_pct(
                s2_union,
                geometry,
                20,
            )
        )

        # ----------------------------------------------------
        # HLS-S30
        # ----------------------------------------------------

        s30_period = (
            local_s30
            .filterDate(
                start_string,
                end_string,
            )
        )

        s30_union = (
            build_valid_union(
                s30_period,
                lambda image:
                    build_hls_common_valid_mask(
                        image,
                        sensor="S30",
                        exclude_high_aerosol=True,
                    ),
            )
        )

        s30_coverage = (
            calculate_area_coverage_pct(
                s30_union,
                geometry,
                30,
            )
        )

        # ----------------------------------------------------
        # HLS-L30
        # ----------------------------------------------------

        l30_period = (
            local_l30
            .filterDate(
                start_string,
                end_string,
            )
        )

        l30_union = (
            build_valid_union(
                l30_period,
                lambda image:
                    build_hls_common_valid_mask(
                        image,
                        sensor="L30",
                        exclude_high_aerosol=False,
                    ),
            )
        )

        l30_coverage = (
            calculate_area_coverage_pct(
                l30_union,
                geometry,
                30,
            )
        )

        # ----------------------------------------------------
        # HLS combined
        #
        # Union is performed at pixel level, not by simply
        # adding S30 and L30 date counts.
        # ----------------------------------------------------

        combined_union = (
            ee.ImageCollection(
                [
                    s30_union,
                    l30_union,
                ]
            )
            .max()
            .rename("valid")
        )

        combined_coverage = (
            calculate_area_coverage_pct(
                combined_union,
                geometry,
                30,
            )
        )

        # ----------------------------------------------------
        # One feature per source
        # ----------------------------------------------------

        source_values = [
            (
                "S2",
                20,
                s2_coverage,
            ),
            (
                "HLS_S30",
                30,
                s30_coverage,
            ),
            (
                "HLS_L30",
                30,
                l30_coverage,
            ),
            (
                "HLS_COMBINED",
                30,
                combined_coverage,
            ),
        ]

        for (
            source,
            scale_m,
            coverage,
        ) in source_values:

            station_features.append(
                ee.Feature(
                    None,
                    {
                        "station":
                            station_name,

                        "period_start":
                            start_string,

                        "period_end":
                            end_string,

                        "source":
                            source,

                        "scale_m":
                            scale_m,

                        "coverage_pct":
                            coverage,
                    },
                )
            )

    # --------------------------------------------------------
    # Evaluate one station at a time
    # --------------------------------------------------------

    station_collection = (
        ee.FeatureCollection(
            station_features
        )
    )

    station_info = (
        station_collection
        .getInfo()
    )

    station_rows = [
        feature["properties"]
        for feature
        in station_info["features"]
    ]

    coverage_rows.extend(
        station_rows
    )

    print(
        "  rows:",
        len(station_rows),
    )

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Processing: Pastos limpios
  rows: 552
Processing: Palma
  rows: 552
Processing: Bananera
  rows: 552
Processing: Manglar
  rows: 552
Processing: Bosque seco
  rows: 552


In [33]:
optical_period_coverage = pd.DataFrame(
    coverage_rows
)

optical_period_coverage[
    "coverage_pct"
] = pd.to_numeric(
    optical_period_coverage[
        "coverage_pct"
    ],
    errors="coerce",
)

print(
    "Rows:",
    len(optical_period_coverage),
)

print(
    "Expected:",
    5 * 138 * 4,
)

display(
    optical_period_coverage.head()
)

assert (
    len(optical_period_coverage)
    == 2760
)

assert (
    optical_period_coverage[
        "coverage_pct"
    ]
    .between(
        0,
        100.5,
    )
    .all()
)

Rows: 2760
Expected: 2760


,coverage_pct,period_end,period_start,scale_m,source,station
0,99.304466,2021-01-09,2021-01-01,20,S2,Pastos limpios
1,99.822055,2021-01-09,2021-01-01,30,HLS_S30,Pastos limpios
2,0.000000,2021-01-09,2021-01-01,30,HLS_L30,Pastos limpios
3,99.822055,2021-01-09,2021-01-01,30,HLS_COMBINED,Pastos limpios
4,99.304466,2021-01-17,2021-01-09,20,S2,Pastos limpios


In [34]:
coverage_thresholds = [
    0,
    50,
    70,
    80,
    90,
    99,
]


summary_rows = []


for (
    station,
    source,
), group in (
    optical_period_coverage
    .groupby(
        [
            "station",
            "source",
        ]
    )
):

    row = {
        "station": station,
        "source": source,
        "periods": len(group),
        "mean_coverage": (
            group["coverage_pct"].mean()
        ),
        "median_coverage": (
            group["coverage_pct"].median()
        ),
    }

    for threshold in coverage_thresholds:

        if threshold == 0:
            count = (
                group["coverage_pct"]
                .gt(0)
                .sum()
            )

            column_name = (
                "periods_gt_0"
            )

        else:
            count = (
                group["coverage_pct"]
                .ge(threshold)
                .sum()
            )

            column_name = (
                f"periods_ge_{threshold}"
            )

        row[column_name] = int(
            count
        )

    summary_rows.append(
        row
    )


coverage_summary = pd.DataFrame(
    summary_rows
)

display(
    coverage_summary
)

,station,source,periods,mean_coverage,median_coverage,periods_gt_0,periods_ge_50,periods_ge_70,periods_ge_80,periods_ge_90,periods_ge_99
0,Bananera,HLS_COMBINED,138,49.013911,52.851213,87,69,63,59,53,48
1,Bananera,HLS_L30,138,29.224186,0.000000,52,41,39,37,34,33
2,Bananera,HLS_S30,138,37.709612,0.004818,69,54,49,45,40,28
3,Bananera,S2,138,72.158393,99.288740,118,104,94,89,86,79
4,Bosque seco,HLS_COMBINED,138,64.838187,99.819940,103,89,86,85,83,79
5,Bosque seco,HLS_L30,138,39.593553,0.000000,61,54,53,53,50,50
6,Bosque seco,HLS_S30,138,53.551947,73.976419,91,75,70,67,62,57
7,Bosque seco,S2,138,79.743930,99.303323,120,113,108,106,102,95
8,Manglar,HLS_COMBINED,138,75.468691,99.020272,114,105,102,102,96,91
9,Manglar,HLS_L30,138,40.691397,0.000000,64,56,55,55,52,52


In [35]:
# ============================================================
# Temporal distribution of optical coverage
# ============================================================

temporal_coverage = (
    optical_period_coverage
    .copy()
)

temporal_coverage[
    "period_start"
] = pd.to_datetime(
    temporal_coverage[
        "period_start"
    ]
)

temporal_coverage[
    "year"
] = (
    temporal_coverage[
        "period_start"
    ]
    .dt.year
)

temporal_coverage[
    "month"
] = (
    temporal_coverage[
        "period_start"
    ]
    .dt.month
)


# Keep the two main candidates.
main_candidates = (
    temporal_coverage[
        temporal_coverage[
            "source"
        ].isin(
            [
                "S2",
                "HLS_COMBINED",
            ]
        )
    ]
    .copy()
)


# ============================================================
# Annual summary
# ============================================================

annual_summary = (
    main_candidates
    .groupby(
        [
            "source",
            "year",
        ],
        as_index=False,
    )
    .agg(
        periods=(
            "coverage_pct",
            "size",
        ),
        mean_coverage=(
            "coverage_pct",
            "mean",
        ),
        median_coverage=(
            "coverage_pct",
            "median",
        ),
        periods_ge_80=(
            "coverage_pct",
            lambda x: int(
                (x >= 80).sum()
            ),
        ),
        periods_ge_90=(
            "coverage_pct",
            lambda x: int(
                (x >= 90).sum()
            ),
        ),
        periods_ge_99=(
            "coverage_pct",
            lambda x: int(
                (x >= 99).sum()
            ),
        ),
    )
)

display(
    annual_summary
)


# ============================================================
# Monthly/climatological summary
# ============================================================

monthly_summary = (
    main_candidates
    .groupby(
        [
            "source",
            "month",
        ],
        as_index=False,
    )
    .agg(
        periods=(
            "coverage_pct",
            "size",
        ),
        mean_coverage=(
            "coverage_pct",
            "mean",
        ),
        median_coverage=(
            "coverage_pct",
            "median",
        ),
        periods_ge_80=(
            "coverage_pct",
            lambda x: int(
                (x >= 80).sum()
            ),
        ),
        periods_ge_90=(
            "coverage_pct",
            lambda x: int(
                (x >= 90).sum()
            ),
        ),
        periods_ge_99=(
            "coverage_pct",
            lambda x: int(
                (x >= 99).sum()
            ),
        ),
    )
)

display(
    monthly_summary
)

,source,year,periods,mean_coverage,median_coverage,periods_ge_80,periods_ge_90,periods_ge_99
0,HLS_COMBINED,2021,230,62.367794,98.065103,133,124,115
1,HLS_COMBINED,2022,230,49.219403,52.819627,101,95,91
2,HLS_COMBINED,2023,230,66.833023,99.020272,147,142,130
3,S2,2021,230,72.564360,99.295049,158,151,141
4,S2,2022,230,69.275297,99.288740,150,141,126
5,S2,2023,230,85.005593,99.301358,189,185,173


,source,month,periods,mean_coverage,median_coverage,periods_ge_80,periods_ge_90,periods_ge_99
0,HLS_COMBINED,1,60,95.641566,99.819940,56,56,52
1,HLS_COMBINED,2,60,98.486673,99.819940,60,56,52
2,HLS_COMBINED,3,60,54.881753,80.767842,31,27,25
3,HLS_COMBINED,4,45,42.516658,11.728224,16,14,13
4,HLS_COMBINED,5,60,48.049617,37.553136,24,23,21
5,HLS_COMBINED,6,60,43.342120,13.720982,22,21,19
6,HLS_COMBINED,7,60,65.311999,99.020272,36,33,32
7,HLS_COMBINED,8,60,37.122753,1.943151,18,17,15
8,HLS_COMBINED,9,60,43.493088,7.118997,25,23,21
9,HLS_COMBINED,10,45,32.217966,0.000000,12,12,10


In [9]:
# ============================================================
# Controlled common optical predictor configuration
# ============================================================

COMMON_REFLECTANCE_BANDS = [
    "Blue",
    "Green",
    "Red",
    "NIR",
    "SWIR1",
    "SWIR2",
]

COMMON_INDEX_BANDS = [
    "NDVI",
    "EVI",
    "SAVI",
    "NDWI",
    "NDMI",
]

COMMON_PREDICTOR_BANDS = (
    COMMON_REFLECTANCE_BANDS
    + COMMON_INDEX_BANDS
)

COMMON_MEDOID_SCORE_BANDS = (
    COMMON_REFLECTANCE_BANDS.copy()
)


def safe_ratio(
    numerator,
    denominator,
    output_name,
    epsilon=1e-6,
):
    numerator = ee.Image(
        numerator
    )

    denominator = ee.Image(
        denominator
    )

    return (
        numerator
        .divide(denominator)
        .updateMask(
            denominator
            .abs()
            .gt(epsilon)
        )
        .rename(output_name)
        .toFloat()
    )


def add_common_indices(image):
    image = ee.Image(
        image
    )

    blue = image.select("Blue")
    green = image.select("Green")
    red = image.select("Red")
    nir = image.select("NIR")
    swir1 = image.select("SWIR1")

    ndvi = safe_ratio(
        nir.subtract(red),
        nir.add(red),
        "NDVI",
    )

    evi = safe_ratio(
        nir
        .subtract(red)
        .multiply(2.5),
        nir
        .add(
            red.multiply(6.0)
        )
        .subtract(
            blue.multiply(7.5)
        )
        .add(1.0),
        "EVI",
    )

    savi = safe_ratio(
        nir
        .subtract(red)
        .multiply(1.5),
        nir
        .add(red)
        .add(0.5),
        "SAVI",
    )

    ndwi = safe_ratio(
        green.subtract(nir),
        green.add(nir),
        "NDWI",
    )

    ndmi = safe_ratio(
        nir.subtract(swir1),
        nir.add(swir1),
        "NDMI",
    )

    return (
        image
        .addBands(
            [
                ndvi,
                evi,
                savi,
                ndwi,
                ndmi,
            ]
        )
        .select(
            COMMON_PREDICTOR_BANDS
        )
    )

In [10]:
# ============================================================
# Controlled Sentinel-2 common-band preparation
# ============================================================

def prepare_s2_common(image):
    image = ee.Image(
        image
    )

    reference_projection = (
        image
        .select("B8A")
        .projection()
    )

    reflectance_10m = (
        image
        .select(
            [
                "B2",
                "B3",
                "B4",
            ],
            [
                "Blue",
                "Green",
                "Red",
            ],
        )
        .multiply(0.0001)
        .toFloat()
    )

    reflectance_20m_from_10m = (
        reflectance_10m
        .reduceResolution(
            reducer=ee.Reducer.mean(),
            maxPixels=4,
        )
        .reproject(
            reference_projection
        )
    )

    reflectance_native_20m = (
        image
        .select(
            [
                "B8A",
                "B11",
                "B12",
            ],
            [
                "NIR",
                "SWIR1",
                "SWIR2",
            ],
        )
        .multiply(0.0001)
        .reproject(
            reference_projection
        )
        .toFloat()
    )

    valid_mask = (
        build_s2_common_valid_mask(
            image
        )
    )

    prepared = (
        reflectance_20m_from_10m
        .addBands(
            reflectance_native_20m
        )
        .select(
            COMMON_REFLECTANCE_BANDS
        )
        .updateMask(
            valid_mask
        )
        .toFloat()
    )

    return ee.Image(
        prepared
        .copyProperties(
            image,
            [
                "system:time_start",
                "system:index",
                "date_key",
                "MGRS_TILE",
            ],
        )
    )

In [11]:
# ============================================================
# Controlled HLS common-band preparation
# ============================================================

def prepare_hls_common(
    image,
    sensor,
    exclude_high_aerosol=False,
):
    image = ee.Image(
        image
    )

    if sensor == "S30":

        source_bands = [
            "B2",
            "B3",
            "B4",
            "B8A",
            "B11",
            "B12",
        ]

    elif sensor == "L30":

        source_bands = [
            "B2",
            "B3",
            "B4",
            "B5",
            "B6",
            "B7",
        ]

    else:
        raise ValueError(
            f"Unsupported HLS sensor: {sensor}"
        )

    reflectance = (
        image
        .select(
            source_bands,
            COMMON_REFLECTANCE_BANDS,
        )
        .toFloat()
    )

    valid_mask = (
        build_hls_common_valid_mask(
            image=image,
            sensor=sensor,
            exclude_high_aerosol=(
                exclude_high_aerosol
            ),
        )
    )

    prepared = (
        reflectance
        .updateMask(
            valid_mask
        )
        .toFloat()
    )

    return ee.Image(
        prepared
        .copyProperties(
            image,
            [
                "system:time_start",
                "system:index",
                "date_key",
                "sensor",
                "hls_sensor",
                "hls_mgrs_tile",
            ],
        )
    )

In [12]:
# ============================================================
# Controlled common-band temporal medoid
# ============================================================

def build_empty_common_image():
    return (
        ee.Image.constant(
            [0]
            * len(
                COMMON_REFLECTANCE_BANDS
            )
        )
        .rename(
            COMMON_REFLECTANCE_BANDS
        )
        .updateMask(
            ee.Image.constant(0)
        )
        .toFloat()
    )


def build_daily_common_collection(
    collection,
    prepare_function,
    geometry,
):
    collection = ee.ImageCollection(
        collection
    )

    date_keys = (
        ee.List(
            collection
            .aggregate_array(
                "date_key"
            )
        )
        .distinct()
        .sort()
    )

    def build_daily_image(
        date_key,
    ):
        date_key = ee.String(
            date_key
        )

        same_date = (
            collection
            .filter(
                ee.Filter.eq(
                    "date_key",
                    date_key,
                )
            )
            .map(
                prepare_function
            )
        )

        return (
            same_date
            .mosaic()
            .clip(
                ee.Geometry(
                    geometry
                )
                .buffer(100)
            )
            .set(
                "date_key",
                date_key,
            )
        )

    return (
        ee.ImageCollection
        .fromImages(
            date_keys.map(
                build_daily_image
            )
        )
    )


def build_common_medoid(
    daily_collection,
):
    daily_collection = (
        ee.ImageCollection(
            daily_collection
        )
    )

    image_count = (
        daily_collection.size()
    )

    median_image = (
        daily_collection
        .select(
            COMMON_MEDOID_SCORE_BANDS
        )
        .median()
    )

    def add_medoid_score(
        image,
    ):
        image = ee.Image(
            image
        )

        distance = (
            image
            .select(
                COMMON_MEDOID_SCORE_BANDS
            )
            .subtract(
                median_image
            )
            .pow(2)
            .reduce(
                ee.Reducer.sum()
            )
        )

        medoid_score = (
            distance
            .multiply(-1)
            .rename(
                "medoid_score"
            )
        )

        return (
            image
            .addBands(
                medoid_score
            )
        )

    selected = (
        daily_collection
        .map(
            add_medoid_score
        )
        .qualityMosaic(
            "medoid_score"
        )
        .select(
            COMMON_REFLECTANCE_BANDS
        )
    )

    return ee.Image(
        ee.Algorithms.If(
            image_count.gt(0),
            selected,
            build_empty_common_image(),
        )
    )

In [13]:
# ============================================================
# Controlled HLS combined daily collection
# ============================================================

def build_hls_combined_common_daily_collection(
    s30_period,
    l30_period,
    geometry,
):
    s30_daily = (
        build_daily_common_collection(
            collection=s30_period,
            prepare_function=(
                lambda image:
                    prepare_hls_common(
                        image=image,
                        sensor="S30",
                        exclude_high_aerosol=True,
                    )
            ),
            geometry=geometry,
        )
    )

    l30_daily = (
        build_daily_common_collection(
            collection=l30_period,
            prepare_function=(
                lambda image:
                    prepare_hls_common(
                        image=image,
                        sensor="L30",
                        exclude_high_aerosol=False,
                    )
            ),
            geometry=geometry,
        )
    )

    return (
        s30_daily
        .merge(
            l30_daily
        )
        .sort(
            "date_key"
        )
    )

In [14]:
# ============================================================
# Notebook environment
# ============================================================

import ee
import pandas as pd

EE_PROJECT = "ee-change"

ee.Initialize(
    project=EE_PROJECT
)

print(
    "Earth Engine initialized with project:",
    EE_PROJECT,
)

Earth Engine initialized with project: ee-change


In [15]:
# ============================================================
# Single footprint-period validation
# ============================================================

test_footprint = ee.Feature(
    station_footprints
    .filter(
        ee.Filter.eq(
            "station",
            "Bananera",
        )
    )
    .first()
)

test_geometry = (
    test_footprint.geometry()
)

test_period_start = periods[0][0]
test_period_end = periods[0][1]

test_start = (
    test_period_start.isoformat()
)

test_end = (
    test_period_end.isoformat()
)

local_tiles = (
    get_local_mgrs_tiles(
        test_footprint
    )
)


# Sentinel-2
s2_test_period = (
    s2_collection
    .filterBounds(
        test_geometry
    )
    .filterDate(
        test_start,
        test_end,
    )
)

s2_test_daily = (
    build_daily_common_collection(
        collection=s2_test_period,
        prepare_function=prepare_s2_common,
        geometry=test_geometry,
    )
)

s2_test_medoid = (
    build_common_medoid(
        s2_test_daily
    )
)

s2_test_predictors = (
    add_common_indices(
        s2_test_medoid
    )
)


# HLS-S30
s30_test_period = (
    hls_s30_tiled
    .filter(
        ee.Filter.inList(
            "hls_mgrs_tile",
            local_tiles,
        )
    )
    .filterDate(
        test_start,
        test_end,
    )
)


# HLS-L30
l30_test_period = (
    hls_l30_tiled
    .filter(
        ee.Filter.inList(
            "hls_mgrs_tile",
            local_tiles,
        )
    )
    .filterDate(
        test_start,
        test_end,
    )
)


hls_test_daily = (
    build_hls_combined_common_daily_collection(
        s30_period=s30_test_period,
        l30_period=l30_test_period,
        geometry=test_geometry,
    )
)

hls_test_medoid = (
    build_common_medoid(
        hls_test_daily
    )
)

hls_test_predictors = (
    add_common_indices(
        hls_test_medoid
    )
)


print(
    "Period:",
    test_start,
    "->",
    test_end,
)

print(
    "S2 raw products:",
    s2_test_period.size().getInfo(),
)

print(
    "S2 daily candidates:",
    s2_test_daily.size().getInfo(),
)

print(
    "HLS S30 raw products:",
    s30_test_period.size().getInfo(),
)

print(
    "HLS L30 raw products:",
    l30_test_period.size().getInfo(),
)

print(
    "HLS daily candidates:",
    hls_test_daily.size().getInfo(),
)

NameError: name 'station_footprints' is not defined

In [1]:
# ============================================================
# Clean session bootstrap
# ============================================================

from datetime import date, timedelta
from pathlib import Path
import importlib
import sys

import ee
import numpy as np
import pandas as pd


# ============================================================
# Repository
# ============================================================

def find_repo_root(start=None):
    current = Path(
        start or Path.cwd()
    ).resolve()

    for candidate in [
        current,
        *current.parents,
    ]:
        if (
            candidate
            / "pyproject.toml"
        ).exists():
            return candidate

    raise FileNotFoundError(
        "Repository root not found."
    )


REPO_ROOT = find_repo_root()

SRC_PATH = (
    REPO_ROOT
    / "src"
)

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )


print(
    "Repository:",
    REPO_ROOT,
)


# ============================================================
# Earth Engine
# ============================================================

EE_PROJECT = "ee-change"

ee.Initialize(
    project=EE_PROJECT
)

ee.Number(1).getInfo()

print(
    "Earth Engine project:",
    EE_PROJECT,
)


# ============================================================
# Current repository modules
# ============================================================

import et_downscaling.hls as hls_module
import et_downscaling.modis as modis_module
import et_downscaling.sentinel2 as sentinel2_module

importlib.reload(
    hls_module
)

importlib.reload(
    modis_module
)

importlib.reload(
    sentinel2_module
)


# ============================================================
# MODIS inputs and station footprints
# ============================================================

modis_inputs = (
    modis_module
    .build_modis_inputs()
)

modis_collection = (
    ee.ImageCollection(
        modis_inputs[
            "collection"
        ]
    )
)

station_footprints = (
    ee.FeatureCollection(
        modis_inputs[
            "station_footprints"
        ]
    )
)

analysis_geometry = (
    station_footprints
    .geometry()
)


# ============================================================
# Station metadata
# ============================================================

station_features = (
    station_footprints
    .select(
        [
            "station",
            "station_id",
            "longitude",
            "latitude",
            "footprint_area_m2",
        ]
    )
    .getInfo()[
        "features"
    ]
)

stations = [
    feature[
        "properties"
    ]
    for feature
    in station_features
]


# ============================================================
# Optical collections
# ============================================================

s2_collection = (
    sentinel2_module
    .get_sentinel2_collection(
        station_footprints
    )
)

hls_collection = (
    hls_module
    .get_hls_collection(
        station_footprints
    )
)

hls_s30_collection = (
    hls_collection
    .filter(
        ee.Filter.eq(
            "sensor",
            "S30",
        )
    )
)

hls_l30_collection = (
    hls_collection
    .filter(
        ee.Filter.eq(
            "sensor",
            "L30",
        )
    )
)


# ============================================================
# Standardize HLS MGRS identifier
# ============================================================

def add_hls_mgrs_tile(
    image,
):
    image = ee.Image(
        image
    )

    system_index = ee.String(
        image.get(
            "system:index"
        )
    )

    tile_token = ee.String(
        system_index
        .split("_")
        .get(1)
    )

    mgrs_tile = (
        tile_token
        .slice(1)
    )

    return image.set(
        "hls_mgrs_tile",
        mgrs_tile,
    )


hls_s30_tiled = (
    hls_s30_collection
    .map(
        add_hls_mgrs_tile
    )
)

hls_l30_tiled = (
    hls_l30_collection
    .map(
        add_hls_mgrs_tile
    )
)


# ============================================================
# Local MGRS tiles
# ============================================================

def get_local_mgrs_tiles(
    footprint,
):
    footprint = ee.Feature(
        footprint
    )

    local_s2 = (
        ee.ImageCollection(
            "COPERNICUS/S2_SR_HARMONIZED"
        )
        .filterBounds(
            footprint.geometry()
        )
    )

    return (
        local_s2
        .aggregate_array(
            "MGRS_TILE"
        )
        .distinct()
        .getInfo()
    )


# ============================================================
# MODIS periods
# ============================================================

period_millis = (
    modis_collection
    .aggregate_array(
        "system:time_start"
    )
    .getInfo()
)

period_starts = sorted(
    {
        pd.to_datetime(
            value,
            unit="ms",
            utc=True,
        ).date()
        for value
        in period_millis
    }
)


def get_period_end(
    period_start,
):
    regular_end = (
        period_start
        + timedelta(
            days=8
        )
    )

    next_year_start = date(
        period_start.year + 1,
        1,
        1,
    )

    return min(
        regular_end,
        next_year_start,
    )


periods = [
    (
        period_start,
        get_period_end(
            period_start
        ),
    )
    for period_start
    in period_starts
]


# ============================================================
# Session QA
# ============================================================

print()
print(
    "SESSION READY"
)

print(
    "============="
)

print(
    "Stations:",
    len(stations),
)

print(
    [
        station[
            "station"
        ]
        for station
        in stations
    ]
)

print(
    "MODIS periods:",
    len(periods),
)

print(
    "First period:",
    periods[0],
)

print(
    "Last period:",
    periods[-1],
)

print()

for station in stations:

    footprint = (
        station_footprints
        .filter(
            ee.Filter.eq(
                "station",
                station[
                    "station"
                ],
            )
        )
        .first()
    )

    local_tiles = (
        get_local_mgrs_tiles(
            footprint
        )
    )

    print(
        station["station"],
        "-> MGRS:",
        local_tiles,
    )


assert len(stations) == 5
assert len(periods) == 138

Repository: C:\Users\User\Desktop\cristian\GEE\26-et-downscaling-fundacion


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(
c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Earth Engine project: ee-change

SESSION READY
Stations: 5
['Pastos limpios', 'Palma', 'Bananera', 'Manglar', 'Bosque seco']
MODIS periods: 138
First period: (datetime.date(2021, 1, 1), datetime.date(2021, 1, 9))
Last period: (datetime.date(2023, 12, 27), datetime.date(2024, 1, 1))

Pastos limpios -> MGRS: ['18PWS']
Palma -> MGRS: ['18PWS']
Bananera -> MGRS: ['18PWS']
Manglar -> MGRS: ['18PWT', '18PWS']
Bosque seco -> MGRS: ['18PWS']


In [2]:
# ============================================================
# Controlled common optical comparison
# ============================================================

COMMON_REFLECTANCE_BANDS = [
    "Blue",
    "Green",
    "Red",
    "NIR",
    "SWIR1",
    "SWIR2",
]

COMMON_INDEX_BANDS = [
    "NDVI",
    "EVI",
    "SAVI",
    "NDWI",
    "NDMI",
]

COMMON_PREDICTOR_BANDS = (
    COMMON_REFLECTANCE_BANDS
    + COMMON_INDEX_BANDS
)


# ============================================================
# Sentinel-2 common preparation
# ============================================================

def prepare_s2_common(image):
    """
    Prepare only the six optical bands shared with HLS.

    Output grid:
        20 m, defined by Sentinel-2 B8A.

    QA:
        Cloud Score+ using the current repository threshold.

    The 10 m bands are aggregated to 20 m by mean while
    requiring all four nested 10 m pixels to be clear.
    """
    image = ee.Image(image)

    reference_projection = (
        image
        .select("B8A")
        .projection()
    )

    # --------------------------------------------------------
    # 10 m spectral validity + Cloud Score+
    # --------------------------------------------------------

    valid_10m_spectral = (
        image
        .select(
            [
                "B2",
                "B3",
                "B4",
            ]
        )
        .mask()
        .reduce(
            ee.Reducer.min()
        )
    )

    qa_valid = (
        image
        .select(
            sentinel2_module.S2_QA_BAND
        )
        .mask()
    )

    clear_10m = (
        image
        .select(
            sentinel2_module.S2_QA_BAND
        )
        .gte(
            sentinel2_module.S2_CLEAR_THRESHOLD
        )
    )

    valid_clear_10m = (
        valid_10m_spectral
        .And(qa_valid)
        .And(clear_10m)
        .unmask(0)
    )

    valid_clear_20m = (
        valid_clear_10m
        .reduceResolution(
            reducer=ee.Reducer.min(),
            maxPixels=4,
        )
        .reproject(
            reference_projection
        )
        .eq(1)
    )

    # --------------------------------------------------------
    # Native 20 m bands
    # --------------------------------------------------------

    valid_20m_spectral = (
        image
        .select(
            [
                "B8A",
                "B11",
                "B12",
            ]
        )
        .mask()
        .reduce(
            ee.Reducer.min()
        )
        .reproject(
            reference_projection
        )
    )

    common_mask = (
        valid_clear_20m
        .And(valid_20m_spectral)
    )

    # --------------------------------------------------------
    # Reflectance
    # --------------------------------------------------------

    reflectance_10m = (
        image
        .select(
            [
                "B2",
                "B3",
                "B4",
            ],
            [
                "Blue",
                "Green",
                "Red",
            ],
        )
        .multiply(0.0001)
        .toFloat()
    )

    reflectance_20m_from_10m = (
        reflectance_10m
        .reduceResolution(
            reducer=ee.Reducer.mean(),
            maxPixels=4,
        )
        .reproject(
            reference_projection
        )
    )

    reflectance_native_20m = (
        image
        .select(
            [
                "B8A",
                "B11",
                "B12",
            ],
            [
                "NIR",
                "SWIR1",
                "SWIR2",
            ],
        )
        .multiply(0.0001)
        .reproject(
            reference_projection
        )
        .toFloat()
    )

    prepared = (
        reflectance_20m_from_10m
        .addBands(
            reflectance_native_20m
        )
        .select(
            COMMON_REFLECTANCE_BANDS
        )
        .updateMask(
            common_mask
        )
    )

    return ee.Image(
        prepared
        .copyProperties(
            image,
            [
                "system:time_start",
                "system:index",
                "date_key",
                "MGRS_TILE",
            ],
        )
    )


# ============================================================
# HLS common preparation
# ============================================================

def prepare_hls_common(
    image,
    sensor,
):
    """
    Prepare only the six bands shared with Sentinel-2.

    S30:
        high aerosol is excluded.

    L30:
        current Fmask criteria are retained; high-aerosol
        exclusion remains a sensitivity analysis.
    """
    image = ee.Image(image)

    if sensor == "S30":

        source_bands = [
            "B2",
            "B3",
            "B4",
            "B8A",
            "B11",
            "B12",
        ]

    elif sensor == "L30":

        source_bands = [
            "B2",
            "B3",
            "B4",
            "B5",
            "B6",
            "B7",
        ]

    else:
        raise ValueError(
            f"Unsupported HLS sensor: {sensor}"
        )

    reflectance = (
        image
        .select(
            source_bands,
            COMMON_REFLECTANCE_BANDS,
        )
        .toFloat()
    )

    spectral_valid = (
        reflectance
        .mask()
        .reduce(
            ee.Reducer.min()
        )
    )

    fmask = image.select("Fmask")

    no_cloud = (
        fmask
        .bitwiseAnd(1 << 1)
        .eq(0)
    )

    no_adjacent = (
        fmask
        .bitwiseAnd(1 << 2)
        .eq(0)
    )

    no_shadow = (
        fmask
        .bitwiseAnd(1 << 3)
        .eq(0)
    )

    no_snow = (
        fmask
        .bitwiseAnd(1 << 4)
        .eq(0)
    )

    valid_mask = (
        spectral_valid
        .And(no_cloud)
        .And(no_adjacent)
        .And(no_shadow)
        .And(no_snow)
    )

    if sensor == "S30":

        aerosol_level = (
            fmask
            .rightShift(6)
            .bitwiseAnd(3)
        )

        valid_mask = (
            valid_mask
            .And(
                aerosol_level.neq(3)
            )
        )

    prepared = (
        reflectance
        .updateMask(
            valid_mask
        )
        .toFloat()
    )

    return ee.Image(
        prepared
        .copyProperties(
            image,
            [
                "system:time_start",
                "system:index",
                "date_key",
                "sensor",
                "hls_sensor",
                "hls_mgrs_tile",
            ],
        )
    )


# ============================================================
# Spectral indices
# ============================================================

def safe_ratio(
    numerator,
    denominator,
    name,
    epsilon=1e-6,
):
    denominator = ee.Image(
        denominator
    )

    return (
        ee.Image(numerator)
        .divide(denominator)
        .updateMask(
            denominator
            .abs()
            .gt(epsilon)
        )
        .rename(name)
        .toFloat()
    )


def add_common_indices(image):
    image = ee.Image(image)

    blue = image.select("Blue")
    green = image.select("Green")
    red = image.select("Red")
    nir = image.select("NIR")
    swir1 = image.select("SWIR1")

    ndvi = safe_ratio(
        nir.subtract(red),
        nir.add(red),
        "NDVI",
    )

    evi = safe_ratio(
        nir
        .subtract(red)
        .multiply(2.5),
        nir
        .add(
            red.multiply(6.0)
        )
        .subtract(
            blue.multiply(7.5)
        )
        .add(1.0),
        "EVI",
    )

    savi = safe_ratio(
        nir
        .subtract(red)
        .multiply(1.5),
        nir
        .add(red)
        .add(0.5),
        "SAVI",
    )

    ndwi = safe_ratio(
        green.subtract(nir),
        green.add(nir),
        "NDWI",
    )

    ndmi = safe_ratio(
        nir.subtract(swir1),
        nir.add(swir1),
        "NDMI",
    )

    return (
        image
        .addBands(
            [
                ndvi,
                evi,
                savi,
                ndwi,
                ndmi,
            ]
        )
        .select(
            COMMON_PREDICTOR_BANDS
        )
    )


# ============================================================
# Daily mosaics
# ============================================================

def build_daily_common_collection(
    collection,
    prepare_function,
    geometry,
):
    collection = ee.ImageCollection(
        collection
    )

    date_keys = (
        ee.List(
            collection
            .aggregate_array(
                "date_key"
            )
        )
        .distinct()
        .sort()
    )

    def build_daily(date_key):

        date_key = ee.String(
            date_key
        )

        daily = (
            collection
            .filter(
                ee.Filter.eq(
                    "date_key",
                    date_key,
                )
            )
            .map(
                prepare_function
            )
            .mosaic()
            .clip(
                ee.Geometry(
                    geometry
                )
                .buffer(100)
            )
        )

        return ee.Image(
            daily.set(
                "date_key",
                date_key,
            )
        )

    return (
        ee.ImageCollection
        .fromImages(
            date_keys.map(
                build_daily
            )
        )
    )


# ============================================================
# Common six-band medoid
# ============================================================

def build_empty_common_image():

    return (
        ee.Image.constant(
            [0]
            * len(
                COMMON_REFLECTANCE_BANDS
            )
        )
        .rename(
            COMMON_REFLECTANCE_BANDS
        )
        .updateMask(
            ee.Image.constant(0)
        )
        .toFloat()
    )


def build_common_medoid(
    daily_collection,
):
    daily_collection = (
        ee.ImageCollection(
            daily_collection
        )
    )

    safe_collection = (
        daily_collection
        .merge(
            ee.ImageCollection(
                [
                    build_empty_common_image()
                ]
            )
        )
    )

    spectral_median = (
        safe_collection
        .select(
            COMMON_REFLECTANCE_BANDS
        )
        .median()
    )

    def add_score(image):

        image = ee.Image(image)

        distance = (
            image
            .select(
                COMMON_REFLECTANCE_BANDS
            )
            .subtract(
                spectral_median
            )
            .pow(2)
            .reduce(
                ee.Reducer.sum()
            )
        )

        return (
            image
            .addBands(
                distance
                .multiply(-1)
                .rename(
                    "medoid_score"
                )
            )
        )

    return (
        safe_collection
        .map(add_score)
        .qualityMosaic(
            "medoid_score"
        )
        .select(
            COMMON_REFLECTANCE_BANDS
        )
        .toFloat()
    )

In [3]:
# ============================================================
# Single footprint-period test
# ============================================================

test_footprint = ee.Feature(
    station_footprints
    .filter(
        ee.Filter.eq(
            "station",
            "Bananera",
        )
    )
    .first()
)

test_geometry = (
    test_footprint.geometry()
)

period_start = periods[0][0]
period_end = periods[0][1]

start_text = (
    period_start.isoformat()
)

end_text = (
    period_end.isoformat()
)

local_tiles = (
    get_local_mgrs_tiles(
        test_footprint
    )
)


# ------------------------------------------------------------
# Sentinel-2
# ------------------------------------------------------------

s2_period = (
    s2_collection
    .filterBounds(
        test_geometry
    )
    .filterDate(
        start_text,
        end_text,
    )
)

s2_daily = (
    build_daily_common_collection(
        collection=s2_period,
        prepare_function=prepare_s2_common,
        geometry=test_geometry,
    )
)

s2_medoid = (
    build_common_medoid(
        s2_daily
    )
)

s2_predictors = (
    add_common_indices(
        s2_medoid
    )
)


# ------------------------------------------------------------
# HLS-S30
# ------------------------------------------------------------

s30_period = (
    hls_s30_tiled
    .filter(
        ee.Filter.inList(
            "hls_mgrs_tile",
            local_tiles,
        )
    )
    .filterDate(
        start_text,
        end_text,
    )
)

s30_daily = (
    build_daily_common_collection(
        collection=s30_period,
        prepare_function=(
            lambda image:
                prepare_hls_common(
                    image,
                    "S30",
                )
        ),
        geometry=test_geometry,
    )
)


# ------------------------------------------------------------
# HLS-L30
# ------------------------------------------------------------

l30_period = (
    hls_l30_tiled
    .filter(
        ee.Filter.inList(
            "hls_mgrs_tile",
            local_tiles,
        )
    )
    .filterDate(
        start_text,
        end_text,
    )
)

l30_daily = (
    build_daily_common_collection(
        collection=l30_period,
        prepare_function=(
            lambda image:
                prepare_hls_common(
                    image,
                    "L30",
                )
        ),
        geometry=test_geometry,
    )
)


hls_daily = (
    s30_daily
    .merge(
        l30_daily
    )
)

hls_medoid = (
    build_common_medoid(
        hls_daily
    )
)

hls_predictors = (
    add_common_indices(
        hls_medoid
    )
)


print(
    "Period:",
    start_text,
    "->",
    end_text,
)

print(
    "Local MGRS tiles:",
    local_tiles,
)

print(
    "S2 raw products:",
    s2_period.size().getInfo(),
)

print(
    "S2 daily candidates:",
    s2_daily.size().getInfo(),
)

print(
    "HLS-S30 raw products:",
    s30_period.size().getInfo(),
)

print(
    "HLS-S30 daily candidates:",
    s30_daily.size().getInfo(),
)

print(
    "HLS-L30 raw products:",
    l30_period.size().getInfo(),
)

print(
    "HLS-L30 daily candidates:",
    l30_daily.size().getInfo(),
)

print(
    "HLS combined daily candidates:",
    hls_daily.size().getInfo(),
)

Period: 2021-01-01 -> 2021-01-09
Local MGRS tiles: ['18PWS']
S2 raw products: 2
S2 daily candidates: 2
HLS-S30 raw products: 3
HLS-S30 daily candidates: 3
HLS-L30 raw products: 0
HLS-L30 daily candidates: 0
HLS combined daily candidates: 3


In [4]:
# ============================================================
# Single footprint-period spectral sanity check
# ============================================================

def get_predictor_means(
    image,
    geometry,
    scale_m,
):
    image = ee.Image(image)

    values = (
        image
        .select(
            COMMON_PREDICTOR_BANDS
        )
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geometry,
            scale=scale_m,
            maxPixels=1e6,
        )
        .getInfo()
    )

    return values


def get_valid_coverage_pct(
    image,
    geometry,
    scale_m,
):
    image = ee.Image(image)

    valid_mask = (
        image
        .select(
            COMMON_REFLECTANCE_BANDS
        )
        .mask()
        .reduce(
            ee.Reducer.min()
        )
        .rename("valid")
    )

    valid_area = (
        ee.Image.pixelArea()
        .updateMask(
            valid_mask
        )
        .rename("valid_area")
    )

    valid_area_m2 = (
        valid_area
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=geometry,
            scale=scale_m,
            maxPixels=1e6,
        )
        .get("valid_area")
    )

    valid_area_m2 = ee.Number(
        ee.Algorithms.If(
            valid_area_m2,
            valid_area_m2,
            0,
        )
    )

    total_area_m2 = (
        ee.Geometry(
            geometry
        )
        .area(1)
    )

    return (
        valid_area_m2
        .divide(
            total_area_m2
        )
        .multiply(100)
        .getInfo()
    )


s2_means = (
    get_predictor_means(
        image=s2_predictors,
        geometry=test_geometry,
        scale_m=20,
    )
)

hls_means = (
    get_predictor_means(
        image=hls_predictors,
        geometry=test_geometry,
        scale_m=30,
    )
)


s2_coverage = (
    get_valid_coverage_pct(
        image=s2_predictors,
        geometry=test_geometry,
        scale_m=20,
    )
)

hls_coverage = (
    get_valid_coverage_pct(
        image=hls_predictors,
        geometry=test_geometry,
        scale_m=30,
    )
)


spectral_sanity_check = pd.DataFrame(
    [
        {
            "source": "S2",
            "coverage_pct": s2_coverage,
            **s2_means,
        },
        {
            "source": "HLS_COMBINED",
            "coverage_pct": hls_coverage,
            **hls_means,
        },
    ]
)


ordered_columns = [
    "source",
    "coverage_pct",
    *COMMON_REFLECTANCE_BANDS,
    *COMMON_INDEX_BANDS,
]


spectral_sanity_check = (
    spectral_sanity_check[
        ordered_columns
    ]
)


display(
    spectral_sanity_check.round(4)
)

,source,coverage_pct,Blue,Green,Red,NIR,SWIR1,SWIR2,NDVI,EVI,SAVI,NDWI,NDMI
0,S2,99.2887,0.0695,0.0939,0.0669,0.4294,0.1833,0.0971,0.7344,0.6928,0.5463,-0.6447,0.4022
1,HLS_COMBINED,99.8201,0.0163,0.0476,0.0287,0.4012,0.1501,0.0599,0.8651,0.6413,0.5999,-0.7862,0.4550


In [5]:
# ============================================================
# Same-date S2 vs HLS-S30 control
# ============================================================

s2_test_dates = set(
    s2_period
    .aggregate_array("date_key")
    .getInfo()
)

s30_test_dates = set(
    s30_period
    .aggregate_array("date_key")
    .getInfo()
)

shared_test_dates = sorted(
    s2_test_dates
    & s30_test_dates
)

print(
    "S2 dates:",
    sorted(s2_test_dates),
)

print(
    "HLS-S30 dates:",
    sorted(s30_test_dates),
)

print(
    "Shared dates:",
    shared_test_dates,
)


def build_single_date_image(
    collection,
    date_key,
    prepare_function,
    geometry,
):
    daily_collection = (
        ee.ImageCollection(collection)
        .filter(
            ee.Filter.eq(
                "date_key",
                date_key,
            )
        )
        .map(
            prepare_function
        )
    )

    return (
        daily_collection
        .mosaic()
        .clip(
            ee.Geometry(
                geometry
            )
            .buffer(100)
        )
    )


matched_rows = []


for date_key in shared_test_dates:

    # --------------------------------------------------------
    # Sentinel-2
    # --------------------------------------------------------

    s2_daily_image = (
        build_single_date_image(
            collection=s2_period,
            date_key=date_key,
            prepare_function=prepare_s2_common,
            geometry=test_geometry,
        )
    )

    s2_daily_predictors = (
        add_common_indices(
            s2_daily_image
        )
    )

    s2_values = (
        get_predictor_means(
            image=s2_daily_predictors,
            geometry=test_geometry,
            scale_m=20,
        )
    )

    s2_valid_coverage = (
        get_valid_coverage_pct(
            image=s2_daily_predictors,
            geometry=test_geometry,
            scale_m=20,
        )
    )

    matched_rows.append(
        {
            "date": date_key,
            "source": "S2",
            "coverage_pct": s2_valid_coverage,
            **s2_values,
        }
    )

    # --------------------------------------------------------
    # HLS-S30
    # --------------------------------------------------------

    hls_daily_image = (
        build_single_date_image(
            collection=s30_period,
            date_key=date_key,
            prepare_function=(
                lambda image:
                    prepare_hls_common(
                        image,
                        "S30",
                    )
            ),
            geometry=test_geometry,
        )
    )

    hls_daily_predictors = (
        add_common_indices(
            hls_daily_image
        )
    )

    hls_values = (
        get_predictor_means(
            image=hls_daily_predictors,
            geometry=test_geometry,
            scale_m=30,
        )
    )

    hls_valid_coverage = (
        get_valid_coverage_pct(
            image=hls_daily_predictors,
            geometry=test_geometry,
            scale_m=30,
        )
    )

    matched_rows.append(
        {
            "date": date_key,
            "source": "HLS_S30",
            "coverage_pct": hls_valid_coverage,
            **hls_values,
        }
    )


matched_date_comparison = pd.DataFrame(
    matched_rows
)

matched_date_comparison = (
    matched_date_comparison[
        [
            "date",
            "source",
            "coverage_pct",
            *COMMON_REFLECTANCE_BANDS,
            *COMMON_INDEX_BANDS,
        ]
    ]
)

display(
    matched_date_comparison.round(4)
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


S2 dates: ['2021-01-03', '2021-01-08']
HLS-S30 dates: ['2021-01-03', '2021-01-06', '2021-01-08']
Shared dates: ['2021-01-03', '2021-01-08']


,date,source,coverage_pct,Blue,Green,Red,NIR,SWIR1,SWIR2,NDVI,EVI,SAVI,NDWI,NDMI
0,2021-01-03,S2,99.2887,0.0350,0.0659,0.0407,0.4231,0.1686,0.0765,0.8229,0.6803,0.5939,-0.7286,0.4295
1,2021-01-03,HLS_S30,99.8201,0.0163,0.0476,0.0287,0.4012,0.1501,0.0599,0.8651,0.6413,0.5999,-0.7862,0.4550
2,2021-01-08,S2,99.2887,0.1021,0.1203,0.0917,0.4347,0.1972,0.1165,0.6507,0.7031,0.5007,-0.5654,0.3756
3,2021-01-08,HLS_S30,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# ============================================================
# Same-date spectral differences
# ============================================================

matched_wide = (
    matched_date_comparison
    .pivot(
        index="date",
        columns="source",
        values=COMMON_PREDICTOR_BANDS,
    )
)


difference_rows = []


for date_key in shared_test_dates:

    for predictor in COMMON_PREDICTOR_BANDS:

        s2_value = (
            matched_wide
            .loc[
                date_key,
                (
                    predictor,
                    "S2",
                ),
            ]
        )

        hls_value = (
            matched_wide
            .loc[
                date_key,
                (
                    predictor,
                    "HLS_S30",
                ),
            ]
        )

        difference_rows.append(
            {
                "date": date_key,
                "predictor": predictor,
                "S2": s2_value,
                "HLS_S30": hls_value,
                "HLS_minus_S2": (
                    hls_value
                    - s2_value
                ),
            }
        )


matched_date_differences = pd.DataFrame(
    difference_rows
)

display(
    matched_date_differences.round(4)
)

,date,predictor,S2,HLS_S30,HLS_minus_S2
0,2021-01-03,Blue,0.0350,0.0163,-0.0187
1,2021-01-03,Green,0.0659,0.0476,-0.0182
2,2021-01-03,Red,0.0407,0.0287,-0.0120
3,2021-01-03,NIR,0.4231,0.4012,-0.0219
4,2021-01-03,SWIR1,0.1686,0.1501,-0.0185
5,2021-01-03,SWIR2,0.0765,0.0599,-0.0166
6,2021-01-03,NDVI,0.8229,0.8651,0.0422
7,2021-01-03,EVI,0.6803,0.6413,-0.0390
8,2021-01-03,SAVI,0.5939,0.5999,0.0060
9,2021-01-03,NDWI,-0.7286,-0.7862,-0.0576


In [7]:
# ============================================================
# Full footprint-period spectral dataset
# ============================================================

OUTPUT_DIR = (
    REPO_ROOT
    / "outputs"
    / "diagnostics"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

COMMON_DATASET_PATH = (
    OUTPUT_DIR
    / "optical_source_common_predictors.csv"
)


def calculate_predictor_coverage(
    image,
    geometry,
    scale_m,
):
    """
    Calculate the percentage of footprint area containing
    valid values in all six common reflectance bands.
    """
    image = ee.Image(image)

    valid_mask = (
        image
        .select(
            COMMON_REFLECTANCE_BANDS
        )
        .mask()
        .reduce(
            ee.Reducer.min()
        )
        .rename("valid")
    )

    valid_area = (
        ee.Image.pixelArea()
        .updateMask(
            valid_mask
        )
        .rename("valid_area")
    )

    valid_area_m2 = (
        valid_area
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=geometry,
            scale=scale_m,
            maxPixels=1e7,
        )
        .get("valid_area")
    )

    valid_area_m2 = ee.Number(
        ee.Algorithms.If(
            valid_area_m2,
            valid_area_m2,
            0,
        )
    )

    total_area_m2 = (
        ee.Geometry(
            geometry
        )
        .area(1)
    )

    return (
        valid_area_m2
        .divide(
            total_area_m2
        )
        .multiply(100)
    )


def build_predictor_feature(
    image,
    geometry,
    scale_m,
    station,
    period_start,
    period_end,
    source,
):
    """
    Summarize one optical predictor image at MODIS footprint
    support.
    """
    image = ee.Image(image)

    predictor_means = (
        image
        .select(
            COMMON_PREDICTOR_BANDS
        )
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geometry,
            scale=scale_m,
            maxPixels=1e7,
        )
    )

    coverage_pct = (
        calculate_predictor_coverage(
            image=image,
            geometry=geometry,
            scale_m=scale_m,
        )
    )

    metadata = ee.Dictionary(
        {
            "station": station,
            "period_start": period_start,
            "period_end": period_end,
            "source": source,
            "scale_m": scale_m,
            "coverage_pct": coverage_pct,
        }
    )

    return ee.Feature(
        None,
        metadata.combine(
            predictor_means,
            overwrite=True,
        ),
    )

In [8]:
# ============================================================
# Build both optical alternatives for one MODIS period
# ============================================================

def build_period_features(
    footprint,
    station,
    local_tiles,
    period_start,
    period_end,
):
    geometry = (
        ee.Feature(
            footprint
        )
        .geometry()
    )

    start_text = (
        period_start.isoformat()
    )

    end_text = (
        period_end.isoformat()
    )

    # --------------------------------------------------------
    # Sentinel-2
    # --------------------------------------------------------

    s2_period = (
        s2_collection
        .filterBounds(
            geometry
        )
        .filterDate(
            start_text,
            end_text,
        )
    )

    s2_daily = (
        build_daily_common_collection(
            collection=s2_period,
            prepare_function=prepare_s2_common,
            geometry=geometry,
        )
    )

    s2_medoid = (
        build_common_medoid(
            s2_daily
        )
    )

    s2_predictors = (
        add_common_indices(
            s2_medoid
        )
    )

    s2_feature = (
        build_predictor_feature(
            image=s2_predictors,
            geometry=geometry,
            scale_m=20,
            station=station,
            period_start=start_text,
            period_end=end_text,
            source="S2",
        )
    )

    # --------------------------------------------------------
    # HLS-S30
    # --------------------------------------------------------

    s30_period = (
        hls_s30_tiled
        .filter(
            ee.Filter.inList(
                "hls_mgrs_tile",
                local_tiles,
            )
        )
        .filterDate(
            start_text,
            end_text,
        )
    )

    s30_daily = (
        build_daily_common_collection(
            collection=s30_period,
            prepare_function=(
                lambda image:
                    prepare_hls_common(
                        image,
                        "S30",
                    )
            ),
            geometry=geometry,
        )
    )

    # --------------------------------------------------------
    # HLS-L30
    # --------------------------------------------------------

    l30_period = (
        hls_l30_tiled
        .filter(
            ee.Filter.inList(
                "hls_mgrs_tile",
                local_tiles,
            )
        )
        .filterDate(
            start_text,
            end_text,
        )
    )

    l30_daily = (
        build_daily_common_collection(
            collection=l30_period,
            prepare_function=(
                lambda image:
                    prepare_hls_common(
                        image,
                        "L30",
                    )
            ),
            geometry=geometry,
        )
    )

    # --------------------------------------------------------
    # HLS combined medoid
    # --------------------------------------------------------

    hls_daily = (
        s30_daily
        .merge(
            l30_daily
        )
        .sort(
            "date_key"
        )
    )

    hls_medoid = (
        build_common_medoid(
            hls_daily
        )
    )

    hls_predictors = (
        add_common_indices(
            hls_medoid
        )
    )

    hls_feature = (
        build_predictor_feature(
            image=hls_predictors,
            geometry=geometry,
            scale_m=30,
            station=station,
            period_start=start_text,
            period_end=end_text,
            source="HLS_COMBINED",
        )
    )

    return [
        s2_feature,
        hls_feature,
    ]

In [9]:
# ============================================================
# Execute full controlled comparison with checkpoints
# ============================================================

all_rows = []


for station_info in stations:

    station_name = (
        station_info[
            "station"
        ]
    )

    footprint = ee.Feature(
        station_footprints
        .filter(
            ee.Filter.eq(
                "station",
                station_name,
            )
        )
        .first()
    )

    local_tiles = (
        get_local_mgrs_tiles(
            footprint
        )
    )

    print()
    print(
        "Station:",
        station_name,
    )

    print(
        "MGRS:",
        local_tiles,
    )


    for year in [
        2021,
        2022,
        2023,
    ]:

        year_periods = [
            (
                period_start,
                period_end,
            )
            for (
                period_start,
                period_end,
            )
            in periods
            if period_start.year == year
        ]

        print(
            "  Processing:",
            year,
            "- periods:",
            len(year_periods),
        )

        features = []


        for (
            period_start,
            period_end,
        ) in year_periods:

            features.extend(
                build_period_features(
                    footprint=footprint,
                    station=station_name,
                    local_tiles=local_tiles,
                    period_start=period_start,
                    period_end=period_end,
                )
            )


        block_collection = (
            ee.FeatureCollection(
                features
            )
        )

        block_info = (
            block_collection
            .getInfo()
        )

        block_rows = [
            feature[
                "properties"
            ]
            for feature
            in block_info[
                "features"
            ]
        ]

        all_rows.extend(
            block_rows
        )

        # ----------------------------------------------------
        # Checkpoint after every station-year block
        # ----------------------------------------------------

        checkpoint_df = (
            pd.DataFrame(
                all_rows
            )
        )

        checkpoint_df.to_csv(
            COMMON_DATASET_PATH,
            index=False,
        )

        print(
            "    rows saved:",
            len(all_rows),
        )


print()
print(
    "COMPLETE"
)

print(
    "Rows:",
    len(all_rows),
)

print(
    "Expected:",
    5 * 138 * 2,
)

print(
    "Saved to:",
    COMMON_DATASET_PATH,
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(



Station: Pastos limpios
MGRS: ['18PWS']
  Processing: 2021 - periods: 46
    rows saved: 92
  Processing: 2022 - periods: 46


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


    rows saved: 184
  Processing: 2023 - periods: 46


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


...

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(



Station: Bosque seco
MGRS: ['18PWS']
  Processing: 2021 - periods: 46
    rows saved: 1196
  Processing: 2022 - periods: 46


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


    rows saved: 1288
  Processing: 2023 - periods: 46
    rows saved: 1380

COMPLETE
Rows: 1380
Expected: 1380
Saved to: C:\Users\User\Desktop\cristian\GEE\26-et-downscaling-fundacion\outputs\diagnostics\optical_source_common_predictors.csv


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


In [10]:
# ============================================================
# Dataset QA
# ============================================================

common_predictor_dataset = pd.read_csv(
    COMMON_DATASET_PATH
)

common_predictor_dataset[
    "coverage_pct"
] = pd.to_numeric(
    common_predictor_dataset[
        "coverage_pct"
    ],
    errors="coerce",
)


print(
    "Rows:",
    len(
        common_predictor_dataset
    )
)

print(
    "Expected:",
    1380,
)

print()

print(
    common_predictor_dataset[
        "source"
    ]
    .value_counts()
)

print()

print(
    "Rows with predictors:"
)

print(
    common_predictor_dataset[
        COMMON_PREDICTOR_BANDS
    ]
    .notna()
    .all(axis=1)
    .groupby(
        common_predictor_dataset[
            "source"
        ]
    )
    .sum()
)


display(
    common_predictor_dataset.head()
)


assert (
    len(
        common_predictor_dataset
    )
    == 1380
)

Rows: 1380
Expected: 1380

source
S2              690
HLS_COMBINED    690
Name: count, dtype: int64

Rows with predictors:
source
HLS_COMBINED    489
S2              586
dtype: int64


,Blue,EVI,Green,NDMI,NDVI,NDWI,NIR,Red,SAVI,SWIR1,SWIR2,coverage_pct,period_end,period_start,scale_m,source,station
0,0.054652,0.450081,0.079506,0.135723,0.626683,-0.574951,0.302728,0.067541,0.401016,0.231254,0.130561,99.304466,2021-01-09,2021-01-01,20,S2,Pastos limpios
1,0.030586,0.438961,0.059628,0.159468,0.695576,-0.655355,0.293470,0.051431,0.425348,0.214364,0.104184,99.822055,2021-01-09,2021-01-01,30,HLS_COMBINED,Pastos limpios
2,0.133314,0.517690,0.151414,0.173723,0.476169,-0.419966,0.372458,0.131616,0.358852,0.262745,0.162751,99.304466,2021-01-17,2021-01-09,20,S2,Pastos limpios
3,0.041058,0.461708,0.077072,0.169681,0.650942,-0.628896,0.341803,0.071492,0.442056,0.244026,0.127737,95.060982,2021-01-17,2021-01-09,30,HLS_COMBINED,Pastos limpios
4,0.044343,0.461248,0.073722,0.105742,0.655157,-0.627490,0.326326,0.067730,0.431801,0.267407,0.150179,99.304466,2021-01-25,2021-01-17,20,S2,Pastos limpios


In [11]:
# ============================================================
# Final QA and paired optical dataset
# ============================================================

common_predictor_dataset[
    "period_start"
] = pd.to_datetime(
    common_predictor_dataset[
        "period_start"
    ]
)

common_predictor_dataset[
    "period_end"
] = pd.to_datetime(
    common_predictor_dataset[
        "period_end"
    ]
)


KEY_COLUMNS = [
    "station",
    "period_start",
    "period_end",
]


# ------------------------------------------------------------
# Duplicate QA
# ------------------------------------------------------------

duplicate_count = (
    common_predictor_dataset
    .duplicated(
        subset=[
            *KEY_COLUMNS,
            "source",
        ]
    )
    .sum()
)

print(
    "Duplicate station-period-source rows:",
    duplicate_count,
)


# ------------------------------------------------------------
# Predictor availability QA
# ------------------------------------------------------------

common_predictor_dataset[
    "predictors_complete"
] = (
    common_predictor_dataset[
        COMMON_PREDICTOR_BANDS
    ]
    .notna()
    .all(axis=1)
)


availability_qa = (
    common_predictor_dataset
    .groupby(
        "source",
        as_index=False,
    )
    .agg(
        rows=(
            "source",
            "size",
        ),
        coverage_gt_0=(
            "coverage_pct",
            lambda x: int(
                (x > 0).sum()
            ),
        ),
        predictors_complete=(
            "predictors_complete",
            "sum",
        ),
    )
)

display(
    availability_qa
)


# ------------------------------------------------------------
# Wide paired dataset
# ------------------------------------------------------------

paired_optical = (
    common_predictor_dataset
    .pivot(
        index=KEY_COLUMNS,
        columns="source",
        values=[
            "coverage_pct",
            *COMMON_PREDICTOR_BANDS,
        ],
    )
)

paired_optical.columns = [
    f"{variable}_{source}"
    for variable, source
    in paired_optical.columns
]

paired_optical = (
    paired_optical
    .reset_index()
)


print(
    "Paired footprint-periods:",
    len(paired_optical),
)

assert duplicate_count == 0
assert len(paired_optical) == 690

display(
    paired_optical.head()
)

Duplicate station-period-source rows: 0


,source,rows,coverage_gt_0,predictors_complete
0,HLS_COMBINED,690,489,489
1,S2,690,586,586


Paired footprint-periods: 690


,station,period_start,period_end,coverage_pct_HLS_COMBINED,coverage_pct_S2,Blue_HLS_COMBINED,Blue_S2,Green_HLS_COMBINED,Green_S2,Red_HLS_COMBINED,...,NDVI_HLS_COMBINED,NDVI_S2,EVI_HLS_COMBINED,EVI_S2,SAVI_HLS_COMBINED,SAVI_S2,NDWI_HLS_COMBINED,NDWI_S2,NDMI_HLS_COMBINED,NDMI_S2
0,Bananera,2021-01-01,2021-01-09,99.820118,99.28874,0.016253,0.069526,0.047644,0.093863,0.028724,...,0.865104,0.734414,0.641344,0.692789,0.599875,0.546346,-0.786163,-0.644690,0.455046,0.402188
1,Bananera,2021-01-09,2021-01-17,99.820118,99.28874,0.017127,0.041872,0.049192,0.071895,0.032514,...,0.850191,0.792167,0.633968,0.667992,0.595627,0.576676,-0.781898,-0.707586,0.452948,0.420985
2,Bananera,2021-01-17,2021-01-25,99.820118,99.28874,0.017788,0.031055,0.049718,0.063349,0.032173,...,0.846656,0.823077,0.618967,0.667015,0.582415,0.593956,-0.772773,-0.737256,0.420787,0.402305
3,Bananera,2021-01-25,2021-02-02,99.820118,99.28874,0.018329,0.038921,0.050884,0.068035,0.036524,...,0.819346,0.793226,0.577182,0.653991,0.552868,0.572431,-0.757238,-0.716661,0.414169,0.371049
4,Bananera,2021-02-02,2021-02-10,99.820118,99.28874,0.016181,0.033145,0.047510,0.063854,0.031613,...,0.841273,0.800060,0.584242,0.625028,0.559042,0.562199,-0.769802,-0.722279,0.410476,0.388003


In [12]:
# ============================================================
# Paired spatial-support availability
# ============================================================

threshold_rows = []


for threshold in [
    80,
    90,
    99,
]:

    s2_ok = (
        paired_optical[
            "coverage_pct_S2"
        ]
        >= threshold
    )

    hls_ok = (
        paired_optical[
            "coverage_pct_HLS_COMBINED"
        ]
        >= threshold
    )

    threshold_rows.append(
        {
            "threshold_pct":
                threshold,

            "both":
                int(
                    (
                        s2_ok
                        & hls_ok
                    )
                    .sum()
                ),

            "S2_only":
                int(
                    (
                        s2_ok
                        & ~hls_ok
                    )
                    .sum()
                ),

            "HLS_only":
                int(
                    (
                        ~s2_ok
                        & hls_ok
                    )
                    .sum()
                ),

            "neither":
                int(
                    (
                        ~s2_ok
                        & ~hls_ok
                    )
                    .sum()
                ),
        }
    )


paired_support_summary = pd.DataFrame(
    threshold_rows
)

display(
    paired_support_summary
)

,threshold_pct,both,S2_only,HLS_only,neither
0,80,354,143,27,166
1,90,335,142,26,187
2,99,309,131,27,223


In [14]:
# ============================================================
# Spectral agreement metrics
# Primary comparison: both sources >= 90% coverage
# ============================================================

PRIMARY_COVERAGE_THRESHOLD = 90


high_support_pairs = (
    paired_optical[
        (
            paired_optical[
                "coverage_pct_S2"
            ]
            >= PRIMARY_COVERAGE_THRESHOLD
        )
        &
        (
            paired_optical[
                "coverage_pct_HLS_COMBINED"
            ]
            >= PRIMARY_COVERAGE_THRESHOLD
        )
    ]
    .copy()
)


print(
    "High-support paired footprint-periods:",
    len(high_support_pairs),
)


spectral_metric_rows = []


for predictor in (
    COMMON_PREDICTOR_BANDS
):

    s2_column = (
        f"{predictor}_S2"
    )

    hls_column = (
        f"{predictor}_HLS_COMBINED"
    )

    valid_pairs = (
        high_support_pairs[
            [
                s2_column,
                hls_column,
            ]
        ]
        .dropna()
    )

    s2_values = (
        valid_pairs[
            s2_column
        ]
    )

    hls_values = (
        valid_pairs[
            hls_column
        ]
    )

    difference = (
        hls_values
        - s2_values
    )

    spectral_metric_rows.append(
        {
            "predictor":
                predictor,

            "n":
                len(valid_pairs),

            "mean_S2":
                s2_values.mean(),

            "mean_HLS":
                hls_values.mean(),

            "mean_bias_HLS_minus_S2":
                difference.mean(),

            "median_bias_HLS_minus_S2":
                difference.median(),

            "MAE":
                difference.abs().mean(),

            "RMSE":
                np.sqrt(
                    np.mean(
                        difference ** 2
                    )
                ),

            "pearson_r":
                s2_values.corr(
                    hls_values
                ),
        }
    )


spectral_agreement_90 = (
    pd.DataFrame(
        spectral_metric_rows
    )
)


display(
    spectral_agreement_90.round(4)
)

High-support paired footprint-periods: 335


,predictor,n,mean_S2,mean_HLS,mean_bias_HLS_minus_S2,median_bias_HLS_minus_S2,MAE,RMSE,pearson_r
0,Blue,335,0.0397,0.0275,-0.0122,-0.0118,0.0155,0.0270,0.0902
1,Green,335,0.0711,0.0592,-0.0119,-0.0109,0.0140,0.0246,0.2233
2,Red,335,0.0512,0.0435,-0.0077,-0.0071,0.0107,0.0229,0.4191
3,NIR,335,0.3849,0.3536,-0.0313,-0.0297,0.0334,0.0390,0.9129
4,SWIR1,335,0.2154,0.1874,-0.0280,-0.0265,0.0295,0.0328,0.8868
5,SWIR2,335,0.1120,0.0854,-0.0266,-0.0251,0.0276,0.0299,0.8875
6,NDVI,335,0.7564,0.7756,0.0191,0.0151,0.0313,0.0487,0.8804
7,EVI,335,0.5952,0.5482,-0.0470,-0.0445,0.0476,0.0536,0.9753
8,SAVI,335,0.5279,0.5140,-0.0139,-0.0150,0.0241,0.0319,0.9381
9,NDWI,335,-0.6807,-0.7087,-0.0279,-0.0241,0.0352,0.0513,0.7941


In [15]:
# ============================================================
# Locate the current master ET dataset
# ============================================================

required_master_columns = {
    "station",
    "period_start",
    "period_end",
    "ET_mm_day",
    "predictor_support",
    "VV_dB_mean",
    "VH_dB_mean",
    "VV_minus_VH_dB_mean",
    "Tair_mean_C",
    "VPD_mean_kPa",
    "SolarRad_MJ_m2_day",
    "Wind_mean_ms",
    "Precip_period_mm",
}

candidate_rows = []


for csv_path in (
    REPO_ROOT
    / "outputs"
).rglob("*.csv"):

    # Diagnostic CSVs are not candidates for the master dataset.
    if "diagnostics" in csv_path.parts:
        continue

    try:
        sample = pd.read_csv(
            csv_path,
            nrows=5,
        )

    except Exception:
        continue

    columns = set(
        sample.columns
    )

    matched_columns = (
        required_master_columns
        & columns
    )

    candidate_rows.append(
        {
            "path":
                str(
                    csv_path.relative_to(
                        REPO_ROOT
                    )
                ),

            "columns":
                len(columns),

            "required_found":
                len(
                    matched_columns
                ),

            "required_expected":
                len(
                    required_master_columns
                ),

            "complete_candidate":
                required_master_columns
                .issubset(columns),
        }
    )


master_candidates = (
    pd.DataFrame(
        candidate_rows
    )
    .sort_values(
        [
            "complete_candidate",
            "required_found",
        ],
        ascending=[
            False,
            False,
        ],
    )
)


display(
    master_candidates.head(20)
)

,path,columns,required_found,required_expected,complete_candidate
0,outputs\ET_S2_S1_METEO_MULTISCALE_2021_2023.csv,83,13,13,True


In [16]:
# ============================================================
# Audit current master ET dataset before optical-source modeling
# ============================================================

MASTER_DATASET_PATH = (
    REPO_ROOT
    / "outputs"
    / "ET_S2_S1_METEO_MULTISCALE_2021_2023.csv"
)

master_dataset = pd.read_csv(
    MASTER_DATASET_PATH
)

print(
    "Rows:",
    len(master_dataset),
)

print(
    "Columns:",
    len(master_dataset.columns),
)

print()


# ============================================================
# Support / scale structure
# ============================================================

for column in [
    "scale",
    "predictor_support",
    "target_support",
]:

    if column in master_dataset.columns:

        print(
            f"{column}:"
        )

        print(
            master_dataset[
                column
            ]
            .value_counts(
                dropna=False
            )
        )

        print()


# ============================================================
# Key QA flags
# ============================================================

for column in [
    "valid_observation",
    "modis_good",
    "modis_qc_good",
    "s2_valid",
    "s1_valid",
    "meteo_complete",
    "meteo_temporal_complete",
    "stats_complete",
]:

    if column in master_dataset.columns:

        print(
            f"{column}:"
        )

        print(
            master_dataset[
                column
            ]
            .value_counts(
                dropna=False
            )
            .sort_index()
        )

        print()


# ============================================================
# Footprint-only candidate
# ============================================================

if (
    "predictor_support"
    in master_dataset.columns
):

    footprint_master = (
        master_dataset[
            master_dataset[
                "predictor_support"
            ]
            == "MODIS_footprint"
        ]
        .copy()
    )

elif (
    "scale"
    in master_dataset.columns
):

    footprint_master = (
        master_dataset[
            master_dataset[
                "scale"
            ]
            == "footprint"
        ]
        .copy()
    )

else:

    footprint_master = (
        master_dataset.copy()
    )


print(
    "Footprint rows:",
    len(footprint_master),
)

print(
    "Stations:",
    footprint_master[
        "station"
    ]
    .nunique()
)

print(
    "Unique station-periods:",
    footprint_master[
        [
            "station",
            "period_start",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Expected complete universe:",
    5 * 138,
)

print()


# ============================================================
# Target QA
# ============================================================

target_candidates = [
    column
    for column
    in master_dataset.columns
    if (
        "ET" in column
        or "et_" in column.lower()
    )
]

print(
    "Possible ET columns:"
)

print(
    target_candidates
)

print()


if (
    "ET_mm_day"
    in footprint_master.columns
):

    print(
        "ET_mm_day non-null:",
        footprint_master[
            "ET_mm_day"
        ]
        .notna()
        .sum()
    )

    print(
        "ET_mm_day range:",
        footprint_master[
            "ET_mm_day"
        ]
        .min(),
        "to",
        footprint_master[
            "ET_mm_day"
        ]
        .max(),
    )

    print()


# ============================================================
# S1 / meteorology completeness
# ============================================================

BASE_MODEL_COLUMNS = [
    "ET_mm_day",
    "VV_dB_mean",
    "VH_dB_mean",
    "VV_minus_VH_dB_mean",
    "Tair_mean_C",
    "VPD_mean_kPa",
    "SolarRad_MJ_m2_day",
    "Wind_mean_ms",
    "Precip_period_mm",
]

available_base_columns = [
    column
    for column
    in BASE_MODEL_COLUMNS
    if column
    in footprint_master.columns
]

print(
    "Base model columns available:"
)

print(
    available_base_columns
)

print()


if available_base_columns:

    complete_base = (
        footprint_master[
            available_base_columns
        ]
        .notna()
        .all(axis=1)
    )

    print(
        "Rows complete for target + S1 + meteorology:",
        int(
            complete_base.sum()
        ),
    )

Rows: 608
Columns: 83

scale:
scale
footprint    304
local_60m    304
Name: count, dtype: int64

predictor_support:
predictor_support
MODIS_footprint    304
60m_x_60m          304
Name: count, dtype: int64

target_support:
target_support
MODIS_footprint    608
Name: count, dtype: int64

modis_good:
modis_good
1    608
Name: count, dtype: int64

meteo_complete:
meteo_complete
1    608
Name: count, dtype: int64

meteo_temporal_complete:
meteo_temporal_complete
1    608
Name: count, dtype: int64

stats_complete:
stats_complete
1    608
Name: count, dtype: int64

Footprint rows: 304
Stations: 5
Unique station-periods: 304
Expected complete universe: 690

Possible ET columns:
['ET_mm_period', 'ET_mm_day', 'target_support']

ET_mm_day non-null: 304
ET_mm_day range: 2.0375 to 8.325000000000001

Base model columns available:
['ET_mm_day', 'VV_dB_mean', 'VH_dB_mean', 'VV_minus_VH_dB_mean', 'Tair_mean_C', 'VPD_mean_kPa', 'SolarRad_MJ_m2_day', 'Wind_mean_ms', 'Precip_period_mm']

Rows complete fo

In [18]:
# ============================================================
# Reload dataset module after source-code fix
# ============================================================

import importlib
import et_downscaling.dataset as dataset_module

importlib.reload(
    dataset_module
)

build_availability_table = (
    dataset_module
    .build_availability_table
)

build_observations_with_stats = (
    dataset_module
    .build_observations_with_stats
)

build_footprint_rows = (
    dataset_module
    .build_footprint_rows
)

print(
    "dataset.py reloaded"
)

dataset.py reloaded


In [19]:
neutral_availability = (
    build_availability_table(
        modis_inputs=modis_inputs,
        s2_collection=empty_s2_collection,
        s1_collection=s1_collection,
    )
)

In [20]:
# ============================================================
# Neutral master — repository functions
# ============================================================

from et_downscaling.dataset import (
    build_availability_table,
    build_observations_with_stats,
    build_footprint_rows,
)

from et_downscaling.meteorology import (
    build_meteorology_inputs,
)

from et_downscaling.sentinel1 import (
    get_sentinel1_collection,
)


# ============================================================
# Sentinel-1 and meteorological inputs
# ============================================================

s1_collection = (
    get_sentinel1_collection(
        station_footprints
    )
)

meteorology_inputs = (
    build_meteorology_inputs(
        station_footprints
    )
)


# ============================================================
# Empty optical collection
#
# Optical information is intentionally excluded here.
# It will be merged later from the controlled optical dataset.
# ============================================================

empty_s2_collection = (
    ee.ImageCollection([])
)


# ============================================================
# Full station × MODIS-period availability universe
# ============================================================

neutral_availability = (
    build_availability_table(
        modis_inputs=modis_inputs,
        s2_collection=empty_s2_collection,
        s1_collection=s1_collection,
    )
)


neutral_observations = (
    neutral_availability
    .filter(
        ee.Filter.eq(
            "period_within_analysis",
            1,
        )
    )
)


print(
    "Neutral observations:",
    neutral_observations.size().getInfo(),
)

print(
    "Expected:",
    5 * 138,
)

print(
    "Valid MODIS targets:",
    neutral_observations
    .filter(
        ee.Filter.eq(
            "modis_good",
            1,
        )
    )
    .size()
    .getInfo(),
)

Neutral observations: 690
Expected: 690
Valid MODIS targets: 690


In [21]:
# ============================================================
# Add Sentinel-1 and meteorology
# ============================================================

neutral_with_stats = (
    build_observations_with_stats(
        valid_observations=neutral_observations,
        s2_collection=empty_s2_collection,
        s1_collection=s1_collection,
        meteorology_inputs=meteorology_inputs,
    )
)


# ============================================================
# FOOTPRINT ONLY
#
# Do not use build_output_table() here because it also creates
# the deprecated local_60m records.
# ============================================================

neutral_footprint_rows = (
    build_footprint_rows(
        neutral_with_stats
    )
)


print(
    "Neutral footprint rows:",
    neutral_footprint_rows
    .size()
    .getInfo(),
)

Neutral footprint rows: 690


In [ ]:
# ============================================================
# Export neutral footprint master locally
# ============================================================

NEUTRAL_MASTER_PATH = (
    REPO_ROOT
    / "outputs"
    / "diagnostics"
    / "neutral_master_2021_2023.csv"
)


neutral_info = (
    neutral_footprint_rows
    .getInfo()
)


neutral_rows = [
    feature["properties"]
    for feature
    in neutral_info["features"]
]


neutral_master = pd.DataFrame(
    neutral_rows
)


neutral_master.to_csv(
    NEUTRAL_MASTER_PATH,
    index=False,
)


print(
    "Rows:",
    len(neutral_master),
)

print(
    "Columns:",
    len(neutral_master.columns),
)

print(
    "Saved:",
    NEUTRAL_MASTER_PATH,
)

In [2]:
# ============================================================
# Clean bootstrap for neutral MODIS master
# ============================================================

from datetime import date, timedelta
from pathlib import Path
import importlib
import sys

import ee
import pandas as pd


# ============================================================
# Repository
# ============================================================

def find_repo_root(start=None):
    current = Path(
        start or Path.cwd()
    ).resolve()

    for candidate in [
        current,
        *current.parents,
    ]:
        if (
            candidate
            / "pyproject.toml"
        ).exists():
            return candidate

    raise FileNotFoundError(
        "Repository root not found."
    )


REPO_ROOT = find_repo_root()

SRC_PATH = (
    REPO_ROOT
    / "src"
)

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )

print(
    "Repository:",
    REPO_ROOT,
)


# ============================================================
# Earth Engine
# ============================================================

EE_PROJECT = "ee-change"

ee.Initialize(
    project=EE_PROJECT
)

print(
    "Earth Engine project:",
    EE_PROJECT,
)


# ============================================================
# Current repository modules
# ============================================================

import et_downscaling.modis as modis_module
import et_downscaling.dataset as dataset_module

importlib.reload(
    modis_module
)

importlib.reload(
    dataset_module
)


# ============================================================
# MODIS inputs
# ============================================================

modis_inputs = (
    modis_module
    .build_modis_inputs()
)

station_footprints = (
    ee.FeatureCollection(
        modis_inputs[
            "station_footprints"
        ]
    )
)

stations_info = (
    station_footprints
    .select(
        [
            "station",
            "station_id",
            "longitude",
            "latitude",
        ]
    )
    .getInfo()[
        "features"
    ]
)

stations = [
    feature[
        "properties"
    ]
    for feature
    in stations_info
]

print(
    "Stations:",
    [
        station["station"]
        for station
        in stations
    ],
)


# ============================================================
# Build full availability universe
# ============================================================

# We intentionally use empty optical and S1 collections here.
# This stage is only for MODIS target + metadata.

empty_collection = (
    ee.ImageCollection([])
)

neutral_availability = (
    dataset_module
    .build_availability_table(
        modis_inputs=modis_inputs,
        s2_collection=empty_collection,
        s1_collection=empty_collection,
    )
)

neutral_observations = (
    neutral_availability
    .filter(
        ee.Filter.eq(
            "period_within_analysis",
            1,
        )
    )
)

print(
    "Neutral observations:",
    neutral_observations
    .size()
    .getInfo(),
)

print(
    "Expected:",
    5 * 138,
)

Repository: C:\Users\User\Desktop\cristian\GEE\26-et-downscaling-fundacion


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Earth Engine project: ee-change


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Stations: ['Pastos limpios', 'Palma', 'Bananera', 'Manglar', 'Bosque seco']
Neutral observations: 690
Expected: 690


In [3]:
# ============================================================
# Lightweight neutral MODIS master
# ============================================================

NEUTRAL_BASE_PATH = (
    REPO_ROOT
    / "outputs"
    / "diagnostics"
    / "neutral_modis_master_2021_2023.csv"
)

NEUTRAL_BASE_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

NEUTRAL_BASE_COLUMNS = [
    "station",
    "station_id",
    "period_start",
    "period_end",
    "number_days",
    "longitude",
    "latitude",
    "ET_mm_period",
    "ET_mm_day",
    "modis_good",
    "modis_qc_good",
    "period_within_analysis",
    "target_support",
]


available_properties = (
    ee.Feature(
        neutral_observations.first()
    )
    .propertyNames()
    .getInfo()
)

selected_columns = [
    column
    for column
    in NEUTRAL_BASE_COLUMNS
    if column
    in available_properties
]

print(
    "Selected columns:",
    selected_columns,
)


neutral_base_rows = []


for station_info in stations:

    station_name = (
        station_info[
            "station"
        ]
    )

    print(
        "Processing:",
        station_name,
    )

    station_collection = (
        neutral_observations
        .filter(
            ee.Filter.eq(
                "station",
                station_name,
            )
        )
        .select(
            selected_columns
        )
    )

    station_info_ee = (
        station_collection
        .getInfo()
    )

    station_rows = [
        feature[
            "properties"
        ]
        for feature
        in station_info_ee[
            "features"
        ]
    ]

    neutral_base_rows.extend(
        station_rows
    )

    pd.DataFrame(
        neutral_base_rows
    ).to_csv(
        NEUTRAL_BASE_PATH,
        index=False,
    )

    print(
        "  rows saved:",
        len(
            neutral_base_rows
        ),
    )


neutral_modis_master = (
    pd.DataFrame(
        neutral_base_rows
    )
)

print()
print(
    "Rows:",
    len(
        neutral_modis_master
    ),
)

print(
    "Expected:",
    690,
)

print(
    "Saved:",
    NEUTRAL_BASE_PATH,
)

Selected columns: ['station', 'station_id', 'period_start', 'period_end', 'number_days', 'longitude', 'latitude', 'ET_mm_period', 'ET_mm_day', 'modis_good', 'modis_qc_good', 'period_within_analysis', 'target_support']
Processing: Pastos limpios
  rows saved: 138
Processing: Palma


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


  rows saved: 276
Processing: Bananera


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


  rows saved: 414
Processing: Manglar


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


  rows saved: 552
Processing: Bosque seco
  rows saved: 690

Rows: 690
Expected: 690
Saved: C:\Users\User\Desktop\cristian\GEE\26-et-downscaling-fundacion\outputs\diagnostics\neutral_modis_master_2021_2023.csv


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


In [4]:
# ============================================================
# Add meteorology + reference ET to the neutral master
# ============================================================

import importlib

import et_downscaling.meteorology as meteorology_module
import et_downscaling.reference_et as reference_et_module

importlib.reload(meteorology_module)
importlib.reload(reference_et_module)


METEO_REF_PATH = (
    REPO_ROOT
    / "outputs"
    / "diagnostics"
    / "neutral_meteo_reference_et_2021_2023.csv"
)

METEO_REF_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Reusable source inputs
# ============================================================

print(
    "Building meteorology inputs..."
)

meteorology_inputs = (
    meteorology_module
    .build_meteorology_inputs(
        station_footprints
    )
)


print(
    "Building reference ET inputs..."
)

reference_et_inputs = (
    reference_et_module
    .build_reference_et_inputs(
        station_footprints=station_footprints,
        meteorology_inputs=meteorology_inputs,
        start_date="2021-01-01",
        end_date="2024-01-01",
    )
)


# ============================================================
# Add meteorology + ETo/ETr to one neutral observation
# ============================================================

def add_meteo_reference_et(
    feature,
):
    feature = ee.Feature(
        feature
    )

    period_start = ee.Date(
        feature.get(
            "system:time_start"
        )
    )

    period_end = ee.Date(
        feature.get(
            "period_end"
        )
    )

    number_days = ee.Number(
        feature.get(
            "number_days"
        )
    )

    station_point = (
        ee.Geometry.Point(
            [
                feature.get(
                    "longitude"
                ),
                feature.get(
                    "latitude"
                ),
            ]
        )
    )

    meteorology = (
        meteorology_module
        .get_meteorological_properties(
            period_start=period_start,
            period_end=period_end,
            number_days=number_days,
            station_point=station_point,
            meteorology_inputs=meteorology_inputs,
        )
    )

    reference_et = (
        reference_et_module
        .get_reference_et_properties(
            period_start=period_start,
            period_end=period_end,
            station_id=feature.get(
                "station_id"
            ),
            reference_et_inputs=reference_et_inputs,
        )
    )

    return (
        feature
        .set(
            meteorology
        )
        .set(
            reference_et
        )
    )


neutral_meteo_reference = (
    neutral_observations
    .map(
        add_meteo_reference_et
    )
)


# ============================================================
# Download in station × year blocks
# ============================================================

all_meteo_ref_rows = []


for station_info in stations:

    station_name = (
        station_info[
            "station"
        ]
    )

    print()
    print(
        "Station:",
        station_name,
    )

    for year in [
        2021,
        2022,
        2023,
    ]:

        year_start = (
            ee.Date(
                f"{year}-01-01"
            )
            .millis()
        )

        year_end = (
            ee.Date(
                f"{year + 1}-01-01"
            )
            .millis()
        )

        block = (
            neutral_meteo_reference
            .filter(
                ee.Filter.eq(
                    "station",
                    station_name,
                )
            )
            .filter(
                ee.Filter.gte(
                    "system:time_start",
                    year_start,
                )
            )
            .filter(
                ee.Filter.lt(
                    "system:time_start",
                    year_end,
                )
            )
        )

        print(
            "  Processing:",
            year,
        )

        block_info = (
            block.getInfo()
        )

        block_rows = [
            feature[
                "properties"
            ]
            for feature
            in block_info[
                "features"
            ]
        ]

        all_meteo_ref_rows.extend(
            block_rows
        )

        # Checkpoint after every station-year.
        pd.DataFrame(
            all_meteo_ref_rows
        ).to_csv(
            METEO_REF_PATH,
            index=False,
        )

        print(
            "    rows saved:",
            len(
                all_meteo_ref_rows
            ),
        )


# ============================================================
# Local QA
# ============================================================

meteo_reference_master = (
    pd.DataFrame(
        all_meteo_ref_rows
    )
)

print()
print(
    "Rows:",
    len(
        meteo_reference_master
    ),
)

print(
    "Expected:",
    690,
)


assert (
    len(
        meteo_reference_master
    )
    == 690
)


required_columns = [
    "Tair_mean_C",
    "VPD_mean_kPa",
    "SolarRad_MJ_m2_day",
    "Wind_mean_ms",
    "Precip_period_mm",
    "ETo_period_mm",
    "ETr_period_mm",
    "ETo_mean_mm_day",
    "ETr_mean_mm_day",
    "reference_et_complete",
]


missing_columns = [
    column
    for column
    in required_columns
    if column
    not in meteo_reference_master.columns
]


assert not missing_columns, (
    f"Missing expected columns: "
    f"{missing_columns}"
)


print(
    "Reference ET complete:",
    int(
        (
            meteo_reference_master[
                "reference_et_complete"
            ]
            == 1
        )
        .sum()
    ),
    "/ 690",
)


print(
    "Meteorology + reference ET checkpoint complete."
)

print(
    "Saved:",
    METEO_REF_PATH,
)

Building meteorology inputs...


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Building reference ET inputs...

Station: Pastos limpios
  Processing: 2021


EEException: User memory limit exceeded.

In [5]:
# ============================================================
# Reuse existing reference ET diagnostics when available
# ============================================================

from pathlib import Path
import pandas as pd


def find_repo_root(start=None):
    current = Path(
        start or Path.cwd()
    ).resolve()

    for candidate in [
        current,
        *current.parents,
    ]:
        if (
            candidate
            / "pyproject.toml"
        ).exists():
            return candidate

    raise FileNotFoundError(
        "Repository root not found."
    )


REPO_ROOT = find_repo_root()

DIAGNOSTIC_DIR = (
    REPO_ROOT
    / "outputs"
    / "diagnostics"
)

DAILY_REFERENCE_ET_PATH = (
    DIAGNOSTIC_DIR
    / "reference_et_daily_qa_2021_2023.csv"
)

PERIOD_REFERENCE_ET_PATH = (
    DIAGNOSTIC_DIR
    / "reference_et_modis_periods_2021_2023.csv"
)

NEUTRAL_MODIS_PATH = (
    DIAGNOSTIC_DIR
    / "neutral_modis_master_2021_2023.csv"
)


print(
    "Daily reference ET:",
    DAILY_REFERENCE_ET_PATH.exists(),
)

print(
    "MODIS-period reference ET:",
    PERIOD_REFERENCE_ET_PATH.exists(),
)

print(
    "Neutral MODIS master:",
    NEUTRAL_MODIS_PATH.exists(),
)


# ============================================================
# Case 1: period-level reference ET already exists
# ============================================================

if PERIOD_REFERENCE_ET_PATH.exists():

    reference_et_periods = pd.read_csv(
        PERIOD_REFERENCE_ET_PATH,
        parse_dates=[
            "period_start",
            "period_end",
        ],
    )

    print()
    print(
        "Existing period-level reference ET loaded."
    )

    print(
        "Rows:",
        len(reference_et_periods),
    )

    if (
        "reference_et_complete"
        in reference_et_periods.columns
    ):
        print(
            "Complete periods:",
            int(
                reference_et_periods[
                    "reference_et_complete"
                ].sum()
            ),
        )


# ============================================================
# Case 2: daily reference ET exists; aggregate locally
# ============================================================

elif (
    DAILY_REFERENCE_ET_PATH.exists()
    and NEUTRAL_MODIS_PATH.exists()
):

    daily_et = pd.read_csv(
        DAILY_REFERENCE_ET_PATH,
        dtype={
            "station_id": "string",
        },
        parse_dates=[
            "local_date",
        ],
    )

    neutral_modis = pd.read_csv(
        NEUTRAL_MODIS_PATH,
        dtype={
            "station_id": "string",
        },
        parse_dates=[
            "period_start",
            "period_end",
        ],
    )

    periods = (
        neutral_modis[
            [
                "period_start",
                "period_end",
                "number_days",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            "period_start"
        )
        .reset_index(
            drop=True
        )
    )

    stations = (
        neutral_modis[
            [
                "station",
                "station_id",
            ]
        ]
        .drop_duplicates()
    )

    rows = []


    for period in periods.itertuples(
        index=False
    ):

        period_daily = daily_et[
            (
                daily_et[
                    "local_date"
                ]
                >= period.period_start
            )
            &
            (
                daily_et[
                    "local_date"
                ]
                < period.period_end
            )
        ]

        for station in stations.itertuples(
            index=False
        ):

            station_daily = (
                period_daily[
                    period_daily[
                        "station"
                    ]
                    == station.station
                ]
                .copy()
            )

            n_days = len(
                station_daily
            )

            complete = (
                n_days
                == period.number_days
            )

            if (
                "era5_hours_total"
                in station_daily.columns
                and n_days > 0
            ):
                complete = (
                    complete
                    and station_daily[
                        "era5_hours_total"
                    ]
                    .eq(24)
                    .all()
                )

            rows.append(
                {
                    "station":
                        station.station,

                    "station_id":
                        station.station_id,

                    "period_start":
                        period.period_start,

                    "period_end":
                        period.period_end,

                    "number_days":
                        period.number_days,

                    "reference_et_days_total":
                        n_days,

                    "reference_et_complete":
                        int(
                            complete
                        ),

                    "ETo_period_mm":
                        station_daily[
                            "ETo_mm_day"
                        ]
                        .sum(),

                    "ETr_period_mm":
                        station_daily[
                            "ETr_mm_day"
                        ]
                        .sum(),

                    "ETo_mean_mm_day":
                        station_daily[
                            "ETo_mm_day"
                        ]
                        .mean(),

                    "ETr_mean_mm_day":
                        station_daily[
                            "ETr_mm_day"
                        ]
                        .mean(),
                }
            )


    reference_et_periods = (
        pd.DataFrame(
            rows
        )
    )

    reference_et_periods.to_csv(
        PERIOD_REFERENCE_ET_PATH,
        index=False,
    )

    print()
    print(
        "Period-level reference ET built locally."
    )

    print(
        "Rows:",
        len(reference_et_periods),
    )

    print(
        "Expected:",
        690,
    )

    print(
        "Complete periods:",
        int(
            reference_et_periods[
                "reference_et_complete"
            ]
            .sum()
        ),
    )

    print(
        "Saved:",
        PERIOD_REFERENCE_ET_PATH,
    )


# ============================================================
# Case 3: reference ET checkpoint does not exist
# ============================================================

else:

    print()
    print(
        "Reference ET checkpoint is not available."
    )

    print(
        "We need to rebuild the daily reference ET "
        "with a memory-safe station/year workflow."
    )

Daily reference ET: False
MODIS-period reference ET: False
Neutral MODIS master: True

Reference ET checkpoint is not available.
We need to rebuild the daily reference ET with a memory-safe station/year workflow.


In [6]:
# ============================================================
# Memory-safe daily reference ET extraction
# Station × year checkpoints
# ============================================================

from pathlib import Path
import importlib
import sys

import ee
import pandas as pd


# ============================================================
# Repository
# ============================================================

def find_repo_root(start=None):
    current = Path(
        start or Path.cwd()
    ).resolve()

    for candidate in [
        current,
        *current.parents,
    ]:
        if (
            candidate
            / "pyproject.toml"
        ).exists():
            return candidate

    raise FileNotFoundError(
        "Repository root not found."
    )


REPO_ROOT = find_repo_root()

SRC_PATH = (
    REPO_ROOT
    / "src"
)

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )


# ============================================================
# Earth Engine
# ============================================================

EE_PROJECT = "ee-change"

ee.Initialize(
    project=EE_PROJECT
)


# ============================================================
# Repository modules
# ============================================================

import et_downscaling.modis as modis_module
import et_downscaling.meteorology as meteorology_module
import et_downscaling.reference_et as reference_et_module

importlib.reload(modis_module)
importlib.reload(meteorology_module)
importlib.reload(reference_et_module)


# ============================================================
# Inputs
# ============================================================

modis_inputs = (
    modis_module
    .build_modis_inputs()
)

station_footprints = (
    ee.FeatureCollection(
        modis_inputs[
            "station_footprints"
        ]
    )
)

meteorology_inputs = (
    meteorology_module
    .build_meteorology_inputs(
        station_footprints
    )
)


# ============================================================
# Output checkpoint
# ============================================================

OUTPUT_DIR = (
    REPO_ROOT
    / "outputs"
    / "diagnostics"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

DAILY_REFERENCE_ET_PATH = (
    OUTPUT_DIR
    / "reference_et_daily_qa_2021_2023.csv"
)


# ============================================================
# Daily variables
# ============================================================

QA_BANDS = [
    "Tmin_day_C",
    "Tmax_day_C",
    "Tmean_day_C",
    "ea_day_kPa",
    "Wind10m_mean_ms",
    "Wind2m_mean_ms",
    "Rs_day_MJ_m2",
    "Rso_day_MJ_m2",
    "Rs_Rso_raw",
    "Rs_Rso_used",
    "Rn_day_MJ_m2",
    "ETo_mm_day",
    "ETr_mm_day",
]


# ============================================================
# Load existing checkpoint if present
# ============================================================

if DAILY_REFERENCE_ET_PATH.exists():

    daily_rows_df = pd.read_csv(
        DAILY_REFERENCE_ET_PATH,
        dtype={
            "station_id": "string",
        },
    )

    print(
        "Existing checkpoint rows:",
        len(daily_rows_df),
    )

else:

    daily_rows_df = pd.DataFrame()

    print(
        "No existing checkpoint."
    )


# ============================================================
# Process one year at a time
# ============================================================

for year in [
    2021,
    2022,
    2023,
]:

    year_start = (
        f"{year}-01-01"
    )

    year_end = (
        f"{year + 1}-01-01"
    )

    print()
    print(
        "=" * 60
    )

    print(
        "YEAR:",
        year,
    )

    print(
        "=" * 60
    )

    # --------------------------------------------------------
    # Build only one year of daily ERA5/reference-ET inputs
    # --------------------------------------------------------

    reference_et_inputs = (
        reference_et_module
        .build_reference_et_inputs(
            station_footprints=(
                station_footprints
            ),
            meteorology_inputs=(
                meteorology_inputs
            ),
            start_date=year_start,
            end_date=year_end,
        )
    )

    # --------------------------------------------------------
    # Materialize only five static station supports
    # --------------------------------------------------------

    station_support_info = (
        ee.FeatureCollection(
            reference_et_inputs[
                "station_supports"
            ]
        )
        .getInfo()
    )

    station_supports = [
        feature["properties"]
        for feature
        in station_support_info[
            "features"
        ]
    ]

    daily_meteorology = (
        ee.ImageCollection(
            reference_et_inputs[
                "daily_meteorology"
            ]
        )
    )

    era5_projection = (
        reference_et_inputs[
            "era5_projection"
        ]
    )

    era5_scale = (
        reference_et_inputs[
            "era5_scale"
        ]
    )

    # --------------------------------------------------------
    # Process each station separately
    # --------------------------------------------------------

    for support in station_supports:

        station = (
            support[
                "station"
            ]
        )

        station_id = str(
            support[
                "station_id"
            ]
        )

        # ----------------------------------------------------
        # Skip complete checkpoint block
        # ----------------------------------------------------

        if not daily_rows_df.empty:

            existing_block = (
                daily_rows_df[
                    (
                        daily_rows_df[
                            "station"
                        ]
                        == station
                    )
                    &
                    (
                        pd.to_datetime(
                            daily_rows_df[
                                "local_date"
                            ]
                        )
                        .dt.year
                        == year
                    )
                ]
            )

            expected_days = (
                pd.Timestamp(
                    year_end
                )
                - pd.Timestamp(
                    year_start
                )
            ).days

            if (
                len(existing_block)
                == expected_days
            ):

                print(
                    station,
                    year,
                    "-> cached",
                )

                continue

        print(
            station,
            year,
            "-> processing",
        )

        latitude = (
            support[
                "footprint_centroid_latitude"
            ]
        )

        elevation = (
            support[
                "footprint_mean_elevation_m"
            ]
        )

        era5_point = (
            ee.Geometry.Point(
                [
                    support[
                        "era5_sampling_longitude"
                    ],
                    support[
                        "era5_sampling_latitude"
                    ],
                ]
            )
        )

        # ----------------------------------------------------
        # Daily ETo/ETr for this station and this year only
        # ----------------------------------------------------

        daily_reference = (
            daily_meteorology
            .map(
                lambda image:
                    reference_et_module
                    .calculate_daily_reference_et(
                        image,
                        latitude,
                        elevation,
                    )
            )
        )


        def image_to_feature(image):

            image = ee.Image(
                image
            )

            values = (
                image
                .select(
                    QA_BANDS
                )
                .reduceRegion(
                    reducer=ee.Reducer.first(),
                    geometry=era5_point,
                    crs=era5_projection,
                    scale=era5_scale,
                    maxPixels=100,
                )
            )

            return (
                ee.Feature(
                    None,
                    values,
                )
                .set(
                    {
                        "station":
                            station,

                        "station_id":
                            station_id,

                        "local_date":
                            image.get(
                                "local_date"
                            ),

                        "era5_hours_total":
                            image.get(
                                "era5_hours_total"
                            ),

                        "elevation_mean_m":
                            elevation,
                    }
                )
            )


        station_year_fc = (
            ee.FeatureCollection(
                daily_reference
                .map(
                    image_to_feature
                )
            )
        )

        # ----------------------------------------------------
        # Only ~365 features evaluated at once
        # ----------------------------------------------------

        station_year_info = (
            station_year_fc
            .getInfo()
        )

        block_rows = [
            feature[
                "properties"
            ]
            for feature
            in station_year_info[
                "features"
            ]
        ]

        block_df = pd.DataFrame(
            block_rows
        )

        # ----------------------------------------------------
        # Replace incomplete prior block, if any
        # ----------------------------------------------------

        if not daily_rows_df.empty:

            daily_dates = pd.to_datetime(
                daily_rows_df[
                    "local_date"
                ]
            )

            keep_mask = ~(
                (
                    daily_rows_df[
                        "station"
                    ]
                    == station
                )
                &
                (
                    daily_dates.dt.year
                    == year
                )
            )

            daily_rows_df = (
                daily_rows_df[
                    keep_mask
                ]
                .copy()
            )

        daily_rows_df = pd.concat(
            [
                daily_rows_df,
                block_df,
            ],
            ignore_index=True,
        )

        daily_rows_df = (
            daily_rows_df
            .sort_values(
                [
                    "station",
                    "local_date",
                ]
            )
            .reset_index(
                drop=True
            )
        )

        # ----------------------------------------------------
        # Checkpoint after every station-year
        # ----------------------------------------------------

        daily_rows_df.to_csv(
            DAILY_REFERENCE_ET_PATH,
            index=False,
        )

        print(
            "  block rows:",
            len(block_df),
        )

        print(
            "  total saved:",
            len(daily_rows_df),
        )


# ============================================================
# Final QA
# ============================================================

expected_daily_rows = (
    5
    * (
        365
        + 365
        + 365
    )
)

# 2021, 2022 and 2023 are non-leap years.
print()
print(
    "DAILY REFERENCE ET COMPLETE"
)

print(
    "Rows:",
    len(daily_rows_df),
)

print(
    "Expected:",
    expected_daily_rows,
)

print(
    "Saved:",
    DAILY_REFERENCE_ET_PATH,
)


daily_rows_df[
    "local_date"
] = pd.to_datetime(
    daily_rows_df[
        "local_date"
    ]
)

daily_rows_df[
    "reference_et_complete_day"
] = (
    daily_rows_df[
        "era5_hours_total"
    ]
    == 24
)


print(
    "Complete 24-hour days:",
    int(
        daily_rows_df[
            "reference_et_complete_day"
        ]
        .sum()
    ),
)

print(
    "Missing ETo:",
    int(
        daily_rows_df[
            "ETo_mm_day"
        ]
        .isna()
        .sum()
    ),
)

print(
    "Missing ETr:",
    int(
        daily_rows_df[
            "ETr_mm_day"
        ]
        .isna()
        .sum()
    ),
)

No existing checkpoint.

YEAR: 2021


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Pastos limpios 2021 -> processing
  block rows: 365
  total saved: 365
Palma 2021 -> processing


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


  block rows: 365
  total saved: 730
Bananera 2021 -> processing


...

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


  block rows: 365
  total saved: 4745
Manglar 2023 -> processing


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


  block rows: 365
  total saved: 5110
Bosque seco 2023 -> processing
  block rows: 365
  total saved: 5475

DAILY REFERENCE ET COMPLETE
Rows: 5475
Expected: 5475
Saved: C:\Users\User\Desktop\cristian\GEE\26-et-downscaling-fundacion\outputs\diagnostics\reference_et_daily_qa_2021_2023.csv
Complete 24-hour days: 5475
Missing ETo: 0
Missing ETr: 0


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


In [7]:
# ============================================================
# Aggregate daily ETo/ETr to MODIS periods and build Kc target
# Local processing only - no Earth Engine
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# Paths
# ============================================================

OUTPUT_DIR = (
    REPO_ROOT
    / "outputs"
    / "diagnostics"
)

DAILY_REFERENCE_ET_PATH = (
    OUTPUT_DIR
    / "reference_et_daily_qa_2021_2023.csv"
)

NEUTRAL_MODIS_PATH = (
    OUTPUT_DIR
    / "neutral_modis_master_2021_2023.csv"
)

PERIOD_REFERENCE_ET_PATH = (
    OUTPUT_DIR
    / "reference_et_modis_periods_2021_2023.csv"
)

NEUTRAL_TARGET_PATH = (
    OUTPUT_DIR
    / "neutral_modis_reference_et_target_2021_2023.csv"
)


# ============================================================
# Load
# ============================================================

daily_et = pd.read_csv(
    DAILY_REFERENCE_ET_PATH,
)

neutral_modis = pd.read_csv(
    NEUTRAL_MODIS_PATH,
)


# ============================================================
# Parse dates
# ============================================================

daily_et[
    "local_date"
] = pd.to_datetime(
    daily_et[
        "local_date"
    ]
)

neutral_modis[
    "period_start"
] = pd.to_datetime(
    neutral_modis[
        "period_start"
    ]
)

neutral_modis[
    "period_end"
] = pd.to_datetime(
    neutral_modis[
        "period_end"
    ]
)


# ============================================================
# Aggregate reference ET to each station-period
# ============================================================

period_rows = []


for row in neutral_modis.itertuples(
    index=False
):

    station_daily = daily_et[
        (
            daily_et[
                "station"
            ]
            == row.station
        )
        &
        (
            daily_et[
                "local_date"
            ]
            >= row.period_start
        )
        &
        (
            daily_et[
                "local_date"
            ]
            < row.period_end
        )
    ]

    days_total = len(
        station_daily
    )

    expected_days = int(
        row.number_days
    )

    complete = (
        days_total
        == expected_days
    )

    if (
        "era5_hours_total"
        in station_daily.columns
    ):
        complete = (
            complete
            and station_daily[
                "era5_hours_total"
            ]
            .eq(24)
            .all()
        )

    period_rows.append(
        {
            "station":
                row.station,

            "period_start":
                row.period_start,

            "period_end":
                row.period_end,

            "reference_et_days_total":
                days_total,

            "reference_et_complete":
                int(
                    complete
                ),

            "ETo_period_mm":
                station_daily[
                    "ETo_mm_day"
                ]
                .sum(
                    min_count=1
                ),

            "ETr_period_mm":
                station_daily[
                    "ETr_mm_day"
                ]
                .sum(
                    min_count=1
                ),

            "ETo_mean_mm_day":
                station_daily[
                    "ETo_mm_day"
                ]
                .mean(),

            "ETr_mean_mm_day":
                station_daily[
                    "ETr_mm_day"
                ]
                .mean(),
        }
    )


reference_et_periods = pd.DataFrame(
    period_rows
)


# ============================================================
# Save period-level reference ET checkpoint
# ============================================================

reference_et_periods.to_csv(
    PERIOD_REFERENCE_ET_PATH,
    index=False,
)


# ============================================================
# Merge with neutral MODIS target
# ============================================================

neutral_target = neutral_modis.merge(
    reference_et_periods,
    on=[
        "station",
        "period_start",
        "period_end",
    ],
    how="left",
    validate="one_to_one",
)


# ============================================================
# Construct Kc target
# ============================================================

valid_target = (
    neutral_target[
        "modis_good"
    ]
    .eq(1)
    &
    neutral_target[
        "reference_et_complete"
    ]
    .eq(1)
    &
    neutral_target[
        "ET_mm_period"
    ]
    .notna()
    &
    neutral_target[
        "ETo_period_mm"
    ]
    .notna()
    &
    neutral_target[
        "ETo_period_mm"
    ]
    .gt(0)
)


neutral_target[
    "Kc_target"
] = np.where(
    valid_target,
    (
        neutral_target[
            "ET_mm_period"
        ]
        /
        neutral_target[
            "ETo_period_mm"
        ]
    ),
    np.nan,
)


neutral_target[
    "target_complete"
] = valid_target.astype(
    int
)


# ============================================================
# Internal consistency check
# ============================================================

neutral_target[
    "ET_reconstructed_mm_period"
] = (
    neutral_target[
        "Kc_target"
    ]
    *
    neutral_target[
        "ETo_period_mm"
    ]
)


valid_rows = neutral_target[
    neutral_target[
        "target_complete"
    ]
    == 1
]


max_reconstruction_error = (
    (
        valid_rows[
            "ET_reconstructed_mm_period"
        ]
        -
        valid_rows[
            "ET_mm_period"
        ]
    )
    .abs()
    .max()
)


# ============================================================
# Save final neutral target checkpoint
# ============================================================

neutral_target.to_csv(
    NEUTRAL_TARGET_PATH,
    index=False,
)


# ============================================================
# QA
# ============================================================

print(
    "Rows:",
    len(
        neutral_target
    ),
)

print(
    "Expected:",
    690,
)

print(
    "Reference ET complete:",
    int(
        neutral_target[
            "reference_et_complete"
        ]
        .sum()
    ),
)

print(
    "Valid MODIS targets:",
    int(
        neutral_target[
            "modis_good"
        ]
        .eq(1)
        .sum()
    ),
)

print(
    "Complete Kc targets:",
    int(
        neutral_target[
            "target_complete"
        ]
        .sum()
    ),
)

print()

print(
    "ETo mean mm/day:",
    round(
        neutral_target[
            "ETo_mean_mm_day"
        ]
        .mean(),
        3,
    ),
)

print(
    "ETr mean mm/day:",
    round(
        neutral_target[
            "ETr_mean_mm_day"
        ]
        .mean(),
        3,
    ),
)

print(
    "Mean ETr / ETo:",
    round(
        (
            neutral_target[
                "ETr_period_mm"
            ]
            /
            neutral_target[
                "ETo_period_mm"
            ]
        )
        .mean(),
        3,
    ),
)

print()

print(
    "Kc min:",
    round(
        valid_rows[
            "Kc_target"
        ]
        .min(),
        3,
    ),
)

print(
    "Kc median:",
    round(
        valid_rows[
            "Kc_target"
        ]
        .median(),
        3,
    ),
)

print(
    "Kc max:",
    round(
        valid_rows[
            "Kc_target"
        ]
        .max(),
        3,
    ),
)

print()

print(
    "Maximum ET reconstruction error:",
    max_reconstruction_error,
)

print()

print(
    "Saved period reference ET:",
    PERIOD_REFERENCE_ET_PATH,
)

print(
    "Saved neutral target:",
    NEUTRAL_TARGET_PATH,
)

Rows: 690
Expected: 690
Reference ET complete: 0
Valid MODIS targets: 690
Complete Kc targets: 0

ETo mean mm/day: 4.079
ETr mean mm/day: 4.594
Mean ETr / ETo: 1.119

Kc min: nan
Kc median: nan
Kc max: nan

Maximum ET reconstruction error: nan

Saved period reference ET: C:\Users\User\Desktop\cristian\GEE\26-et-downscaling-fundacion\outputs\diagnostics\reference_et_modis_periods_2021_2023.csv
Saved neutral target: C:\Users\User\Desktop\cristian\GEE\26-et-downscaling-fundacion\outputs\diagnostics\neutral_modis_reference_et_target_2021_2023.csv


In [8]:
# ============================================================
# Correct reference ET aggregation to MODIS periods
# Use period_start + number_days as exclusive end
# ============================================================

import numpy as np
import pandas as pd


OUTPUT_DIR = (
    REPO_ROOT
    / "outputs"
    / "diagnostics"
)

DAILY_REFERENCE_ET_PATH = (
    OUTPUT_DIR
    / "reference_et_daily_qa_2021_2023.csv"
)

NEUTRAL_MODIS_PATH = (
    OUTPUT_DIR
    / "neutral_modis_master_2021_2023.csv"
)

PERIOD_REFERENCE_ET_PATH = (
    OUTPUT_DIR
    / "reference_et_modis_periods_2021_2023.csv"
)

NEUTRAL_TARGET_PATH = (
    OUTPUT_DIR
    / "neutral_modis_reference_et_target_2021_2023.csv"
)


# ============================================================
# Load data
# ============================================================

daily_et = pd.read_csv(
    DAILY_REFERENCE_ET_PATH
)

neutral_modis = pd.read_csv(
    NEUTRAL_MODIS_PATH
)


daily_et["local_date"] = pd.to_datetime(
    daily_et["local_date"]
)

neutral_modis["period_start"] = pd.to_datetime(
    neutral_modis["period_start"]
)

neutral_modis["period_end"] = pd.to_datetime(
    neutral_modis["period_end"]
)


# ============================================================
# Aggregate daily reference ET
# ============================================================

period_rows = []


for row in neutral_modis.itertuples(
    index=False
):

    expected_days = int(
        row.number_days
    )

    aggregation_end = (
        row.period_start
        + pd.Timedelta(
            days=expected_days
        )
    )

    station_daily = daily_et[
        (
            daily_et["station"]
            == row.station
        )
        &
        (
            daily_et["local_date"]
            >= row.period_start
        )
        &
        (
            daily_et["local_date"]
            < aggregation_end
        )
    ].copy()

    days_total = len(
        station_daily
    )

    complete = (
        days_total
        == expected_days
    )

    if (
        "era5_hours_total"
        in station_daily.columns
    ):
        complete = (
            complete
            and station_daily[
                "era5_hours_total"
            ]
            .eq(24)
            .all()
        )

    period_rows.append(
        {
            "station":
                row.station,

            "period_start":
                row.period_start,

            "period_end":
                row.period_end,

            "aggregation_end":
                aggregation_end,

            "number_days":
                expected_days,

            "reference_et_days_total":
                days_total,

            "reference_et_complete":
                int(
                    complete
                ),

            "ETo_period_mm":
                station_daily[
                    "ETo_mm_day"
                ]
                .sum(
                    min_count=1
                ),

            "ETr_period_mm":
                station_daily[
                    "ETr_mm_day"
                ]
                .sum(
                    min_count=1
                ),

            "ETo_mean_mm_day":
                station_daily[
                    "ETo_mm_day"
                ]
                .mean(),

            "ETr_mean_mm_day":
                station_daily[
                    "ETr_mm_day"
                ]
                .mean(),
        }
    )


reference_et_periods = pd.DataFrame(
    period_rows
)


# ============================================================
# QA temporal support before building Kc
# ============================================================

reference_et_periods[
    "day_count_difference"
] = (
    reference_et_periods[
        "reference_et_days_total"
    ]
    -
    reference_et_periods[
        "number_days"
    ]
)


print(
    "Day-count differences:"
)

print(
    reference_et_periods[
        "day_count_difference"
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# Save corrected reference ET periods
# ============================================================

reference_et_periods.to_csv(
    PERIOD_REFERENCE_ET_PATH,
    index=False,
)


# ============================================================
# Merge with MODIS
# ============================================================

reference_columns = [
    "station",
    "period_start",
    "period_end",
    "reference_et_days_total",
    "reference_et_complete",
    "ETo_period_mm",
    "ETr_period_mm",
    "ETo_mean_mm_day",
    "ETr_mean_mm_day",
]


neutral_target = neutral_modis.merge(
    reference_et_periods[
        reference_columns
    ],
    on=[
        "station",
        "period_start",
        "period_end",
    ],
    how="left",
    validate="one_to_one",
)


# ============================================================
# Build Kc target
# ============================================================

valid_target = (
    neutral_target[
        "modis_good"
    ]
    .eq(1)
    &
    neutral_target[
        "reference_et_complete"
    ]
    .eq(1)
    &
    neutral_target[
        "ET_mm_period"
    ]
    .notna()
    &
    neutral_target[
        "ETo_period_mm"
    ]
    .notna()
    &
    neutral_target[
        "ETo_period_mm"
    ]
    .gt(0)
)


neutral_target[
    "Kc_target"
] = np.where(
    valid_target,
    (
        neutral_target[
            "ET_mm_period"
        ]
        /
        neutral_target[
            "ETo_period_mm"
        ]
    ),
    np.nan,
)


neutral_target[
    "target_complete"
] = valid_target.astype(
    int
)


# ============================================================
# Reconstruction control
# ============================================================

neutral_target[
    "ET_reconstructed_mm_period"
] = (
    neutral_target[
        "Kc_target"
    ]
    *
    neutral_target[
        "ETo_period_mm"
    ]
)


valid_rows = neutral_target[
    neutral_target[
        "target_complete"
    ]
    == 1
].copy()


max_reconstruction_error = (
    (
        valid_rows[
            "ET_reconstructed_mm_period"
        ]
        -
        valid_rows[
            "ET_mm_period"
        ]
    )
    .abs()
    .max()
)


# ============================================================
# Save
# ============================================================

neutral_target.to_csv(
    NEUTRAL_TARGET_PATH,
    index=False,
)


# ============================================================
# Final QA
# ============================================================

print()
print(
    "Rows:",
    len(
        neutral_target
    ),
)

print(
    "Reference ET complete:",
    int(
        neutral_target[
            "reference_et_complete"
        ]
        .sum()
    ),
)

print(
    "Complete Kc targets:",
    int(
        neutral_target[
            "target_complete"
        ]
        .sum()
    ),
)

print()

print(
    "ETo mean mm/day:",
    round(
        neutral_target[
            "ETo_mean_mm_day"
        ]
        .mean(),
        3,
    ),
)

print(
    "ETr mean mm/day:",
    round(
        neutral_target[
            "ETr_mean_mm_day"
        ]
        .mean(),
        3,
    ),
)

print(
    "Mean ETr / ETo:",
    round(
        (
            neutral_target[
                "ETr_period_mm"
            ]
            /
            neutral_target[
                "ETo_period_mm"
            ]
        )
        .mean(),
        3,
    ),
)

print()

print(
    "Kc min:",
    round(
        valid_rows[
            "Kc_target"
        ]
        .min(),
        3,
    ),
)

print(
    "Kc median:",
    round(
        valid_rows[
            "Kc_target"
        ]
        .median(),
        3,
    ),
)

print(
    "Kc max:",
    round(
        valid_rows[
            "Kc_target"
        ]
        .max(),
        3,
    ),
)

print()

print(
    "Maximum reconstruction error:",
    max_reconstruction_error,
)

Day-count differences:
day_count_difference
0    690
Name: count, dtype: int64

Rows: 690
Reference ET complete: 690
Complete Kc targets: 690

ETo mean mm/day: 4.076
ETr mean mm/day: 4.589
Mean ETr / ETo: 1.119

Kc min: 0.404
Kc median: 0.997
Kc max: 2.33

Maximum reconstruction error: 7.105427357601002e-15


In [9]:
# ============================================================
# Sentinel-1 orbit reproducibility check
# R077 ASCENDING vs R142 DESCENDING
# ============================================================

import ee
import pandas as pd


# ============================================================
# Base homogeneous Sentinel-1 collection
# ============================================================

s1_base = (
    ee.ImageCollection(
        "COPERNICUS/S1_GRD"
    )
    .filterBounds(
        station_footprints.geometry()
    )
    .filterDate(
        "2021-01-01",
        "2024-01-01",
    )
    .filter(
        ee.Filter.eq(
            "instrumentMode",
            "IW",
        )
    )
    .filter(
        ee.Filter.listContains(
            "transmitterReceiverPolarisation",
            "VV",
        )
    )
    .filter(
        ee.Filter.listContains(
            "transmitterReceiverPolarisation",
            "VH",
        )
    )
    .map(
        lambda image:
            image.set(
                "date_key",
                image.date().format(
                    "yyyy-MM-dd"
                ),
            )
    )
)


# ============================================================
# Only the two conflicting configurations
# ============================================================

candidates = [
    {
        "orbit_pass": "ASCENDING",
        "relative_orbit": 77,
    },
    {
        "orbit_pass": "DESCENDING",
        "relative_orbit": 142,
    },
]


station_list = (
    station_footprints
    .toList(
        station_footprints.size()
    )
)

station_sequence = (
    ee.List.sequence(
        0,
        station_footprints
        .size()
        .subtract(1)
    )
)


candidate_collections = []


for candidate in candidates:

    orbit_pass = candidate[
        "orbit_pass"
    ]

    relative_orbit = candidate[
        "relative_orbit"
    ]

    candidate_collection = (
        s1_base
        .filter(
            ee.Filter.eq(
                "orbitProperties_pass",
                orbit_pass,
            )
        )
        .filter(
            ee.Filter.eq(
                "relativeOrbitNumber_start",
                relative_orbit,
            )
        )
    )


    def summarize_station(index):

        station = ee.Feature(
            station_list.get(
                index
            )
        )

        geometry = (
            station.geometry()
        )

        station_collection = (
            candidate_collection
            .filterBounds(
                geometry
            )
        )

        unique_dates = (
            ee.List(
                station_collection
                .aggregate_array(
                    "date_key"
                )
            )
            .distinct()
        )

        return ee.Feature(
            None,
            {
                "station":
                    station.get(
                        "station"
                    ),

                "orbit_pass":
                    orbit_pass,

                "relative_orbit":
                    relative_orbit,

                "scenes":
                    station_collection
                    .size(),

                "unique_dates":
                    unique_dates
                    .size(),
            }
        )


    candidate_collections.append(
        ee.FeatureCollection(
            station_sequence.map(
                summarize_station
            )
        )
    )


orbit_check = (
    candidate_collections[0]
    .merge(
        candidate_collections[1]
    )
)


# ============================================================
# One Earth Engine evaluation
# ============================================================

orbit_info = (
    orbit_check
    .getInfo()
)


orbit_rows = [
    feature["properties"]
    for feature
    in orbit_info["features"]
]


orbit_table = (
    pd.DataFrame(
        orbit_rows
    )
    .sort_values(
        [
            "orbit_pass",
            "relative_orbit",
            "station",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "ORBIT COVERAGE BY STATION"
)

print(
    orbit_table.to_string(
        index=False
    )
)


# ============================================================
# Summary
# ============================================================

orbit_summary = (
    orbit_table
    .groupby(
        [
            "orbit_pass",
            "relative_orbit",
        ]
    )
    .agg(
        stations_covered=(
            "scenes",
            lambda values:
                values.gt(0).sum()
        ),
        minimum_dates=(
            "unique_dates",
            "min",
        ),
        maximum_dates=(
            "unique_dates",
            "max",
        ),
        total_scenes=(
            "scenes",
            "sum",
        ),
    )
    .reset_index()
)


print()
print(
    "CANDIDATE SUMMARY"
)

print(
    orbit_summary.to_string(
        index=False
    )
)

ORBIT COVERAGE BY STATION
orbit_pass  relative_orbit  scenes        station  unique_dates
 ASCENDING              77     113       Bananera           113
 ASCENDING              77     113    Bosque seco           113
 ASCENDING              77     113        Manglar           113
 ASCENDING              77     113          Palma           113
 ASCENDING              77     113 Pastos limpios           113
DESCENDING             142      86       Bananera            86
DESCENDING             142      86    Bosque seco            86
DESCENDING             142      86        Manglar            86
DESCENDING             142      86          Palma            86
DESCENDING             142      86 Pastos limpios            86

CANDIDATE SUMMARY
orbit_pass  relative_orbit  stations_covered  minimum_dates  maximum_dates  total_scenes
 ASCENDING              77                 5            113            113           565
DESCENDING             142                 5             86             8

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


In [10]:
# ============================================================
# Sentinel-1 acquisition availability by MODIS period
# R077 ASCENDING vs R142 DESCENDING
# ============================================================

import pandas as pd


# ============================================================
# Load the 138 MODIS periods locally
# ============================================================

neutral_modis = pd.read_csv(
    REPO_ROOT
    / "outputs"
    / "diagnostics"
    / "neutral_modis_master_2021_2023.csv"
)

neutral_modis["period_start"] = pd.to_datetime(
    neutral_modis["period_start"]
)

neutral_modis["number_days"] = pd.to_numeric(
    neutral_modis["number_days"]
)


periods = (
    neutral_modis[
        [
            "period_start",
            "number_days",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "period_start"
    )
    .reset_index(
        drop=True
    )
)

periods["aggregation_end"] = (
    periods["period_start"]
    + pd.to_timedelta(
        periods["number_days"],
        unit="D",
    )
)

print(
    "MODIS periods:",
    len(periods),
)


# ============================================================
# Candidate Sentinel-1 configurations
# ============================================================

candidate_definitions = [
    {
        "candidate": "R077_ASC",
        "orbit_pass": "ASCENDING",
        "relative_orbit": 77,
    },
    {
        "candidate": "R142_DESC",
        "orbit_pass": "DESCENDING",
        "relative_orbit": 142,
    },
]


results = []


for candidate in candidate_definitions:

    collection = (
        s1_base
        .filter(
            ee.Filter.eq(
                "orbitProperties_pass",
                candidate["orbit_pass"],
            )
        )
        .filter(
            ee.Filter.eq(
                "relativeOrbitNumber_start",
                candidate["relative_orbit"],
            )
        )
    )

    timestamps = (
        collection
        .aggregate_array(
            "system:time_start"
        )
        .getInfo()
    )

    acquisition_dates = (
        pd.to_datetime(
            timestamps,
            unit="ms",
            utc=True,
        )
        .tz_convert(None)
        .normalize()
        .drop_duplicates()
        .sort_values()
    )

    for period in periods.itertuples(
        index=False
    ):

        dates_in_period = acquisition_dates[
            (
                acquisition_dates
                >= period.period_start
            )
            &
            (
                acquisition_dates
                < period.aggregation_end
            )
        ]

        results.append(
            {
                "candidate":
                    candidate["candidate"],

                "period_start":
                    period.period_start,

                "dates_total":
                    len(
                        dates_in_period
                    ),

                "has_acquisition":
                    int(
                        len(
                            dates_in_period
                        )
                        > 0
                    ),
            }
        )


period_availability = pd.DataFrame(
    results
)


# ============================================================
# Summary
# ============================================================

summary = (
    period_availability
    .groupby(
        "candidate"
    )
    .agg(
        periods_total=(
            "period_start",
            "size",
        ),
        periods_with_acquisition=(
            "has_acquisition",
            "sum",
        ),
        mean_dates_per_period=(
            "dates_total",
            "mean",
        ),
        max_dates_per_period=(
            "dates_total",
            "max",
        ),
    )
    .reset_index()
)

summary[
    "availability_pct"
] = (
    100
    * summary[
        "periods_with_acquisition"
    ]
    / summary[
        "periods_total"
    ]
)


print()
print(
    "SENTINEL-1 PERIOD AVAILABILITY"
)

print(
    summary.to_string(
        index=False
    )
)


# ============================================================
# Paired comparison
# ============================================================

paired = (
    period_availability
    .pivot(
        index="period_start",
        columns="candidate",
        values="has_acquisition",
    )
    .reset_index()
)


paired_summary = {
    "both":
        int(
            (
                (
                    paired["R077_ASC"]
                    == 1
                )
                &
                (
                    paired["R142_DESC"]
                    == 1
                )
            )
            .sum()
        ),

    "R077_only":
        int(
            (
                (
                    paired["R077_ASC"]
                    == 1
                )
                &
                (
                    paired["R142_DESC"]
                    == 0
                )
            )
            .sum()
        ),

    "R142_only":
        int(
            (
                (
                    paired["R077_ASC"]
                    == 0
                )
                &
                (
                    paired["R142_DESC"]
                    == 1
                )
            )
            .sum()
        ),

    "neither":
        int(
            (
                (
                    paired["R077_ASC"]
                    == 0
                )
                &
                (
                    paired["R142_DESC"]
                    == 0
                )
            )
            .sum()
        ),
}


print()
print(
    "PAIRED PERIOD SUPPORT"
)

for key, value in paired_summary.items():
    print(
        f"{key}: {value}"
    )

MODIS periods: 138


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(



SENTINEL-1 PERIOD AVAILABILITY
candidate  periods_total  periods_with_acquisition  mean_dates_per_period  max_dates_per_period  availability_pct
 R077_ASC            138                       101               0.818841                     2         73.188406
R142_DESC            138                        86               0.623188                     1         62.318841

PAIRED PERIOD SUPPORT
both: 56
R077_only: 45
R142_only: 30
neither: 7


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


In [11]:
# ============================================================
# Sentinel-1 neutral footprint extraction
# R077 ASCENDING
# Station × year checkpoints
# ============================================================

from pathlib import Path
import importlib

import ee
import numpy as np
import pandas as pd

import et_downscaling.sentinel1 as sentinel1_module
from et_downscaling.config import (
    ANALYSIS_CRS,
    ANALYSIS_SCALE,
)

importlib.reload(sentinel1_module)


# ============================================================
# Inputs / outputs
# ============================================================

DIAGNOSTIC_DIR = (
    REPO_ROOT
    / "outputs"
    / "diagnostics"
)

TARGET_PATH = (
    DIAGNOSTIC_DIR
    / "neutral_modis_reference_et_target_2021_2023.csv"
)

S1_OUTPUT_PATH = (
    DIAGNOSTIC_DIR
    / "neutral_s1_footprint_2021_2023.csv"
)


target_table = pd.read_csv(
    TARGET_PATH
)

target_table["period_start"] = pd.to_datetime(
    target_table["period_start"]
)

target_table["number_days"] = pd.to_numeric(
    target_table["number_days"]
)


# ============================================================
# Current repository Sentinel-1 collection
# Must resolve to R077 ASCENDING
# ============================================================

s1_collection = (
    sentinel1_module
    .get_sentinel1_collection(
        station_footprints
    )
)

first_s1 = ee.Image(
    s1_collection.first()
)

print(
    "Repository S1 pass:",
    first_s1.get(
        "orbitProperties_pass"
    ).getInfo(),
)

print(
    "Repository S1 relative orbit:",
    first_s1.get(
        "relativeOrbitNumber_start"
    ).getInfo(),
)


# ============================================================
# Load checkpoint
# ============================================================

if S1_OUTPUT_PATH.exists():

    s1_table = pd.read_csv(
        S1_OUTPUT_PATH
    )

    print(
        "Existing checkpoint rows:",
        len(s1_table),
    )

else:

    s1_table = pd.DataFrame()

    print(
        "No existing S1 checkpoint."
    )


# ============================================================
# Helper
# ============================================================

def build_s1_period_feature(
    station_name,
    station_id,
    geometry,
    period_start,
    number_days,
):

    start_text = (
        period_start
        .strftime("%Y-%m-%d")
    )

    end_date = (
        period_start
        + pd.Timedelta(
            days=int(number_days)
        )
    )

    end_text = (
        end_date
        .strftime("%Y-%m-%d")
    )

    period_collection = (
        s1_collection
        .filterDate(
            start_text,
            end_text,
        )
        .filterBounds(
            geometry
        )
    )

    scene_count = (
        period_collection
        .size()
    )

    date_count = (
        period_collection
        .aggregate_array(
            "date_key"
        )
        .distinct()
        .size()
    )

    predictors = (
        sentinel1_module
        .build_s1_median(
            period_collection,
            geometry,
        )
    )

    coverage_fraction = (
        sentinel1_module
        .get_s1_coverage(
            predictors,
            geometry,
        )
    )

    raw_stats = ee.Dictionary(
        ee.Algorithms.If(
            scene_count.gt(0),
            predictors.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=geometry,
                crs=ANALYSIS_CRS,
                scale=ANALYSIS_SCALE,
                maxPixels=1e7,
                tileScale=8,
            ),
            ee.Dictionary({}),
        )
    )

    # Explicit missing-value sentinels make the
    # FeatureCollection safe to download.
    stats = (
        ee.Dictionary(
            {
                "VV_dB": -9999,
                "VH_dB": -9999,
                "VV_minus_VH_dB": -9999,
                "Angle_deg": -9999,
            }
        )
        .combine(
            raw_stats,
            overwrite=True,
        )
    )

    return ee.Feature(
        None,
        {
            "station":
                station_name,

            "station_id":
                station_id,

            "period_start":
                start_text,

            "period_end_exclusive":
                end_text,

            "number_days":
                int(number_days),

            "s1_scene_count":
                scene_count,

            "s1_date_count":
                date_count,

            "s1_acquisition_available":
                ee.Number(
                    scene_count.gt(0)
                ),

            "s1_coverage_fraction":
                coverage_fraction,

            "s1_coverage_pct":
                coverage_fraction
                .multiply(100),

            "VV_dB_mean":
                stats.get(
                    "VV_dB"
                ),

            "VH_dB_mean":
                stats.get(
                    "VH_dB"
                ),

            "VV_minus_VH_dB_mean":
                stats.get(
                    "VV_minus_VH_dB"
                ),

            # QA only; not a final model predictor.
            "Angle_deg_mean":
                stats.get(
                    "Angle_deg"
                ),

            "s1_pass":
                "ASCENDING",

            "s1_relative_orbit":
                77,
        },
    )


# ============================================================
# Extract station × year blocks
# ============================================================

for station_name in (
    target_table[
        "station"
    ]
    .drop_duplicates()
):

    station_feature = ee.Feature(
        station_footprints
        .filter(
            ee.Filter.eq(
                "station",
                station_name,
            )
        )
        .first()
    )

    station_geometry = (
        station_feature.geometry()
    )

    station_id = str(
        target_table.loc[
            target_table[
                "station"
            ]
            == station_name,
            "station_id",
        ]
        .iloc[0]
    )

    for year in [
        2021,
        2022,
        2023,
    ]:

        # ----------------------------------------------------
        # Skip completed checkpoint block
        # ----------------------------------------------------

        if not s1_table.empty:

            existing = s1_table[
                (
                    s1_table[
                        "station"
                    ]
                    == station_name
                )
                &
                (
                    pd.to_datetime(
                        s1_table[
                            "period_start"
                        ]
                    )
                    .dt.year
                    == year
                )
            ]

            expected_block_rows = len(
                target_table[
                    (
                        target_table[
                            "station"
                        ]
                        == station_name
                    )
                    &
                    (
                        target_table[
                            "period_start"
                        ]
                        .dt.year
                        == year
                    )
                ]
            )

            if (
                len(existing)
                == expected_block_rows
            ):

                print(
                    station_name,
                    year,
                    "-> cached",
                )

                continue

        print(
            station_name,
            year,
            "-> processing",
        )

        block_periods = (
            target_table[
                (
                    target_table[
                        "station"
                    ]
                    == station_name
                )
                &
                (
                    target_table[
                        "period_start"
                    ]
                    .dt.year
                    == year
                )
            ]
            [
                [
                    "period_start",
                    "number_days",
                ]
            ]
            .drop_duplicates()
            .sort_values(
                "period_start"
            )
        )

        ee_features = []

        for period in block_periods.itertuples(
            index=False
        ):

            ee_features.append(
                build_s1_period_feature(
                    station_name=station_name,
                    station_id=station_id,
                    geometry=station_geometry,
                    period_start=period.period_start,
                    number_days=period.number_days,
                )
            )

        block_info = (
            ee.FeatureCollection(
                ee_features
            )
            .getInfo()
        )

        block_rows = [
            feature[
                "properties"
            ]
            for feature
            in block_info[
                "features"
            ]
        ]

        block_df = pd.DataFrame(
            block_rows
        )

        # ----------------------------------------------------
        # Convert explicit missing sentinels to NaN
        # ----------------------------------------------------

        predictor_columns = [
            "VV_dB_mean",
            "VH_dB_mean",
            "VV_minus_VH_dB_mean",
            "Angle_deg_mean",
        ]

        for column in predictor_columns:

            block_df.loc[
                block_df[column]
                <= -9990,
                column,
            ] = np.nan

        # ----------------------------------------------------
        # Replace incomplete prior block
        # ----------------------------------------------------

        if not s1_table.empty:

            existing_dates = pd.to_datetime(
                s1_table[
                    "period_start"
                ]
            )

            keep = ~(
                (
                    s1_table[
                        "station"
                    ]
                    == station_name
                )
                &
                (
                    existing_dates
                    .dt.year
                    == year
                )
            )

            s1_table = (
                s1_table[
                    keep
                ]
                .copy()
            )

        s1_table = pd.concat(
            [
                s1_table,
                block_df,
            ],
            ignore_index=True,
        )

        s1_table = (
            s1_table
            .sort_values(
                [
                    "station",
                    "period_start",
                ]
            )
            .reset_index(
                drop=True
            )
        )

        s1_table.to_csv(
            S1_OUTPUT_PATH,
            index=False,
        )

        print(
            "  block rows:",
            len(block_df),
        )

        print(
            "  total saved:",
            len(s1_table),
        )


# ============================================================
# Final QA
# ============================================================

print()
print(
    "SENTINEL-1 FOOTPRINT EXTRACTION"
)

print(
    "Rows:",
    len(s1_table),
)

print(
    "Expected:",
    690,
)

print(
    "With acquisition:",
    int(
        s1_table[
            "s1_acquisition_available"
        ]
        .sum()
    ),
)

print(
    "With VV/VH predictors:",
    int(
        (
            s1_table[
                [
                    "VV_dB_mean",
                    "VH_dB_mean",
                    "VV_minus_VH_dB_mean",
                ]
            ]
            .notna()
            .all(
                axis=1
            )
        )
        .sum()
    ),
)

print()

print(
    "Coverage >= 99%:",
    int(
        (
            s1_table[
                "s1_coverage_pct"
            ]
            >= 99
        )
        .sum()
    ),
)

print(
    "Coverage > 0%:",
    int(
        (
            s1_table[
                "s1_coverage_pct"
            ]
            > 0
        )
        .sum()
    ),
)

print()

print(
    "Saved:",
    S1_OUTPUT_PATH,
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Repository S1 pass: ASCENDING
Repository S1 relative orbit: 77
No existing S1 checkpoint.
Pastos limpios 2021 -> processing


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


  block rows: 46
  total saved: 46
Pastos limpios 2022 -> processing


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


...

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


  block rows: 46
  total saved: 598
Bosque seco 2022 -> processing


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


  block rows: 46
  total saved: 644
Bosque seco 2023 -> processing
  block rows: 46
  total saved: 690

SENTINEL-1 FOOTPRINT EXTRACTION
Rows: 690
Expected: 690
With acquisition: 505
With VV/VH predictors: 505

Coverage >= 99%: 505
Coverage > 0%: 505

Saved: C:\Users\User\Desktop\cristian\GEE\26-et-downscaling-fundacion\outputs\diagnostics\neutral_s1_footprint_2021_2023.csv


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


In [12]:
# ============================================================
# Neutral meteorology extraction
# Station × year checkpoints
# No optical, no S1, no reference ET recalculation
# ============================================================

from pathlib import Path
import importlib

import ee
import pandas as pd

import et_downscaling.meteorology as meteorology_module

importlib.reload(meteorology_module)


# ============================================================
# Paths
# ============================================================

DIAGNOSTIC_DIR = (
    REPO_ROOT
    / "outputs"
    / "diagnostics"
)

TARGET_PATH = (
    DIAGNOSTIC_DIR
    / "neutral_modis_reference_et_target_2021_2023.csv"
)

METEO_OUTPUT_PATH = (
    DIAGNOSTIC_DIR
    / "neutral_meteorology_2021_2023.csv"
)


# ============================================================
# Load local target table
# ============================================================

target_table = pd.read_csv(
    TARGET_PATH
)

target_table["period_start"] = pd.to_datetime(
    target_table["period_start"]
)

target_table["number_days"] = pd.to_numeric(
    target_table["number_days"]
)


# ============================================================
# Meteorological source inputs
# ============================================================

meteorology_inputs = (
    meteorology_module
    .build_meteorology_inputs(
        station_footprints
    )
)


# ============================================================
# Existing checkpoint
# ============================================================

if METEO_OUTPUT_PATH.exists():

    meteo_table = pd.read_csv(
        METEO_OUTPUT_PATH
    )

    print(
        "Existing checkpoint rows:",
        len(meteo_table),
    )

else:

    meteo_table = pd.DataFrame()

    print(
        "No existing meteorology checkpoint."
    )


# ============================================================
# Helper
# ============================================================

def build_meteorology_feature(
    station_name,
    station_id,
    longitude,
    latitude,
    period_start,
    number_days,
):

    start_date = ee.Date(
        period_start.strftime(
            "%Y-%m-%d"
        )
    )

    # Explicit exclusive end:
    # period_start + number_days
    end_date = start_date.advance(
        int(number_days),
        "day",
    )

    station_point = ee.Geometry.Point(
        [
            float(longitude),
            float(latitude),
        ]
    )

    properties = (
        meteorology_module
        .get_meteorological_properties(
            period_start=start_date,
            period_end=end_date,
            number_days=ee.Number(
                int(number_days)
            ),
            station_point=station_point,
            meteorology_inputs=meteorology_inputs,
        )
    )

    return (
        ee.Feature(
            None,
            properties,
        )
        .set(
            {
                "station":
                    station_name,

                "station_id":
                    station_id,

                "period_start":
                    period_start.strftime(
                        "%Y-%m-%d"
                    ),

                "period_end_exclusive":
                    (
                        period_start
                        + pd.Timedelta(
                            days=int(
                                number_days
                            )
                        )
                    )
                    .strftime(
                        "%Y-%m-%d"
                    ),

                "number_days":
                    int(
                        number_days
                    ),
            }
        )
    )


# ============================================================
# Process station × year
# ============================================================

for station_name in (
    target_table[
        "station"
    ]
    .drop_duplicates()
):

    station_rows = target_table[
        target_table[
            "station"
        ]
        == station_name
    ]

    station_id = str(
        station_rows[
            "station_id"
        ]
        .iloc[0]
    )

    longitude = float(
        station_rows[
            "longitude"
        ]
        .iloc[0]
    )

    latitude = float(
        station_rows[
            "latitude"
        ]
        .iloc[0]
    )


    for year in [
        2021,
        2022,
        2023,
    ]:

        expected_block = (
            target_table[
                (
                    target_table[
                        "station"
                    ]
                    == station_name
                )
                &
                (
                    target_table[
                        "period_start"
                    ]
                    .dt.year
                    == year
                )
            ]
        )

        expected_rows = len(
            expected_block
        )


        # ----------------------------------------------------
        # Skip complete checkpoint block
        # ----------------------------------------------------

        if not meteo_table.empty:

            meteo_dates = pd.to_datetime(
                meteo_table[
                    "period_start"
                ]
            )

            existing = meteo_table[
                (
                    meteo_table[
                        "station"
                    ]
                    == station_name
                )
                &
                (
                    meteo_dates
                    .dt.year
                    == year
                )
            ]

            if (
                len(existing)
                == expected_rows
            ):

                print(
                    station_name,
                    year,
                    "-> cached",
                )

                continue


        print(
            station_name,
            year,
            "-> processing",
        )


        ee_features = []

        for period in (
            expected_block
            .sort_values(
                "period_start"
            )
            .itertuples(
                index=False
            )
        ):

            ee_features.append(
                build_meteorology_feature(
                    station_name=station_name,
                    station_id=station_id,
                    longitude=longitude,
                    latitude=latitude,
                    period_start=period.period_start,
                    number_days=period.number_days,
                )
            )


        block_info = (
            ee.FeatureCollection(
                ee_features
            )
            .getInfo()
        )

        block_rows = [
            feature[
                "properties"
            ]
            for feature
            in block_info[
                "features"
            ]
        ]

        block_df = pd.DataFrame(
            block_rows
        )


        # ----------------------------------------------------
        # Replace previous incomplete block
        # ----------------------------------------------------

        if not meteo_table.empty:

            existing_dates = pd.to_datetime(
                meteo_table[
                    "period_start"
                ]
            )

            keep = ~(
                (
                    meteo_table[
                        "station"
                    ]
                    == station_name
                )
                &
                (
                    existing_dates
                    .dt.year
                    == year
                )
            )

            meteo_table = (
                meteo_table[
                    keep
                ]
                .copy()
            )


        meteo_table = pd.concat(
            [
                meteo_table,
                block_df,
            ],
            ignore_index=True,
        )

        meteo_table = (
            meteo_table
            .sort_values(
                [
                    "station",
                    "period_start",
                ]
            )
            .reset_index(
                drop=True
            )
        )


        meteo_table.to_csv(
            METEO_OUTPUT_PATH,
            index=False,
        )


        print(
            "  block rows:",
            len(block_df),
        )

        print(
            "  total saved:",
            len(meteo_table),
        )


# ============================================================
# Final QA
# ============================================================

print()
print(
    "NEUTRAL METEOROLOGY"
)

print(
    "Rows:",
    len(meteo_table),
)

print(
    "Expected:",
    690,
)


for column in [
    "Tair_mean_C",
    "VPD_mean_kPa",
    "Wind_mean_ms",
    "SolarRad_MJ_m2_day",
    "Precip_period_mm",
    "meteo_complete",
]:

    if column in meteo_table.columns:

        print(
            column,
            "missing:",
            int(
                meteo_table[
                    column
                ]
                .isna()
                .sum()
            ),
        )


if (
    "meteo_complete"
    in meteo_table.columns
):

    print(
        "Meteorology complete:",
        int(
            meteo_table[
                "meteo_complete"
            ]
            .eq(1)
            .sum()
        ),
        "/ 690",
    )


print()
print(
    "Saved:",
    METEO_OUTPUT_PATH,
)

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


No existing meteorology checkpoint.
Pastos limpios 2021 -> processing


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


EEException: User memory limit exceeded.

In [13]:
# ============================================================
# Build period meteorology from the existing daily checkpoint
# Local processing only
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


DIAGNOSTIC_DIR = (
    REPO_ROOT
    / "outputs"
    / "diagnostics"
)

DAILY_REFERENCE_ET_PATH = (
    DIAGNOSTIC_DIR
    / "reference_et_daily_qa_2021_2023.csv"
)

TARGET_PATH = (
    DIAGNOSTIC_DIR
    / "neutral_modis_reference_et_target_2021_2023.csv"
)

ERA5_PERIOD_PATH = (
    DIAGNOSTIC_DIR
    / "neutral_era5_meteorology_2021_2023.csv"
)


# ============================================================
# Load
# ============================================================

daily = pd.read_csv(
    DAILY_REFERENCE_ET_PATH
)

target = pd.read_csv(
    TARGET_PATH
)


daily["local_date"] = pd.to_datetime(
    daily["local_date"]
)

target["period_start"] = pd.to_datetime(
    target["period_start"]
)


# ============================================================
# Derive daily vapor-pressure variables
# ============================================================

def saturation_vapor_pressure(
    temperature_c,
):
    return (
        0.6108
        * np.exp(
            (
                17.27
                * temperature_c
            )
            /
            (
                temperature_c
                + 237.3
            )
        )
    )


daily["es_tmin_kPa"] = (
    saturation_vapor_pressure(
        daily["Tmin_day_C"]
    )
)

daily["es_tmax_kPa"] = (
    saturation_vapor_pressure(
        daily["Tmax_day_C"]
    )
)

daily["es_day_kPa"] = (
    (
        daily["es_tmin_kPa"]
        + daily["es_tmax_kPa"]
    )
    / 2.0
)

daily["VPD_day_kPa"] = (
    daily["es_day_kPa"]
    - daily["ea_day_kPa"]
)


# Guard against tiny numerical negatives.
daily["VPD_day_kPa"] = (
    daily["VPD_day_kPa"]
    .clip(
        lower=0
    )
)


# ============================================================
# Dew-point temperature from actual vapor pressure
# ============================================================

ln_ratio = np.log(
    daily["ea_day_kPa"]
    / 0.6108
)

daily["Tdew_day_C"] = (
    237.3
    * ln_ratio
    /
    (
        17.27
        - ln_ratio
    )
)


# ============================================================
# Aggregate to every MODIS station-period
# ============================================================

period_rows = []


for row in target.itertuples(
    index=False
):

    end_exclusive = (
        row.period_start
        + pd.Timedelta(
            days=int(
                row.number_days
            )
        )
    )

    subset = daily[
        (
            daily["station"]
            == row.station
        )
        &
        (
            daily["local_date"]
            >= row.period_start
        )
        &
        (
            daily["local_date"]
            < end_exclusive
        )
    ].copy()

    expected_days = int(
        row.number_days
    )

    complete = (
        len(subset)
        == expected_days
    )

    complete = (
        complete
        and subset[
            "era5_hours_total"
        ]
        .eq(24)
        .all()
    )

    period_rows.append(
        {
            "station":
                row.station,

            "period_start":
                row.period_start,

            "number_days":
                expected_days,

            "era5_days_total":
                len(subset),

            "era5_temporal_complete":
                int(
                    complete
                ),

            "Tair_mean_C":
                subset[
                    "Tmean_day_C"
                ]
                .mean(),

            "Tair_min_C":
                subset[
                    "Tmin_day_C"
                ]
                .min(),

            "Tair_max_C":
                subset[
                    "Tmax_day_C"
                ]
                .max(),

            "Tdew_mean_C":
                subset[
                    "Tdew_day_C"
                ]
                .mean(),

            "VPD_mean_kPa":
                subset[
                    "VPD_day_kPa"
                ]
                .mean(),

            "Wind_mean_ms":
                subset[
                    "Wind10m_mean_ms"
                ]
                .mean(),

            "SolarRad_MJ_m2_day":
                subset[
                    "Rs_day_MJ_m2"
                ]
                .mean(),

            "Rn_MJ_m2_day":
                subset[
                    "Rn_day_MJ_m2"
                ]
                .mean(),
        }
    )


era5_period = pd.DataFrame(
    period_rows
)


era5_period.to_csv(
    ERA5_PERIOD_PATH,
    index=False,
)


# ============================================================
# QA
# ============================================================

print(
    "Rows:",
    len(
        era5_period
    )
)

print(
    "Expected:",
    690
)

print(
    "ERA5 complete:",
    int(
        era5_period[
            "era5_temporal_complete"
        ]
        .sum()
    )
)

print(
    "Day-count differences:"
)

print(
    (
        era5_period[
            "era5_days_total"
        ]
        -
        era5_period[
            "number_days"
        ]
    )
    .value_counts()
    .sort_index()
)

print()

for column in [
    "Tair_mean_C",
    "Tdew_mean_C",
    "VPD_mean_kPa",
    "Wind_mean_ms",
    "SolarRad_MJ_m2_day",
]:

    print(
        column,
        "missing:",
        int(
            era5_period[
                column
            ]
            .isna()
            .sum()
        ),
    )

print()

print(
    "Saved:",
    ERA5_PERIOD_PATH
)

Rows: 690
Expected: 690
ERA5 complete: 690
Day-count differences:
0    690
Name: count, dtype: int64

Tair_mean_C missing: 0
Tdew_mean_C missing: 0
VPD_mean_kPa missing: 0
Wind_mean_ms missing: 0
SolarRad_MJ_m2_day missing: 0

Saved: C:\Users\User\Desktop\cristian\GEE\26-et-downscaling-fundacion\outputs\diagnostics\neutral_era5_meteorology_2021_2023.csv


In [14]:
# ============================================================
# Lightweight CHIRPS extraction
# Station × year
# ============================================================

import ee
import pandas as pd


CHIRPS_OUTPUT_PATH = (
    DIAGNOSTIC_DIR
    / "neutral_chirps_2021_2023.csv"
)


chirps = (
    ee.ImageCollection(
        "UCSB-CHG/CHIRPS/DAILY"
    )
    .filterDate(
        "2021-01-01",
        "2024-01-01",
    )
    .select(
        "precipitation"
    )
)


if CHIRPS_OUTPUT_PATH.exists():

    chirps_table = pd.read_csv(
        CHIRPS_OUTPUT_PATH
    )

    print(
        "Existing checkpoint rows:",
        len(chirps_table)
    )

else:

    chirps_table = pd.DataFrame()

    print(
        "No existing CHIRPS checkpoint."
    )


for station_name in (
    target[
        "station"
    ]
    .drop_duplicates()
):

    station_data = target[
        target[
            "station"
        ]
        == station_name
    ]

    longitude = float(
        station_data[
            "longitude"
        ]
        .iloc[0]
    )

    latitude = float(
        station_data[
            "latitude"
        ]
        .iloc[0]
    )

    point = ee.Geometry.Point(
        [
            longitude,
            latitude,
        ]
    )


    for year in [
        2021,
        2022,
        2023,
    ]:

        block = target[
            (
                target[
                    "station"
                ]
                == station_name
            )
            &
            (
                target[
                    "period_start"
                ]
                .dt.year
                == year
            )
        ].copy()


        # --------------------------------------------
        # Resume from checkpoint
        # --------------------------------------------

        if not chirps_table.empty:

            existing_dates = pd.to_datetime(
                chirps_table[
                    "period_start"
                ]
            )

            existing = chirps_table[
                (
                    chirps_table[
                        "station"
                    ]
                    == station_name
                )
                &
                (
                    existing_dates
                    .dt.year
                    == year
                )
            ]

            if (
                len(existing)
                == len(block)
            ):

                print(
                    station_name,
                    year,
                    "-> cached",
                )

                continue


        print(
            station_name,
            year,
            "-> processing",
        )


        ee_features = []


        for period in block.itertuples(
            index=False
        ):

            start_text = (
                period.period_start
                .strftime(
                    "%Y-%m-%d"
                )
            )

            end_date = (
                period.period_start
                + pd.Timedelta(
                    days=int(
                        period.number_days
                    )
                )
            )

            end_text = (
                end_date
                .strftime(
                    "%Y-%m-%d"
                )
            )

            period_collection = (
                chirps
                .filterDate(
                    start_text,
                    end_text,
                )
            )

            day_count = (
                period_collection
                .size()
            )

            precipitation = (
                period_collection
                .sum()
                .reduceRegion(
                    reducer=ee.Reducer.first(),
                    geometry=point,
                    scale=5566,
                    maxPixels=100,
                )
                .get(
                    "precipitation"
                )
            )

            ee_features.append(
                ee.Feature(
                    None,
                    {
                        "station":
                            station_name,

                        "period_start":
                            start_text,

                        "number_days":
                            int(
                                period.number_days
                            ),

                        "chirps_days_total":
                            day_count,

                        "Precip_period_mm":
                            precipitation,
                    },
                )
            )


        block_info = (
            ee.FeatureCollection(
                ee_features
            )
            .getInfo()
        )


        block_df = pd.DataFrame(
            [
                feature[
                    "properties"
                ]
                for feature
                in block_info[
                    "features"
                ]
            ]
        )


        # --------------------------------------------
        # Replace partial block if necessary
        # --------------------------------------------

        if not chirps_table.empty:

            old_dates = pd.to_datetime(
                chirps_table[
                    "period_start"
                ]
            )

            keep = ~(
                (
                    chirps_table[
                        "station"
                    ]
                    == station_name
                )
                &
                (
                    old_dates
                    .dt.year
                    == year
                )
            )

            chirps_table = (
                chirps_table[
                    keep
                ]
                .copy()
            )


        chirps_table = pd.concat(
            [
                chirps_table,
                block_df,
            ],
            ignore_index=True,
        )


        chirps_table.to_csv(
            CHIRPS_OUTPUT_PATH,
            index=False,
        )


        print(
            "  block rows:",
            len(block_df)
        )

        print(
            "  total saved:",
            len(chirps_table)
        )


# ============================================================
# QA
# ============================================================

chirps_table[
    "chirps_complete"
] = (
    chirps_table[
        "chirps_days_total"
    ]
    ==
    chirps_table[
        "number_days"
    ]
).astype(
    int
)


chirps_table.to_csv(
    CHIRPS_OUTPUT_PATH,
    index=False,
)


print()
print(
    "CHIRPS"
)

print(
    "Rows:",
    len(chirps_table)
)

print(
    "Expected:",
    690
)

print(
    "Complete:",
    int(
        chirps_table[
            "chirps_complete"
        ]
        .sum()
    )
)

print(
    "Missing precipitation:",
    int(
        chirps_table[
            "Precip_period_mm"
        ]
        .isna()
        .sum()
    )
)

No existing CHIRPS checkpoint.
Pastos limpios 2021 -> processing


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


  block rows: 46
  total saved: 46
Pastos limpios 2022 -> processing


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


  block rows: 46
  total saved: 92
Pastos limpios 2023 -> processing


...

c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


  block rows: 46
  total saved: 598
Bosque seco 2022 -> processing


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


  block rows: 46
  total saved: 644
Bosque seco 2023 -> processing
  block rows: 46
  total saved: 690

CHIRPS
Rows: 690
Expected: 690
Complete: 690
Missing precipitation: 0


c:\Users\User\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


In [15]:
multisource = core.merge(
    optical_for_merge,
    on=KEYS,
    how="left",
    validate="one_to_many",
)

NameError: name 'core' is not defined

In [16]:
# ============================================================
# Rebuild neutral multisource master from saved checkpoints
# Local processing only
# ============================================================

from pathlib import Path

import pandas as pd


# ============================================================
# Repository
# ============================================================

def find_repo_root(start=None):
    current = Path(
        start or Path.cwd()
    ).resolve()

    for candidate in [
        current,
        *current.parents,
    ]:
        if (
            candidate
            / "pyproject.toml"
        ).exists():
            return candidate

    raise FileNotFoundError(
        "Repository root not found."
    )


REPO_ROOT = find_repo_root()

DIAGNOSTIC_DIR = (
    REPO_ROOT
    / "outputs"
    / "diagnostics"
)


# ============================================================
# Paths
# ============================================================

TARGET_PATH = (
    DIAGNOSTIC_DIR
    / "neutral_modis_reference_et_target_2021_2023.csv"
)

ERA5_PATH = (
    DIAGNOSTIC_DIR
    / "neutral_era5_meteorology_2021_2023.csv"
)

CHIRPS_PATH = (
    DIAGNOSTIC_DIR
    / "neutral_chirps_2021_2023.csv"
)

S1_PATH = (
    DIAGNOSTIC_DIR
    / "neutral_s1_footprint_2021_2023.csv"
)

OPTICAL_PATH = (
    DIAGNOSTIC_DIR
    / "optical_source_common_predictors.csv"
)

CORE_MASTER_PATH = (
    DIAGNOSTIC_DIR
    / "neutral_core_master_2021_2023.csv"
)

MULTISOURCE_MASTER_PATH = (
    DIAGNOSTIC_DIR
    / "neutral_multisource_master_2021_2023.csv"
)


# ============================================================
# Load checkpoints
# ============================================================

target = pd.read_csv(TARGET_PATH)
era5 = pd.read_csv(ERA5_PATH)
chirps = pd.read_csv(CHIRPS_PATH)
s1 = pd.read_csv(S1_PATH)
optical = pd.read_csv(OPTICAL_PATH)


for table in [
    target,
    era5,
    chirps,
    s1,
    optical,
]:
    table["period_start"] = pd.to_datetime(
        table["period_start"]
    )


KEYS = [
    "station",
    "period_start",
]


# ============================================================
# Check one-to-one core blocks
# ============================================================

for name, table in [
    ("target", target),
    ("era5", era5),
    ("chirps", chirps),
    ("s1", s1),
]:

    duplicates = (
        table
        .duplicated(KEYS)
        .sum()
    )

    if duplicates:
        raise ValueError(
            f"{name} has {duplicates} duplicate rows."
        )


# ============================================================
# Select independent predictor columns
# ============================================================

era5_columns = [
    column
    for column in [
        "station",
        "period_start",
        "era5_days_total",
        "era5_temporal_complete",
        "Tair_mean_C",
        "Tair_min_C",
        "Tair_max_C",
        "Tdew_mean_C",
        "VPD_mean_kPa",
        "Wind_mean_ms",
        "SolarRad_MJ_m2_day",
        "Rn_MJ_m2_day",
    ]
    if column in era5.columns
]


chirps_columns = [
    column
    for column in [
        "station",
        "period_start",
        "chirps_days_total",
        "chirps_complete",
        "Precip_period_mm",
    ]
    if column in chirps.columns
]


s1_columns = [
    column
    for column in [
        "station",
        "period_start",
        "s1_scene_count",
        "s1_date_count",
        "s1_acquisition_available",
        "s1_coverage_fraction",
        "s1_coverage_pct",
        "VV_dB_mean",
        "VH_dB_mean",
        "VV_minus_VH_dB_mean",
        "Angle_deg_mean",
        "s1_pass",
        "s1_relative_orbit",
    ]
    if column in s1.columns
]


# ============================================================
# Rebuild 690-row core master
# ============================================================

core = (
    target
    .merge(
        era5[era5_columns],
        on=KEYS,
        how="left",
        validate="one_to_one",
    )
    .merge(
        chirps[chirps_columns],
        on=KEYS,
        how="left",
        validate="one_to_one",
    )
    .merge(
        s1[s1_columns],
        on=KEYS,
        how="left",
        validate="one_to_one",
    )
)


if len(core) != 690:
    raise ValueError(
        f"Expected 690 core rows, found {len(core)}."
    )


print(
    "Core rows:",
    len(core)
)


# ============================================================
# Detect optical source
# ============================================================

source_candidates = [
    "source",
    "optical_source",
    "candidate",
]

source_column = next(
    (
        column
        for column in source_candidates
        if column in optical.columns
    ),
    None,
)

if source_column is None:
    raise ValueError(
        "Optical source column not found. "
        f"Columns: {list(optical.columns)}"
    )


print(
    "Optical source column:",
    source_column
)

print(
    "Optical source counts:"
)

print(
    optical[
        source_column
    ]
    .value_counts(
        dropna=False
    )
)


# ============================================================
# Optical uniqueness check
# ============================================================

optical_duplicates = (
    optical
    .duplicated(
        KEYS
        + [
            source_column
        ]
    )
    .sum()
)

if optical_duplicates:
    raise ValueError(
        f"Optical table has {optical_duplicates} duplicate "
        "station-period-source rows."
    )


# ============================================================
# Prepare optical table
# ============================================================

optical_for_merge = optical.drop(
    columns=[
        column
        for column in [
            "station_id",
            "period_end",
            "number_days",
            "longitude",
            "latitude",
        ]
        if column in optical.columns
    ],
    errors="ignore",
)


# ============================================================
# Merge optical alternatives
# ============================================================

multisource = core.merge(
    optical_for_merge,
    on=KEYS,
    how="left",
    validate="one_to_many",
)


print()
print(
    "Multisource rows:",
    len(multisource)
)


# ============================================================
# Integrity checks
# ============================================================

source_counts = (
    multisource[
        source_column
    ]
    .value_counts(
        dropna=False
    )
)

print()
print(
    "Rows by optical source:"
)

print(
    source_counts
)


if len(multisource) != 1380:
    raise ValueError(
        "Expected 1380 rows "
        "(690 S2 + 690 HLS combined), "
        f"found {len(multisource)}."
    )


if not source_counts.eq(690).all():
    raise ValueError(
        "Each optical source should contain exactly 690 rows."
    )


# ============================================================
# Detect optical coverage
# ============================================================

coverage_candidates = [
    "coverage_pct",
    "optical_coverage_pct",
    "coverage_percent",
    "coverage",
]

coverage_column = next(
    (
        column
        for column in coverage_candidates
        if column in multisource.columns
    ),
    None,
)

if coverage_column is None:

    possible_columns = [
        column
        for column in multisource.columns
        if (
            "coverage"
            in column.lower()
            and "s1"
            not in column.lower()
        )
    ]

    raise ValueError(
        "Optical coverage column not detected. "
        f"Possible columns: {possible_columns}"
    )


print()
print(
    "Optical coverage column:",
    coverage_column
)


if (
    multisource[
        coverage_column
    ]
    .max()
    <= 1.01
):

    multisource[
        "optical_coverage_pct"
    ] = (
        multisource[
            coverage_column
        ]
        * 100.0
    )

else:

    multisource[
        "optical_coverage_pct"
    ] = (
        multisource[
            coverage_column
        ]
    )


# ============================================================
# Predictor completeness
# ============================================================

multisource[
    "s1_predictors_complete"
] = (
    multisource[
        [
            "VV_dB_mean",
            "VH_dB_mean",
            "VV_minus_VH_dB_mean",
        ]
    ]
    .notna()
    .all(axis=1)
    .astype(int)
)


multisource[
    "base_predictors_complete"
] = (
    multisource[
        "era5_temporal_complete"
    ]
    .eq(1)
    &
    multisource[
        "chirps_complete"
    ]
    .eq(1)
    &
    multisource[
        "s1_predictors_complete"
    ]
    .eq(1)
).astype(int)


# ============================================================
# Optical support thresholds
# ============================================================

for threshold in [
    80,
    90,
    99,
]:

    multisource[
        f"optical_valid_{threshold}"
    ] = (
        multisource[
            "optical_coverage_pct"
        ]
        .ge(threshold)
        .astype(int)
    )

    multisource[
        f"model_support_{threshold}"
    ] = (
        multisource[
            "target_complete"
        ]
        .eq(1)
        &
        multisource[
            "base_predictors_complete"
        ]
        .eq(1)
        &
        multisource[
            f"optical_valid_{threshold}"
        ]
        .eq(1)
    ).astype(int)


# ============================================================
# Save
# ============================================================

core.to_csv(
    CORE_MASTER_PATH,
    index=False,
)

multisource.to_csv(
    MULTISOURCE_MASTER_PATH,
    index=False,
)


# ============================================================
# Final summary
# ============================================================

summary = (
    multisource
    .groupby(source_column)
    .agg(
        rows=(
            source_column,
            "size",
        ),
        target_complete=(
            "target_complete",
            "sum",
        ),
        s1_complete=(
            "s1_predictors_complete",
            "sum",
        ),
        optical_gt_0=(
            "optical_coverage_pct",
            lambda x: x.gt(0).sum(),
        ),
        optical_ge_80=(
            "optical_valid_80",
            "sum",
        ),
        optical_ge_90=(
            "optical_valid_90",
            "sum",
        ),
        optical_ge_99=(
            "optical_valid_99",
            "sum",
        ),
        joint_ge_80=(
            "model_support_80",
            "sum",
        ),
        joint_ge_90=(
            "model_support_90",
            "sum",
        ),
        joint_ge_99=(
            "model_support_99",
            "sum",
        ),
    )
    .reset_index()
)


print()
print(
    "NEUTRAL MULTISOURCE MASTER"
)

print(
    summary.to_string(
        index=False
    )
)

print()
print(
    "Saved:",
    MULTISOURCE_MASTER_PATH
)

Core rows: 690
Optical source column: source
Optical source counts:
source
S2              690
HLS_COMBINED    690
Name: count, dtype: int64

Multisource rows: 1380

Rows by optical source:
source
S2              690
HLS_COMBINED    690
Name: count, dtype: int64

Optical coverage column: coverage_pct

NEUTRAL MULTISOURCE MASTER
      source  rows  target_complete  s1_complete  optical_gt_0  optical_ge_80  optical_ge_90  optical_ge_99  joint_ge_80  joint_ge_90  joint_ge_99
HLS_COMBINED   690              690          505           489            381            361            336          273          257          238
          S2   690              690          505           586            497            477            440          365          349          319

Saved: C:\Users\User\Desktop\cristian\GEE\26-et-downscaling-fundacion\outputs\diagnostics\neutral_multisource_master_2021_2023.csv


In [18]:
import sklearn

print(
    "scikit-learn:",
    sklearn.__version__,
)

scikit-learn: 1.9.0


In [19]:
# ============================================================
# Controlled S2 vs HLS source comparison
# Same observations, same predictors, same spatial folds
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import LeaveOneGroupOut


# ============================================================
# Load neutral multisource master
# ============================================================

MULTISOURCE_PATH = (
    REPO_ROOT
    / "outputs"
    / "diagnostics"
    / "neutral_multisource_master_2021_2023.csv"
)

data = pd.read_csv(
    MULTISOURCE_PATH
)

data["period_start"] = pd.to_datetime(
    data["period_start"]
)


# ============================================================
# Shared predictor definitions
# ============================================================

optical_candidates = [
    "Blue",
    "Green",
    "Red",
    "NIR",
    "SWIR1",
    "SWIR2",
    "NDVI",
    "EVI",
    "SAVI",
    "NDWI",
    "NDMI",
]


def resolve_column(
    dataframe,
    base_name,
):
    candidates = [
        base_name,
        f"{base_name}_mean",
        base_name.lower(),
        f"{base_name.lower()}_mean",
    ]

    for candidate in candidates:
        if candidate in dataframe.columns:
            return candidate

    raise ValueError(
        f"Predictor not found for {base_name}. "
        f"Available columns: {list(dataframe.columns)}"
    )


optical_features = [
    resolve_column(
        data,
        predictor,
    )
    for predictor in optical_candidates
]


core_features = [
    "VV_dB_mean",
    "VH_dB_mean",
    "VV_minus_VH_dB_mean",
    "Tair_mean_C",
    "Tdew_mean_C",
    "VPD_mean_kPa",
    "Wind_mean_ms",
    "SolarRad_MJ_m2_day",
    "Precip_period_mm",
]


# ============================================================
# Time harmonics
# ============================================================

day_of_year = (
    data["period_start"]
    .dt.dayofyear
)

data["doy_sin"] = np.sin(
    2.0
    * np.pi
    * day_of_year
    / 365.25
)

data["doy_cos"] = np.cos(
    2.0
    * np.pi
    * day_of_year
    / 365.25
)


MODEL_FEATURES = (
    optical_features
    + core_features
    + [
        "doy_sin",
        "doy_cos",
    ]
)


# ============================================================
# KGE
# ============================================================

def kling_gupta_efficiency(
    observed,
    predicted,
):
    observed = np.asarray(
        observed,
        dtype=float,
    )

    predicted = np.asarray(
        predicted,
        dtype=float,
    )

    correlation = np.corrcoef(
        observed,
        predicted,
    )[0, 1]

    alpha = (
        np.std(
            predicted,
            ddof=1,
        )
        /
        np.std(
            observed,
            ddof=1,
        )
    )

    beta = (
        np.mean(
            predicted
        )
        /
        np.mean(
            observed
        )
    )

    return (
        1.0
        - np.sqrt(
            (
                correlation
                - 1.0
            ) ** 2
            +
            (
                alpha
                - 1.0
            ) ** 2
            +
            (
                beta
                - 1.0
            ) ** 2
        )
    )


# ============================================================
# Spatial leave-one-station-out evaluation
# ============================================================

def evaluate_source(
    dataframe,
):

    dataframe = (
        dataframe
        .sort_values(
            [
                "station",
                "period_start",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    X = dataframe[
        MODEL_FEATURES
    ]

    y = dataframe[
        "Kc_target"
    ]

    groups = dataframe[
        "station"
    ]

    logo = LeaveOneGroupOut()

    predictions = np.full(
        len(dataframe),
        np.nan,
    )

    baseline_predictions = np.full(
        len(dataframe),
        np.nan,
    )

    fold_rows = []


    for train_index, test_index in (
        logo.split(
            X,
            y,
            groups=groups,
        )
    ):

        model = RandomForestRegressor(
            n_estimators=500,
            max_features="sqrt",
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(
            X.iloc[
                train_index
            ],
            y.iloc[
                train_index
            ],
        )

        fold_prediction = model.predict(
            X.iloc[
                test_index
            ]
        )

        predictions[
            test_index
        ] = fold_prediction


        # Training-mean baseline.
        baseline_value = (
            y.iloc[
                train_index
            ]
            .mean()
        )

        baseline_predictions[
            test_index
        ] = baseline_value


        test_station = (
            groups.iloc[
                test_index
            ]
            .iloc[0]
        )

        fold_observed = (
            y.iloc[
                test_index
            ]
            .to_numpy()
        )


        fold_rows.append(
            {
                "station":
                    test_station,

                "n":
                    len(
                        test_index
                    ),

                "R2":
                    r2_score(
                        fold_observed,
                        fold_prediction,
                    ),

                "RMSE":
                    np.sqrt(
                        mean_squared_error(
                            fold_observed,
                            fold_prediction,
                        )
                    ),

                "MAE":
                    mean_absolute_error(
                        fold_observed,
                        fold_prediction,
                    ),

                "bias":
                    np.mean(
                        fold_prediction
                        - fold_observed
                    ),

                "KGE":
                    kling_gupta_efficiency(
                        fold_observed,
                        fold_prediction,
                    ),
            }
        )


    pooled_metrics = {
        "n":
            len(dataframe),

        "R2":
            r2_score(
                y,
                predictions,
            ),

        "RMSE":
            np.sqrt(
                mean_squared_error(
                    y,
                    predictions,
                )
            ),

        "MAE":
            mean_absolute_error(
                y,
                predictions,
            ),

        "bias":
            np.mean(
                predictions
                - y.to_numpy()
            ),

        "KGE":
            kling_gupta_efficiency(
                y,
                predictions,
            ),

        "baseline_R2":
            r2_score(
                y,
                baseline_predictions,
            ),

        "baseline_RMSE":
            np.sqrt(
                mean_squared_error(
                    y,
                    baseline_predictions,
                )
            ),
    }


    return (
        pooled_metrics,
        pd.DataFrame(
            fold_rows
        ),
    )


# ============================================================
# Compare optical sources at common paired support
# ============================================================

comparison_rows = []
fold_results = []


for threshold in [
    80,
    90,
    99,
]:

    support_column = (
        f"model_support_{threshold}"
    )

    s2 = (
        data[
            (
                data["source"]
                == "S2"
            )
            &
            (
                data[
                    support_column
                ]
                == 1
            )
        ]
        .copy()
    )

    hls = (
        data[
            (
                data["source"]
                == "HLS_COMBINED"
            )
            &
            (
                data[
                    support_column
                ]
                == 1
            )
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Keep exactly the same station-periods
    # --------------------------------------------------------

    common_keys = (
        s2[
            [
                "station",
                "period_start",
            ]
        ]
        .merge(
            hls[
                [
                    "station",
                    "period_start",
                ]
            ],
            on=[
                "station",
                "period_start",
            ],
            how="inner",
        )
        .drop_duplicates()
    )


    s2_common = (
        s2.merge(
            common_keys,
            on=[
                "station",
                "period_start",
            ],
            how="inner",
        )
        .sort_values(
            [
                "station",
                "period_start",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    hls_common = (
        hls.merge(
            common_keys,
            on=[
                "station",
                "period_start",
            ],
            how="inner",
        )
        .sort_values(
            [
                "station",
                "period_start",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    # --------------------------------------------------------
    # Require predictor completeness in BOTH alternatives
    # --------------------------------------------------------

    complete_pair = (
        s2_common[
            MODEL_FEATURES
        ]
        .notna()
        .all(
            axis=1
        )
        &
        hls_common[
            MODEL_FEATURES
        ]
        .notna()
        .all(
            axis=1
        )
    )


    s2_common = (
        s2_common[
            complete_pair
        ]
        .reset_index(
            drop=True
        )
    )

    hls_common = (
        hls_common[
            complete_pair
        ]
        .reset_index(
            drop=True
        )
    )


    assert len(
        s2_common
    ) == len(
        hls_common
    )


    assert np.allclose(
        s2_common[
            "Kc_target"
        ],
        hls_common[
            "Kc_target"
        ],
    )


    print()
    print(
        f"Threshold {threshold}%"
    )

    print(
        "Paired observations:",
        len(
            s2_common
        )
    )

    print(
        "By station:"
    )

    print(
        s2_common[
            "station"
        ]
        .value_counts()
        .sort_index()
        .to_string()
    )


    # --------------------------------------------------------
    # Evaluate
    # --------------------------------------------------------

    for source_name, source_data in [
        (
            "S2",
            s2_common,
        ),
        (
            "HLS_COMBINED",
            hls_common,
        ),
    ]:

        metrics, folds = evaluate_source(
            source_data
        )

        comparison_rows.append(
            {
                "threshold":
                    threshold,

                "source":
                    source_name,

                **metrics,
            }
        )

        folds[
            "threshold"
        ] = threshold

        folds[
            "source"
        ] = source_name

        fold_results.append(
            folds
        )


# ============================================================
# Results
# ============================================================

comparison = pd.DataFrame(
    comparison_rows
)

fold_comparison = pd.concat(
    fold_results,
    ignore_index=True,
)


print()
print(
    "=" * 70
)

print(
    "PAIRED SPATIAL SOURCE COMPARISON"
)

print(
    comparison[
        [
            "threshold",
            "source",
            "n",
            "R2",
            "RMSE",
            "MAE",
            "bias",
            "KGE",
            "baseline_R2",
            "baseline_RMSE",
        ]
    ]
    .round(4)
    .to_string(
        index=False
    )
)


# ============================================================
# Save
# ============================================================

comparison.to_csv(
    DIAGNOSTIC_DIR
    / "optical_source_model_comparison.csv",
    index=False,
)

fold_comparison.to_csv(
    DIAGNOSTIC_DIR
    / "optical_source_model_comparison_by_station.csv",
    index=False,
)


Threshold 80%
Paired observations: 259
By station:
station
Bananera          41
Bosque seco       58
Manglar           66
Palma             47
Pastos limpios    47

Threshold 90%
Paired observations: 243
By station:
station
Bananera          37
Bosque seco       56
Manglar           63
Palma             42
Pastos limpios    45

Threshold 99%
Paired observations: 223
By station:
station
Bananera          33
Bosque seco       52
Manglar           61
Palma             38
Pastos limpios    39

PAIRED SPATIAL SOURCE COMPARISON
 threshold       source   n     R2   RMSE    MAE    bias    KGE  baseline_R2  baseline_RMSE
        80           S2 259 0.1851 0.3103 0.2303  0.0021 0.2588      -0.1134         0.3627
        80 HLS_COMBINED 259 0.2064 0.3062 0.2355 -0.0052 0.3306      -0.1134         0.3627
        90           S2 243 0.2231 0.3022 0.2229 -0.0023 0.2794      -0.1178         0.3625
        90 HLS_COMBINED 243 0.1836 0.3098 0.2377  0.0012 0.3437      -0.1178         0.3625
        99 